# QUANTUM PORTFOLIO OPTIMIZATION - MAIN PIPELINE v4.0

**Project**: TFM - Quantum Computing for Portfolio Risk Management  
**Author**: Ignacio López Leis  
**Institution**: Universidad Autónoma de Madrid  
**Version**: 4.0 - Full Module Integration


---
# PHASE 0: INITIALIZATION
---

In [1]:
# ============================================================================
# CELL 1: ENVIRONMENT SETUP & PATH CONFIGURATION
# ============================================================================

import sys
import os
from pathlib import Path
from datetime import datetime
import yaml
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_PATH = PROJECT_ROOT / 'src'

for subdir in ['', 'utils', 'phase_0', 'phase_1', 'phase_2', 'phase_3','phase_4']:
    p = str(SRC_PATH / subdir) if subdir else str(PROJECT_ROOT)
    if p not in sys.path:
        sys.path.insert(0, p)

# ---------------------------------------------------------------------------
# Load YAML configuration
# ---------------------------------------------------------------------------
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Configuration not found: {CONFIG_PATH}")

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Extract phase configurations
global_config  = config.get('global', {})
phase0_config  = config.get('phase0', {})
phase1_config  = config.get('phase1', {})
phase2_config  = config.get('phase2', {})
phase3_config  = config.get('phase3', {})
phase4_config  = config.get('phase4', {})
export_config  = config.get('export', {})

# ---------------------------------------------------------------------------
# Global parameters
# ---------------------------------------------------------------------------
ALPHA       = global_config.get('alpha', 0.05)
RANDOM_SEED = global_config.get('random_seed', 42)
DEBUG_MODE  = global_config.get('debug_mode', False)
VERBOSE     = global_config.get('verbose', True)

np.random.seed(RANDOM_SEED)

CONSTRAINTS = global_config.get('constraints', {})
MAX_WEIGHT  = CONSTRAINTS.get('max_weight', 0.30)
MIN_WEIGHT  = CONSTRAINTS.get('min_weight', 0.00)
LONG_ONLY   = CONSTRAINTS.get('long_only', True)

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("=" * 70)
print("QUANTUM PORTFOLIO OPTIMIZATION - CONFIGURATION")
print("=" * 70)
print(f"  Alpha (CVaR)  : {ALPHA}")
print(f"  Random Seed   : {RANDOM_SEED}")
print(f"  Constraints   : max_weight={MAX_WEIGHT}, long_only={LONG_ONLY}")
print(f"  Project root  : {PROJECT_ROOT}")
print(f"  Config file   : {CONFIG_PATH}")
print("=" * 70)

QUANTUM PORTFOLIO OPTIMIZATION - CONFIGURATION
  Alpha (CVaR)  : 0.05
  Random Seed   : 42
  Constraints   : max_weight=0.3, long_only=True
  Project root  : C:\Users\nacho\Desktop\v2_final\v2_final
  Config file   : C:\Users\nacho\Desktop\v2_final\v2_final\config\config.yaml


In [2]:
# ============================================================================
# CELL 2: LOGGER INITIALIZATION (logger.py)
# ============================================================================
# Logs are OVERWRITTEN on each run.

log_config = phase0_config.get('logging', {})
LOG_DIR = log_config.get('log_directory', 'logs')
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

# Force-clean log file before logger init to guarantee fresh logs
LOG_FILE = Path(LOG_DIR) / 'main_notebook.txt'
if LOG_FILE.exists():
    LOG_FILE.unlink()

# ---------------------------------------------------------------------------
# Import logger module (must succeed -- pipeline cannot run without it)
# ---------------------------------------------------------------------------
from logger import get_logger, set_log_directory, Logger

# Reset singleton caches to force fresh file creation this run
if hasattr(Logger, '_instances'):
    Logger._instances.clear()
if hasattr(Logger, '_initialized_loggers'):
    Logger._initialized_loggers.clear()

# ---------------------------------------------------------------------------
# Create main logger
# ---------------------------------------------------------------------------
set_log_directory(LOG_DIR)
logger = get_logger('main_notebook')

logger.info("=" * 70)
logger.info("QUANTUM PORTFOLIO OPTIMIZATION PIPELINE")
logger.info("=" * 70)
logger.info(f"Execution started: {datetime.now().isoformat()}")
logger.info(f"Alpha: {ALPHA}, Seed: {RANDOM_SEED}")

print(f"Logger initialized -> {LOG_FILE}")

Logger initialized -> logs\main_notebook.txt


In [3]:
# ============================================================================
# CELL 3: PASSPORT SYSTEM (passport_orchestrator.py, passport_types.py)
# ============================================================================
# BUG-001 FIX: dual import pattern — works in both notebook and package context
# BUG-008 FIX: master passport chain — all module passports linked to master
# BUG-009 FIX: explicit export to results/exp_*/passports/pipeline_passport.json
# ============================================================================

import traceback as _tb

passport_config  = phase0_config.get('passport', {})
PASSPORT_ENABLED = passport_config.get('enabled', True)  # default True

orchestrator    = None
master_passport = None
PASSPORT_AVAILABLE = False

# ---------------------------------------------------------------------------
# Import passport modules
# ---------------------------------------------------------------------------
try:
    # Try src.utils first (when running as package)
    from src.utils.passport_orchestrator import get_orchestrator, reset_orchestrator, PassportOrchestrator
    from src.utils.passport_types import DataPassport, Seal, Checkpoint
    from src.utils.passport_utils import seal_function, track_execution
    PASSPORT_AVAILABLE = True
    logger.info("Passport imported via src.utils")
except ImportError:
    try:
        # Fallback: notebook sys.path context
        from passport_orchestrator import get_orchestrator, reset_orchestrator, PassportOrchestrator
        from passport_types import DataPassport, Seal, Checkpoint
        from passport_utils import seal_function, track_execution
        PASSPORT_AVAILABLE = True
        logger.info("Passport imported via bare path")
    except ImportError as _e:
        logger.error(f"Passport import FAILED: {type(_e).__name__}: {_e}")
        logger.error(_tb.format_exc())
        PASSPORT_AVAILABLE = False

# ---------------------------------------------------------------------------
# Initialize orchestrator and master passport
# ---------------------------------------------------------------------------
if PASSPORT_AVAILABLE and PASSPORT_ENABLED:
    try:
        orchestrator = reset_orchestrator()  # fresh singleton for this run
        master_passport = orchestrator.assign_passport(
            data={'pipeline': 'quantum_portfolio_optimization', 'run_id': str(__import__('uuid').uuid4())},
            data_type='pipeline_metadata',
            source='main_notebook',
            metadata={
                'execution_date': datetime.now().isoformat(),
                'alpha': ALPHA,
                'random_seed': RANDOM_SEED,
                'config_file': str(CONFIG_PATH),
                'experiment_id': EXPERIMENT_ID if 'EXPERIMENT_ID' in dir() else 'unknown',
            }
        )
        orchestrator.set_master(master_passport)  # BUG-008: register as chain master
        logger.info(f"Passport initialized: {master_passport.passport_id}")
        logger.info(f"Master passport chain: ACTIVE")
    except Exception as _pe:
        logger.error(f"Passport initialization failed: {_pe}")
        logger.error(_tb.format_exc())
        orchestrator = None
        master_passport = None
else:
    reason = "disabled in config" if not PASSPORT_ENABLED else "import failed"
    logger.warning(f"Passport system not active: {reason}")

print(f"Passport: enabled={PASSPORT_ENABLED}, available={PASSPORT_AVAILABLE}, "
      f"master={'OK:' + master_passport.passport_id[:8] if master_passport else 'None'}")


Passport: enabled=True, available=True, master=OK:8e9b9965


In [4]:
# ============================================================================
# CELL 4: VALIDATION HELPERS (validation_helpers.py)
# ============================================================================

from validation_helpers import (
    validate_type,
    validate_returns_matrix,
    validate_weights_vector,
    validate_alpha,
    ValidationError,
)

logger.info("Validation helpers loaded")
print("Validation helpers: OK")

Validation helpers: OK


In [5]:
# ============================================================================
# CELL 5: DATA LOADING (direct CSV)
# ============================================================================

data_config = phase0_config.get('data', {})
TICKERS    = data_config.get('tickers', ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META'])
START_DATE = data_config.get('start_date', '2020-01-01')
END_DATE   = data_config.get('end_date', '2025-12-31')
CSV_PATH = data_config.get('csv_path', str(PROJECT_ROOT / 'data' / 'returns_sp500_100.csv'))

# Resolve relative paths from project root (notebook may run from a subdirectory)
_csv_path = Path(CSV_PATH)
if not _csv_path.is_absolute() and not _csv_path.exists():
    _csv_path = (PROJECT_ROOT / _csv_path).resolve()
CSV_PATH = str(_csv_path)

# ---------------------------------------------------------------------------
# Load CSV and select tickers
# ---------------------------------------------------------------------------
logger.info(f"Loading CSV: {CSV_PATH}")
returns_df = pd.read_csv(CSV_PATH, index_col=0, parse_dates=True)
logger.info(f"CSV loaded: {returns_df.shape[0]} periods x {returns_df.shape[1]} tickers")

missing = [t for t in TICKERS if t not in returns_df.columns]
if missing:
    raise ValueError(f"Tickers not found in CSV: {missing}")

returns_df = returns_df[TICKERS]
returns_df = returns_df.loc[START_DATE:END_DATE]

if returns_df.isna().any().any():
    returns_df = returns_df.ffill().bfill()
    logger.info("NaN values filled (forward/backward)")

tickers   = returns_df.columns.tolist()
n_assets  = len(tickers)
n_periods = len(returns_df)

# ---------------------------------------------------------------------------
# Out-of-sample temporal split
# ---------------------------------------------------------------------------
oos_config     = phase3_config.get('out_of_sample', {})
OOS_ENABLED    = oos_config.get('enabled', False)
OOS_SPLIT_DATE = oos_config.get('optimization_end_date', '2025-01-01')

returns_df_full = returns_df.copy()
returns_df_oos  = None

if OOS_ENABLED:
    split_ts = pd.Timestamp(OOS_SPLIT_DATE)
    if split_ts <= returns_df.index[0] or split_ts >= returns_df.index[-1]:
        logger.error(
            f"OOS split date {OOS_SPLIT_DATE} outside data range "
            f"[{returns_df.index[0].date()}, {returns_df.index[-1].date()}]"
        )
        OOS_ENABLED = False
    else:
        returns_df     = returns_df_full[returns_df_full.index <= split_ts]
        returns_df_oos = returns_df_full[returns_df_full.index > split_ts]
        n_periods      = len(returns_df)
        logger.info(f"OOS split: train={len(returns_df)}, test={len(returns_df_oos)}")
else:
    logger.info("Out-of-sample validation disabled")

# ---------------------------------------------------------------------------
# Passport checkpoint
# ---------------------------------------------------------------------------
if orchestrator and master_passport:
    master_passport = orchestrator.checkpoint(
        master_passport,
        name='phase0_data_loading',
        data=returns_df.values,
        metadata={
            'n_assets': n_assets, 'n_periods': n_periods,
            'tickers': tickers, 'source': f'csv:{CSV_PATH}',
            'oos_enabled': OOS_ENABLED,
        }
    )

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
logger.info(f"Data ready: {n_periods}x{n_assets}")

print(f"Data loaded: {n_periods} periods x {n_assets} assets")
print(f"  Tickers : {tickers}")
print(f"  Range   : {returns_df.index[0].date()} to {returns_df.index[-1].date()}")
if OOS_ENABLED and returns_df_oos is not None:
    print(f"  OOS split: {OOS_SPLIT_DATE} (train={len(returns_df)}, test={len(returns_df_oos)})")

Data loaded: 1005 periods x 10 assets
  Tickers : ['AIG', 'AMP', 'BAC', 'BRK-B', 'ITW', 'IVZ', 'L', 'MCO', 'PRU', 'TROW']
  Range   : 2020-01-03 to 2023-12-29
  OOS split: 2024-01-01 (train=1005, test=501)


---
# PHASE 0.5: NETWORK-BASED PORTFOLIO SELECTION
---

In [6]:
# ============================================================================
# CELL 5.5a: NETWORK CONSTRUCTION & ANALYSIS
# ============================================================================
# Builds correlation network from full universe, filters via TMFG,
# computes centrality metrics and detects communities.
# Only runs if phase0.network_selection.enabled = true in config.
# ============================================================================

net_sel_config = phase0_config.get('network_selection', {})
NETWORK_SELECTION_ENABLED = net_sel_config.get('enabled', False)

# Placeholders for downstream cells
network_report = None
network_metrics_df = None
G_filtered = None

if NETWORK_SELECTION_ENABLED:
    import time as _ns_time

    logger.info("=" * 70)
    logger.info("PHASE 0.5: NETWORK-BASED PORTFOLIO SELECTION")
    logger.info("=" * 70)

    # Import Phase 0 modules
    from src.phase_0.network_portfolio_selector import (
        CorrelationNetworkBuilder,
        NetworkFilter,
        NetworkAnalyzer,
        DWAVE_AVAILABLE,
    )

    # -----------------------------------------------------------------------
    # Load FULL universe CSV (all tickers, not just config subset)
    # -----------------------------------------------------------------------
    _net_csv_path = net_sel_config.get(
        'csv_path', str(PROJECT_ROOT / 'data' / 'returns_sp500_full.csv')
    )
    _net_csv = Path(_net_csv_path)
    if not _net_csv.is_absolute() and not _net_csv.exists():
        _net_csv = (PROJECT_ROOT / _net_csv).resolve()

    if not _net_csv.exists():
        logger.error(f"Network CSV not found: {_net_csv}")
        logger.error("Run: python scripts/download_sp500_full.py")
        NETWORK_SELECTION_ENABLED = False
        print(f"WARNING: Network selection CSV not found: {_net_csv}")
        print("Falling back to config tickers.")
    else:
        _returns_universe = pd.read_csv(str(_net_csv), index_col=0, parse_dates=True)
        _returns_universe = _returns_universe.loc[START_DATE:END_DATE]
        if _returns_universe.isna().any().any():
            _returns_universe = _returns_universe.ffill().bfill()

        _n_universe = _returns_universe.shape[1]
        _n_periods_u = _returns_universe.shape[0]
        logger.info(f"Universe loaded: {_n_universe} assets, {_n_periods_u} periods")

        # -------------------------------------------------------------------
        # Step 1: Build correlation network
        # -------------------------------------------------------------------
        _net_cfg = net_sel_config.get('network', {})
        _t0 = _ns_time.time()

        builder = CorrelationNetworkBuilder(
            distance_metric=_net_cfg.get('distance_metric', 'mantegna')
        )
        G_full, corr_matrix_full = builder.build(_returns_universe)

        # -------------------------------------------------------------------
        # Step 2: Filter network (TMFG)
        # -------------------------------------------------------------------
        nf = NetworkFilter(method=_net_cfg.get('filtering_method', 'tmfg'))
        G_filtered = nf.filter(G_full)

        # -------------------------------------------------------------------
        # Step 3: Analyze (centrality + communities)
        # -------------------------------------------------------------------
        _comm_cfg = net_sel_config.get('community', {})
        analyzer = NetworkAnalyzer(
            community_algorithm=_comm_cfg.get('algorithm', 'louvain')
        )
        network_metrics_df = analyzer.analyze(G_filtered)

        _elapsed = _ns_time.time() - _t0
        _n_communities = network_metrics_df['community'].nunique()

        logger.info(f"Network analysis complete: {_elapsed:.1f}s")
        logger.info(f"Universe: {_n_universe} assets, "
                     f"filtered edges: {G_filtered.number_of_edges()}, "
                     f"communities: {_n_communities}")

        # -------------------------------------------------------------------
        # Summary
        # -------------------------------------------------------------------
        print("=" * 70)
        print("NETWORK ANALYSIS")
        print("=" * 70)
        print(f"  Universe       : {_n_universe} assets, {_n_periods_u} periods")
        print(f"  Full edges     : {G_full.number_of_edges()}")
        print(f"  Filtered edges : {G_filtered.number_of_edges()} "
              f"(target TMFG: {3 * (_n_universe - 2)})")
        print(f"  Communities    : {_n_communities}")
        print(f"  Time           : {_elapsed:.1f}s")
        print()
        print("  Top-10 PERIPHERAL (best diversifiers):")
        _top_p = network_metrics_df.nlargest(10, 'peripherality_score')
        for _, row in _top_p.iterrows():
            print(f"    {row['ticker']:<6s}  periph={row['peripherality_score']:.4f}  "
                  f"comm={int(row['community'])}  eigvec={row['eigenvector_centrality']:.4f}")
        print()
        print("  Top-10 CENTRAL (highest systemic risk):")
        _top_c = network_metrics_df.nsmallest(10, 'peripherality_score')
        for _, row in _top_c.iterrows():
            print(f"    {row['ticker']:<6s}  periph={row['peripherality_score']:.4f}  "
                  f"comm={int(row['community'])}  eigvec={row['eigenvector_centrality']:.4f}")
        print("=" * 70)

else:
    logger.info("Network selection disabled — using config tickers")
    print("Network selection: DISABLED (using config tickers)")


Network selection: DISABLED (using config tickers)


In [7]:
# ============================================================================
# CELL 5.5b: QUBO PORTFOLIO SELECTION VIA D-WAVE
# ============================================================================
# Builds the QUBO for asset selection, solves on D-Wave (or SimAnneal),
# and overrides tickers/returns_df for the rest of the pipeline.
# ============================================================================

selection_result = None

if NETWORK_SELECTION_ENABLED and network_metrics_df is not None:
    import time as _qs_time

    from src.phase_0.network_portfolio_selector import (
        PortfolioSelectionQUBO,
        DWaveSolver,
    )

    _qubo_cfg = net_sel_config.get('qubo', {})
    _dw_cfg   = net_sel_config.get('dwave', {})
    _comm_cfg = net_sel_config.get('community', {})
    _port_cfg = net_sel_config.get('portfolios', {})

    # Determine which portfolio to build from config tickers presence
    # If config has specific tickers already, respect them (pre-generated config)
    # Otherwise use the default portfolio spec
    _active_portfolio = None
    for _pid in ['A', 'B', 'C', 'D']:
        _pcfg = _port_cfg.get(_pid, {})
        if data_config.get('portfolio_name', '').endswith(f'P{_pid}'):
            _active_portfolio = _pid
            break

    if _active_portfolio is None:
        # Default: build portfolio A (peripheral, K=10)
        _active_portfolio = 'A'
        logger.info("No portfolio ID detected in config, defaulting to A")

    _pcfg = _port_cfg.get(_active_portfolio, {})
    _method = _pcfg.get('method', 'qubo_peripheral')
    _target_k = _pcfg.get('size', 10)

    logger.info(f"Portfolio {_active_portfolio}: method={_method}, K={_target_k}")

    _all_tickers = _returns_universe.columns.tolist()
    _t0 = _qs_time.time()

    if _method == 'qubo_peripheral':
        # Build QUBO
        _max_per_comm = (
            _comm_cfg.get('max_assets_per_community_k10', 3)
            if _target_k <= 20
            else _comm_cfg.get('max_assets_per_community_k100', 15)
        )

        qubo_builder = PortfolioSelectionQUBO(
            lambda_correlation=_qubo_cfg.get('lambda_correlation', 1.0),
            lambda_peripherality=_qubo_cfg.get('lambda_peripherality', 0.5),
            penalty_cardinality=_qubo_cfg.get('penalty_cardinality', 10.0),
            penalty_community=_qubo_cfg.get('penalty_community', 5.0),
            penalty_bridge=_qubo_cfg.get('penalty_bridge', 0.3),
        )

        bqm = qubo_builder.build(
            metrics_df=network_metrics_df,
            corr_matrix=corr_matrix_full,
            tickers=_all_tickers,
            target_k=_target_k,
            max_per_community=_max_per_comm,
        )

        # Solve
        solver = DWaveSolver(
            backend=_dw_cfg.get('backend', 'simulated_annealing'),
            num_reads=_dw_cfg.get('num_reads', 1000),
            qpu_solver=_dw_cfg.get('qpu_solver'),
        )

        selection_result = solver.solve(
            bqm=bqm,
            target_k=_target_k,
            metrics_df=network_metrics_df,
            max_per_community=_max_per_comm,
        )

    elif _method == 'central_ranking':
        _selected = (
            network_metrics_df
            .nlargest(_target_k, 'eigenvector_centrality')['ticker']
            .tolist()
        )
        from src.phase_0.network_portfolio_selector import SelectionResult
        selection_result = SelectionResult(
            portfolio_id=_active_portfolio,
            method='central_ranking',
            selected_tickers=_selected,
            energy=0.0, feasible=True, num_reads=0,
        )

    elif _method == 'random':
        np.random.seed(RANDOM_SEED)
        _selected = list(np.random.choice(_all_tickers, _target_k, replace=False))
        from src.phase_0.network_portfolio_selector import SelectionResult
        selection_result = SelectionResult(
            portfolio_id=_active_portfolio,
            method='random',
            selected_tickers=_selected,
            energy=0.0, feasible=True, num_reads=0,
        )
        np.random.seed(RANDOM_SEED)  # Reset for pipeline determinism

    _elapsed = _qs_time.time() - _t0
    selection_result.portfolio_id = _active_portfolio

    # -------------------------------------------------------------------
    # Override tickers and returns_df for the rest of the pipeline
    # -------------------------------------------------------------------
    _new_tickers = selection_result.selected_tickers

    if _new_tickers:
        # Validate tickers exist in universe
        _valid = [t for t in _new_tickers if t in _returns_universe.columns]
        if len(_valid) < len(_new_tickers):
            _missing = set(_new_tickers) - set(_valid)
            logger.warning(f"Tickers not in universe (dropped): {_missing}")
            _new_tickers = _valid

        # Override pipeline variables
        returns_df = _returns_universe[_new_tickers].loc[START_DATE:END_DATE]
        if returns_df.isna().any().any():
            returns_df = returns_df.ffill().bfill()

        tickers   = returns_df.columns.tolist()
        n_assets  = len(tickers)
        n_periods = len(returns_df)

        # Redo OOS split with new data
        returns_df_full = returns_df.copy()
        returns_df_oos  = None
        if OOS_ENABLED:
            split_ts = pd.Timestamp(OOS_SPLIT_DATE)
            if split_ts > returns_df.index[0] and split_ts < returns_df.index[-1]:
                returns_df     = returns_df_full[returns_df_full.index <= split_ts]
                returns_df_oos = returns_df_full[returns_df_full.index > split_ts]
                n_periods      = len(returns_df)

        logger.info(f"Pipeline tickers overridden: {n_assets} assets")
    else:
        logger.error("Network selection returned empty ticker list!")

    # -------------------------------------------------------------------
    # Network metrics for selected portfolio
    # -------------------------------------------------------------------
    _sel_metrics = network_metrics_df[
        network_metrics_df['ticker'].isin(_new_tickers)
    ]
    _avg_periph = _sel_metrics['peripherality_score'].mean()
    _avg_centr  = _sel_metrics['eigenvector_centrality'].mean()
    _n_comm     = _sel_metrics['community'].nunique()

    # Save metrics CSV
    _net_results_dir = Path('results') / 'network_selection'
    if not _net_results_dir.is_absolute():
        _net_results_dir = PROJECT_ROOT / _net_results_dir
    _net_results_dir.mkdir(parents=True, exist_ok=True)
    network_metrics_df.to_csv(_net_results_dir / 'network_metrics.csv', index=False)
    _sel_metrics.to_csv(_net_results_dir / f'selected_P{_active_portfolio}_metrics.csv',
                        index=False)

    # Save selection result JSON
    import json as _json
    _sel_json = {
        'portfolio_id': _active_portfolio,
        'method': selection_result.method,
        'selected_tickers': selection_result.selected_tickers,
        'energy': float(selection_result.energy) if np.isfinite(selection_result.energy) else None,
        'feasible': selection_result.feasible,
        'num_reads': selection_result.num_reads,
        'timing_ms': selection_result.timing_ms,
        'n_universe': _n_universe,
        'n_selected': len(_new_tickers),
        'avg_peripherality': float(_avg_periph),
        'avg_centrality': float(_avg_centr),
        'n_communities_represented': int(_n_comm),
    }
    with open(_net_results_dir / f'selection_P{_active_portfolio}.json', 'w') as _f:
        _json.dump(_sel_json, _f, indent=2, default=str)

    # Passport checkpoint
    if orchestrator and master_passport:
        master_passport = orchestrator.checkpoint(
            master_passport,
            name='phase0_network_selection',
            data={'selected_tickers': _new_tickers},
            metadata={
                'portfolio_id': _active_portfolio,
                'method': selection_result.method,
                'n_universe': _n_universe,
                'n_selected': len(_new_tickers),
                'energy': float(selection_result.energy) if np.isfinite(selection_result.energy) else None,
                'feasible': selection_result.feasible,
                'avg_peripherality': float(_avg_periph),
            }
        )

    # -------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------
    print("=" * 70)
    print(f"PORTFOLIO {_active_portfolio} — SELECTION RESULT ({_method})")
    print("=" * 70)
    print(f"  Universe         : {_n_universe} assets")
    print(f"  Selected         : {len(_new_tickers)} assets")
    print(f"  Method           : {selection_result.method}")
    print(f"  Energy           : {selection_result.energy:.4f}")
    print(f"  Feasible         : {selection_result.feasible}")
    print(f"  D-Wave reads     : {selection_result.num_reads}")
    print(f"  Time             : {_elapsed:.2f}s")
    print(f"  Avg peripherality: {_avg_periph:.4f}")
    print(f"  Avg centrality   : {_avg_centr:.4f}")
    print(f"  Communities      : {_n_comm}")
    print(f"  Tickers: {tickers}")
    print(f"  Data: {n_periods} periods x {n_assets} assets")
    if OOS_ENABLED and returns_df_oos is not None:
        print(f"  OOS split: train={len(returns_df)}, test={len(returns_df_oos)}")
    print("=" * 70)

else:
    if not NETWORK_SELECTION_ENABLED:
        pass  # Already logged above
    else:
        logger.warning("Network metrics not available, skipping QUBO selection")
        print("Network selection: SKIPPED (no network metrics)")


In [8]:
# ============================================================================
# CELL 6: EFFECTIVE RETURNS
# ============================================================================
# Clustering is disabled in the current configuration.
# This cell defines returns_effective and n_effective for downstream cells.

returns_effective = returns_df.values
n_effective = n_assets

logger.info(f"Effective dimension: {n_effective} assets (no clustering)")
print(f"Effective dimension: {n_effective} assets")

Effective dimension: 10 assets


---
# PHASE 1: CLASSICAL BASELINE
---

In [9]:
# ============================================================================
# CELL 7: INITIAL WEIGHTS
# ============================================================================

weights_config = phase1_config.get('initial_weights', {})
WEIGHTS_METHOD = weights_config.get('method', 'equal')

if WEIGHTS_METHOD == 'random':
    perturb = np.random.normal(0, 0.01, n_effective)
    initial_weights = np.ones(n_effective) / n_effective + perturb
    initial_weights = np.clip(initial_weights, 0, None)
    initial_weights /= initial_weights.sum()
else:
    initial_weights = np.ones(n_effective) / n_effective

validate_weights_vector(initial_weights, expected_size=n_effective, paramname="initial_weights")
logger.info(f"Initial weights: method={WEIGHTS_METHOD}, sum={initial_weights.sum():.6f}")

print(f"Initial weights: {WEIGHTS_METHOD}, dim={n_effective}")

Initial weights: equal, dim=10


In [10]:
# ============================================================================
# CELL 8: CLASSICAL CVaR (cvar_computation.py)
# ============================================================================

from cvar_computation import compute_var, compute_cvar, compute_cvar_baseline

cvar_config = phase1_config.get('cvar', {})
CVAR_METHOD = cvar_config.get('method', 'historical')

logger.info(f"Computing CVaR: method={CVAR_METHOD}, alpha={ALPHA}")

# ---------------------------------------------------------------------------
# Compute baseline CVaR on effective returns
# ---------------------------------------------------------------------------
returns_df_effective = pd.DataFrame(returns_effective)

baseline = compute_cvar_baseline(
    returns_df=returns_df_effective,
    initial_weights=initial_weights,
    alpha=ALPHA,
    passport=master_passport if orchestrator else None,
)

var_classical       = float(baseline['var'])
cvar_classical      = float(baseline['cvar'])
portfolio_returns   = baseline['portfolio_returns']
tail_size           = int(baseline['tail_size'])
portfolio_mean      = float(np.mean(portfolio_returns))
portfolio_std       = float(np.std(portfolio_returns))
tail_prob_classical = tail_size / len(portfolio_returns)

# ---------------------------------------------------------------------------
# Passport seal
# ---------------------------------------------------------------------------
if orchestrator and master_passport:
    master_passport = orchestrator.seal(
        master_passport,
        function_name='compute_cvar_baseline',
        phase='Phase 1',
        result={
            'execution_time_ms': 0,
            'validation_status': 'valid',
            'parameters': {'alpha': float(ALPHA), 'method': CVAR_METHOD},
            'data_shape': list(returns_effective.shape),
            'results': {
                'var': var_classical, 'cvar': cvar_classical,
                'tail_size': tail_size,
            },
        }
    )

logger.info(f"CVaR baseline: VaR={var_classical:.6f}, CVaR={cvar_classical:.6f}")

print(f"Classical CVaR computed (method={CVAR_METHOD})")
print(f"  VaR  ({(1-ALPHA)*100:.0f}%): {var_classical:.6f}")
print(f"  CVaR ({(1-ALPHA)*100:.0f}%): {cvar_classical:.6f}")
print(f"  Tail: {tail_size}/{len(portfolio_returns)} ({tail_prob_classical*100:.1f}%)")

Classical CVaR computed (method=historical)


  VaR  (95%): -0.028764
  CVaR (95%): -0.047654
  Tail: 51/1005 (5.1%)


In [11]:
# ============================================================================
# CELL 9: METRICS COMPUTATION (metricscomputation.py)
# ============================================================================

from metricscomputation import MetricsComputation

risk_free_rate = phase1_config.get('risk_free_rate', 0.02)
metrics_calc   = MetricsComputation(risk_free_rate=risk_free_rate)

portfolio_metrics = metrics_calc.compute_all_metrics(
    returns_df=returns_df_effective,
    weights=initial_weights,
    passport=master_passport,
)

logger.info("Portfolio metrics computed")

print("Portfolio metrics (initial weights)")
for k, v in portfolio_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")

Portfolio metrics (initial weights)
  annual_return: 0.087565
  annual_volatility: 0.322097
  sharpe_ratio: 0.209766
  sortino_ratio: 0.289145
  max_drawdown: -0.516868
  risk_free_rate: 0.020000


In [12]:
# ============================================================================
# CELL 10: NOISE MODELS (noisemodels.py)
# ============================================================================

from noisemodels import create_noise_model

noise_config  = phase1_config.get('noise_models', {})
GATE_ERROR_1Q = noise_config.get('single_qubit_gate_error', 0.001)
GATE_ERROR_2Q = noise_config.get('two_qubit_gate_error', 0.01)
READOUT_ERROR = noise_config.get('readout_error', 0.02)
T1_US         = noise_config.get('t1_microseconds', 50.0)
T2_US         = noise_config.get('t2_microseconds', 70.0)

noise_params = {
    'single_qubit_gate_error': GATE_ERROR_1Q,
    'two_qubit_gate_error': GATE_ERROR_2Q,
    'readout_error': READOUT_ERROR,
    't1_seconds': T1_US * 1e-6,
    't2_seconds': T2_US * 1e-6,
}

# ---------------------------------------------------------------------------
# Create depolarizing noise model
# ---------------------------------------------------------------------------
noise_model = create_noise_model(
    noise_type='depolarizing',
    error_rate=GATE_ERROR_2Q,
)

logger.info(f"Noise model created: 1Q={GATE_ERROR_1Q}, 2Q={GATE_ERROR_2Q}")

print(f"Noise model: 1Q={GATE_ERROR_1Q*100:.2f}%, 2Q={GATE_ERROR_2Q*100:.2f}%")

Noise model: 1Q=0.00%, 2Q=0.00%


---
# PHASE 2: QUANTUM-HYBRID OPTIMIZATION
---

In [13]:
# ============================================================================
# CELL 12: QAE CVaR ESTIMATION
# ============================================================================
# Executes quantum CVaR estimation with 3-tier error mitigation:
#   TIER 1: Measurement Error Mitigation (MEM) -- readout correction
#   TIER 2: ZNE on State Preparation -- corrects Mottonen encoding errors
#   TIER 3: ZNE on amplitudes via unitary folding (inside qae_circuits.py)
#
# Methods: IQAE (recommended), Woerner (noiseless cross-validation)
# ============================================================================

import time

from qae_circuits import (
    QuantumAmplitudeEstimator,
    QuantumCVaRResult,
    QISKIT_AVAILABLE,
    QISKIT_ALGORITHMS_AVAILABLE,
)

QUANTUM_AVAILABLE = QISKIT_AVAILABLE and QISKIT_ALGORITHMS_AVAILABLE

logger.info("=" * 70)
logger.info("QAE CVaR ESTIMATION")
logger.info("=" * 70)

# ---------------------------------------------------------------------------
# QAE configuration
# ---------------------------------------------------------------------------
qae_config = phase2_config.get('qae', {})

NUM_DIST_QUBITS    = int(qae_config.get('num_distribution_qubits', 4))
EPSILON            = float(qae_config.get('epsilon', 0.05))
QAE_SHOTS          = int(qae_config.get('shots',
                         phase2_config.get('backend', {}).get('default_shots', 8192)))
QAE_MAX_TOTAL_QUBITS = int(qae_config.get('max_total_qubits', 16))
QAE_TIMEOUT        = int(qae_config.get('timeout_seconds', 180))

methods_config = qae_config.get('methods', {})
USE_IQAE = (methods_config.get('iqae', {}).get('enabled', True)
            if isinstance(methods_config.get('iqae'), dict)
            else methods_config.get('use_iqae', True))
USE_WOERNER = (methods_config.get('woerner', {}).get('enabled', True)
               if isinstance(methods_config.get('woerner'), dict)
               else methods_config.get('use_woerner', True))

QAE_ACTIVE_METHODS = []
if USE_IQAE:
    QAE_ACTIVE_METHODS.append('iqae')
if USE_WOERNER:
    QAE_ACTIVE_METHODS.append('woerner')

# ---------------------------------------------------------------------------
# Noise model
# ---------------------------------------------------------------------------
noise_model_raw = phase2_config.get('backend', {}).get('noise_model', {})
noise_is_enabled = noise_model_raw.get('enabled', False)

QAE_NOISE_MODEL_CONFIG = None
if noise_is_enabled:
    QAE_NOISE_MODEL_CONFIG = {
        'enabled': True,
        'single_qubit_gate_error': float(noise_model_raw.get('single_qubit_gate_error', 0.001)),
        'two_qubit_gate_error':    float(noise_model_raw.get('two_qubit_gate_error', 0.01)),
        'readout_error':           float(noise_model_raw.get('readout_error', 0.02)),
    }
    logger.info(f"Noise ENABLED: sq={QAE_NOISE_MODEL_CONFIG['single_qubit_gate_error']}, "
                f"tq={QAE_NOISE_MODEL_CONFIG['two_qubit_gate_error']}, "
                f"ro={QAE_NOISE_MODEL_CONFIG['readout_error']}")
else:
    logger.info("Noise DISABLED (ideal simulation)")

# ---------------------------------------------------------------------------
# Error mitigation tiers (all handled inside QuantumAmplitudeEstimator)
# ---------------------------------------------------------------------------
em_config = phase2_config.get('error_mitigation', {})

USE_TIER1_MEM     = em_config.get('readout_mitigation', False) and noise_is_enabled
USE_TIER2_ZNE_SP  = em_config.get('zne_state_prep', False) and noise_is_enabled
USE_TIER3_ZNE_AMP = em_config.get('zne_outer_cvar', False) and noise_is_enabled

ZNE_SCALE_FACTORS = em_config.get('zne_scale_factors', [1.0, 3.0, 5.0])
ZNE_EXTRAPOLATION = em_config.get('zne_extrapolation',
                    em_config.get('zne_outer_extrapolation', 'linear'))

logger.info(f"Methods: {QAE_ACTIVE_METHODS}, Shots: {QAE_SHOTS}")
logger.info(f"Mitigation: TIER1={USE_TIER1_MEM}, TIER2={USE_TIER2_ZNE_SP}, TIER3={USE_TIER3_ZNE_AMP}")

print("=" * 70)
print("QAE CVaR ESTIMATION")
print("=" * 70)
print(f"  Methods: {QAE_ACTIVE_METHODS}  |  Qubits: {NUM_DIST_QUBITS}  |  Shots: {QAE_SHOTS}")
print(f"  Noise: {'ENABLED' if noise_is_enabled else 'DISABLED'}")
print(f"  Mitigation: T1-MEM={USE_TIER1_MEM}, T2-ZNE-SP={USE_TIER2_ZNE_SP}, T3-ZNE-Amp={USE_TIER3_ZNE_AMP}")

# ---------------------------------------------------------------------------
# Portfolio losses (convention: positive = bad)
# ---------------------------------------------------------------------------
portfolio_losses = -(returns_effective @ initial_weights)
logger.info(f"Losses: min={portfolio_losses.min():.6f}, max={portfolio_losses.max():.6f}")

# ---------------------------------------------------------------------------
# Execute QAE estimation
# ---------------------------------------------------------------------------
qae_results = None
cvar_qae_result = None
cvar_qae_estimate = None
var_qae_estimate = None

if QUANTUM_AVAILABLE and len(QAE_ACTIVE_METHODS) > 0:
    try:
        start_time = time.time()

        cvar_qae = QuantumAmplitudeEstimator(
            num_distribution_qubits=NUM_DIST_QUBITS,
            epsilon=EPSILON,
            shots=QAE_SHOTS,
            export_circuits=True,
            circuit_export_dir='results/circuits',
            allow_fallback=False,
            max_total_qubits=QAE_MAX_TOTAL_QUBITS,
            noise_model_config=QAE_NOISE_MODEL_CONFIG,
            use_readout_mitigation=USE_TIER1_MEM,
            use_zne_state_prep=USE_TIER2_ZNE_SP,
            use_zne=USE_TIER3_ZNE_AMP,
            zne_scale_factors=ZNE_SCALE_FACTORS if USE_TIER3_ZNE_AMP else None,
            zne_extrapolation=ZNE_EXTRAPOLATION,
            woerner_timeout=QAE_TIMEOUT,
        )

        cvar_qae_result = cvar_qae.estimate_cvar_complete(
            losses=portfolio_losses,
            alpha=ALPHA,
            methods=QAE_ACTIVE_METHODS,
            returns=returns_effective,
            weights=initial_weights,
        )

        elapsed = time.time() - start_time

        # ---------------------------------------------------------------
        # Extract results
        # ---------------------------------------------------------------
        final_cvar = float(cvar_qae_result.cvar_estimate)
        classical_cvar_ref = float(cvar_qae_result.classical_cvar)
        cvar_rel_error = abs(final_cvar - classical_cvar_ref) / max(abs(classical_cvar_ref), 1e-10)

        zne_corrections = int(cvar_qae.stats.get('zne_corrections', 0))
        zne_total_improvement = float(cvar_qae.stats.get('zne_total_improvement', 0.0))
        per_method_cvar = dict(getattr(cvar_qae_result, 'per_method_cvar', {}) or {})

        qae_results = {
            'quantum_var':          float(cvar_qae_result.var_estimate),
            'quantum_cvar':         final_cvar,
            'quantum_tail_prob':    float(cvar_qae_result.tail_probability),
            'classical_var':        float(cvar_qae_result.classical_var),
            'classical_cvar':       classical_cvar_ref,
            'classical_tail_prob':  float(cvar_qae_result.classical_tail_prob),
            'var_error':            float(cvar_qae_result.var_error_vs_classical),
            'cvar_error':           cvar_rel_error,
            'tail_prob_error':      float(cvar_qae_result.tail_prob_error),
            'total_queries_iqae':   int(cvar_qae_result.total_oracle_queries_iqae),
            'total_queries_woerner': int(cvar_qae_result.total_oracle_queries_woerner),
            'total_circuits':       int(cvar_qae_result.total_circuits_executed),
            'execution_time_ms':    float(cvar_qae_result.execution_time_ms),
            'loss_range':           tuple(float(x) for x in getattr(
                                        cvar_qae_result, 'loss_range',
                                        (float(portfolio_losses.min()), float(portfolio_losses.max())))),
            'png_files':            list(getattr(cvar_qae_result, 'png_files', [])),
            'method_results':       dict(getattr(cvar_qae_result, 'method_results', {})),
            'zne_applied':          bool(cvar_qae_result.zne_applied),
            'zne_improvement':      float(cvar_qae_result.zne_improvement),
            'zne_corrections':      zne_corrections,
            'noise_enabled':        QAE_NOISE_MODEL_CONFIG is not None,
            'woerner_retries':      int(cvar_qae.stats.get('woerner_retries', 0)),
            # Error mitigation tier details
            'tier1_mem_enabled':          USE_TIER1_MEM,
            'tier1_mem_corrections':      int(cvar_qae.stats.get('mem_corrections', 0)),
            'tier2_zne_sp_enabled':       USE_TIER2_ZNE_SP,
            'tier2_zne_sp_corrections':   int(cvar_qae.stats.get('zne_state_prep_corrections', 0)),
            'tier3_zne_amp_enabled':      USE_TIER3_ZNE_AMP,
            'tier3_zne_amp_corrections':  zne_corrections,
            'tier3_zne_amp_improvement':  zne_total_improvement,
            'tier3_zne_cvar_enabled':     USE_TIER3_ZNE_AMP,
            'tier3_data': {
                'method': 'unitary_folding',
                'scale_factors': ZNE_SCALE_FACTORS if USE_TIER3_ZNE_AMP else [],
                'extrapolation': ZNE_EXTRAPOLATION,
                'zne_corrections': zne_corrections,
                'total_improvement': zne_total_improvement,
            },
            'num_distribution_qubits': int(cvar_qae_result.num_distribution_qubits),
            'num_scenarios':           int(cvar_qae_result.num_scenarios),
            'alpha':                   float(cvar_qae_result.alpha),
            'success':                 True,
            'execution_time_s':        elapsed,
            'per_method_cvar':         per_method_cvar,
            # IQAE noise impact comparison
            'iqae_cvar_ideal':         (float(cvar_qae_result.iqae_cvar_ideal)
                                        if cvar_qae_result.iqae_cvar_ideal is not None else None),
            'iqae_cvar_noisy_raw':     (float(cvar_qae_result.iqae_cvar_noisy_raw)
                                        if cvar_qae_result.iqae_cvar_noisy_raw is not None else None),
            'iqae_cvar_noisy_zne':     (float(cvar_qae_result.iqae_cvar_noisy_zne)
                                        if cvar_qae_result.iqae_cvar_noisy_zne is not None else None),
            'iqae_tail_prob_ideal':    (float(cvar_qae_result.iqae_tail_prob_ideal)
                                        if cvar_qae_result.iqae_tail_prob_ideal is not None else None),
            'iqae_tail_prob_noisy_raw': (float(cvar_qae_result.iqae_tail_prob_noisy_raw)
                                         if cvar_qae_result.iqae_tail_prob_noisy_raw is not None else None),
            'iqae_tail_prob_noisy_zne': (float(cvar_qae_result.iqae_tail_prob_noisy_zne)
                                         if cvar_qae_result.iqae_tail_prob_noisy_zne is not None else None),
        }

        cvar_qae_estimate = final_cvar
        var_qae_estimate = cvar_qae_result.var_estimate

        # ---------------------------------------------------------------
        # Output
        # ---------------------------------------------------------------
        logger.info(f"QAE complete: CVaR={final_cvar:.6f}, error={cvar_rel_error*100:.2f}%, time={elapsed:.2f}s")

        print(f"\n  Quantum CVaR:  {final_cvar:.6f}  (classical: {classical_cvar_ref:.6f}, error: {cvar_rel_error*100:.2f}%)")
        print(f"  Tail prob:     {qae_results['quantum_tail_prob']:.4f}  (classical: {qae_results['classical_tail_prob']:.4f})")
        print(f"  Queries:       IQAE={qae_results['total_queries_iqae']}, Woerner={qae_results['total_queries_woerner']}")

        if per_method_cvar:
            print(f"  Per-method:")
            for m_name, m_cvar in per_method_cvar.items():
                m_err = abs(m_cvar - classical_cvar_ref) / max(abs(classical_cvar_ref), 1e-10)
                tag = " (noiseless ref)" if m_name == 'woerner' and noise_is_enabled else ""
                print(f"    {m_name.upper()}{tag}: CVaR={m_cvar:.6f} (error={m_err*100:.2f}%)")

        print(f"  Mitigation:    T1={qae_results['tier1_mem_corrections']} corr, "
              f"T2={qae_results['tier2_zne_sp_corrections']} corr, "
              f"T3={zne_corrections} corr (improvement={zne_total_improvement:.6f})")

        if qae_results.get('iqae_cvar_ideal') is not None:
            print(f"  IQAE noise comparison:")
            for label, key in [("Ideal", 'iqae_cvar_ideal'),
                               ("Noisy raw", 'iqae_cvar_noisy_raw'),
                               ("Noisy+ZNE+MEM", 'iqae_cvar_noisy_zne')]:
                val = qae_results.get(key)
                if val is not None:
                    err = abs(val - classical_cvar_ref) / max(abs(classical_cvar_ref), 1e-10)
                    print(f"    {label:16s}: {val:.6f} (error={err*100:.2f}%)")

        print(f"  Time: {elapsed:.2f}s")
        print("=" * 70)

    except Exception as e:
        logger.error(f"QAE estimation failed: {e}")
        import traceback
        logger.error(traceback.format_exc())

        qae_results = {
            'success': False, 'error': str(e),
            'quantum_cvar': 0.0,
            'classical_cvar': float(-cvar_classical) if 'cvar_classical' in dir() else 0.0,
            'total_queries_iqae': 0, 'total_queries_woerner': 0,
            'execution_time_ms': 0.0,
            'loss_range': (float(portfolio_losses.min()), float(portfolio_losses.max())),
            'per_method_cvar': {},
        }
        cvar_qae_estimate = None
        var_qae_estimate = None
        print(f"\nERROR: QAE estimation failed - {e}")

else:
    logger.warning("QAE not available or no methods enabled")
    qae_results = {
        'success': False, 'error': 'QAE not available',
        'quantum_cvar': 0.0, 'total_queries_iqae': 0,
        'total_queries_woerner': 0, 'per_method_cvar': {},
    }
    print("WARNING: QAE not available or no methods enabled")

# ---------------------------------------------------------------------------
# Passport checkpoint
# ---------------------------------------------------------------------------
if orchestrator and master_passport:
    master_passport = orchestrator.checkpoint(
        master_passport,
        name='phase2_qae_estimation',
        data={
            'qae_cvar': qae_results.get('quantum_cvar', 0) if qae_results else 0,
            'classical_cvar': qae_results.get('classical_cvar', 0) if qae_results else 0,
            'success': qae_results.get('success', False) if qae_results else False,
            'per_method_cvar': qae_results.get('per_method_cvar', {}) if qae_results else {},
        },
        metadata={
            'total_queries_iqae': qae_results.get('total_queries_iqae', 0) if qae_results else 0,
            'total_queries_woerner': qae_results.get('total_queries_woerner', 0) if qae_results else 0,
            'noise_enabled': qae_results.get('noise_enabled', False) if qae_results else False,
            'tier1_mem': qae_results.get('tier1_mem_enabled', False) if qae_results else False,
            'tier2_zne_sp': qae_results.get('tier2_zne_sp_enabled', False) if qae_results else False,
            'tier3_zne_amp': qae_results.get('tier3_zne_amp_enabled', False) if qae_results else False,
        }
    )

QAE CVaR ESTIMATION
  Methods: ['iqae', 'woerner']  |  Qubits: 4  |  Shots: 4096
  Noise: DISABLED
  Mitigation: T1-MEM=False, T2-ZNE-SP=False, T3-ZNE-Amp=False



  Quantum CVaR:  0.047386  (classical: 0.047654, error: 0.56%)
  Tail prob:     0.0507  (classical: 0.0507)
  Queries:       IQAE=418816, Woerner=128
  Per-method:
    IQAE: CVaR=0.047386 (error=0.56%)
    WOERNER: CVaR=0.049129 (error=3.09%)
  Mitigation:    T1=0 corr, T2=0 corr, T3=0 corr (improvement=0.000000)
  IQAE noise comparison:
    Ideal           : 0.047057 (error=1.25%)
    Noisy raw       : 0.047057 (error=1.25%)
    Noisy+ZNE+MEM   : 0.047057 (error=1.25%)
  Time: 88.03s


In [14]:
# ============================================================================
# CELL 13: SUBGRADIENT ESTIMATOR (quantumsubgradient.py)
# ============================================================================
# Estimates CVaR subgradient using QAE for quantum-classical optimization.
# Formula: g_j = -E[r_j | L >= VaR]
# Note: conditional mean computed directly (mean over tail observations);
# this implicitly divides by P(tail). No explicit division needed.
# Ref: Rockafellar & Uryasev (2000), Proposition 2.
# ============================================================================

import time

from quantumsubgradient import (
    QuantumSubgradientEstimator,
    SubgradientResult,
    estimate_cvar_subgradient,
    QAE_AVAILABLE as SUBGRAD_QAE_AVAILABLE,
    QISKIT_ALGORITHMS_AVAILABLE as SUBGRAD_ALGO_AVAILABLE,
)

subgrad_config = phase2_config.get('subgradient', {})

SUBGRAD_USE_ZNE    = subgrad_config.get('use_zne', True)
SUBGRAD_VALIDATE   = subgrad_config.get('validate_vs_classical', True)
SUBGRAD_TOLERANCE  = float(subgrad_config.get('validation_tolerance', 0.20))
SUBGRAD_QUBITS     = int(subgrad_config.get('num_qubits', NUM_DIST_QUBITS))
SUBGRAD_EPSILON    = float(subgrad_config.get('epsilon', EPSILON))
SUBGRAD_USE_IQAE   = subgrad_config.get('use_iqae', USE_IQAE)
SUBGRAD_USE_WOERNER = subgrad_config.get('use_woerner', USE_WOERNER)

logger.info("=" * 70)
logger.info("QUANTUM SUBGRADIENT ESTIMATION")
logger.info("=" * 70)
logger.info(f"Config: qubits={SUBGRAD_QUBITS}, epsilon={SUBGRAD_EPSILON}, ZNE={SUBGRAD_USE_ZNE}")

# ---------------------------------------------------------------------------
# Classical subgradient (always computed as reference)
# ---------------------------------------------------------------------------
def compute_classical_subgradient(weights, returns, alpha):
    """Classical CVaR subgradient: g_j = -E[r_j | L >= VaR]. Ref: R&U 2000 Prop.2."""
    T, N = returns.shape
    portfolio_losses = -(returns @ weights)
    var_threshold = np.percentile(portfolio_losses, (1 - alpha) * 100)
    tail_mask = portfolio_losses >= var_threshold
    n_tail = np.sum(tail_mask)

    if n_tail == 0:
        return np.zeros(N, dtype=np.float64), {
            'tail_prob': 0.0, 'var_threshold': float(var_threshold), 'n_tail': 0}

    tail_prob = n_tail / T
    conditional_exp = np.mean(returns[tail_mask], axis=0)
    subgradient = -conditional_exp / tail_prob

    return subgradient.astype(np.float64), {
        'tail_prob': float(tail_prob), 'var_threshold': float(var_threshold),
        'n_tail': int(n_tail)}

classical_subgradient, classical_subgrad_info = compute_classical_subgradient(
    initial_weights, returns_effective, ALPHA)

logger.info(f"Classical subgradient: norm={np.linalg.norm(classical_subgradient):.6f}, "
            f"tail_prob={classical_subgrad_info['tail_prob']:.6f}")

# ---------------------------------------------------------------------------
# Quantum subgradient estimation
# ---------------------------------------------------------------------------
subgrad_result = None
quantum_subgradient = None
subgrad_results = None

if SUBGRAD_QAE_AVAILABLE and SUBGRAD_ALGO_AVAILABLE:
    try:
        start_time = time.time()

        subgrad_estimator = QuantumSubgradientEstimator(
            num_distribution_qubits=SUBGRAD_QUBITS,
            epsilon=SUBGRAD_EPSILON,
            alpha=ALPHA,
            use_iqae=SUBGRAD_USE_IQAE,
            use_woerner=SUBGRAD_USE_WOERNER,
            backend='aer_simulator',
            passport=master_passport if orchestrator else None,
        )

        # Convert to loss convention (positive = bad)
        var_as_loss = -var_classical if var_classical < 0 else var_classical
        cvar_as_loss = -cvar_classical if cvar_classical < 0 else cvar_classical

        subgrad_result = subgrad_estimator.estimate_subgradient(
            weights=initial_weights,
            returns=returns_effective,
            classical_var=var_as_loss,
            classical_cvar=cvar_as_loss,
        )

        elapsed = time.time() - start_time
        quantum_subgradient = subgrad_result.subgradient

        subgrad_results = {
            'subgradient':                    quantum_subgradient.tolist(),
            'subgradient_norm':               float(np.linalg.norm(quantum_subgradient)),
            'tail_probability_iqae':          float(subgrad_result.tail_probability_iqae),
            'conditional_expectations_iqae':  subgrad_result.conditional_expectations_iqae.tolist(),
            'total_oracle_queries_iqae':      int(subgrad_result.total_oracle_queries_iqae),
            'iqae_converged':                 bool(subgrad_result.iqae_converged),
            'tail_probability_woerner':       float(subgrad_result.tail_probability_woerner),
            'conditional_expectations_woerner': subgrad_result.conditional_expectations_woerner.tolist(),
            'total_oracle_queries_woerner':   int(subgrad_result.total_oracle_queries_woerner),
            'woerner_converged':              bool(subgrad_result.woerner_converged),
            'classical_var':                  float(subgrad_result.classical_var),
            'classical_cvar':                 float(subgrad_result.classical_cvar),
            'classical_tail_prob':            float(subgrad_result.classical_tail_prob),
            'classical_subgradient':          subgrad_result.classical_subgradient.tolist(),
            'tail_prob_error_iqae':           float(subgrad_result.tail_prob_error_iqae),
            'tail_prob_error_woerner':        float(subgrad_result.tail_prob_error_woerner),
            'subgradient_l2_error':           float(subgrad_result.subgradient_l2_error),
            'subgradient_cosine_similarity':  float(subgrad_result.subgradient_cosine_similarity),
            'num_assets':                     int(subgrad_result.num_assets),
            'num_periods':                    int(subgrad_result.num_periods),
            'alpha':                          float(subgrad_result.alpha),
            'epsilon':                        float(subgrad_result.epsilon),
            'num_distribution_qubits':        int(subgrad_result.num_distribution_qubits),
            'execution_time_ms':              float(subgrad_result.execution_time_ms),
            'execution_time_s':               elapsed,
            'method':                         subgrad_result.method,
        }

        logger.info(f"Quantum subgradient complete: norm={subgrad_results['subgradient_norm']:.6f}, "
                    f"cosine={subgrad_results['subgradient_cosine_similarity']:.4f}, time={elapsed:.2f}s")

    except Exception as e:
        logger.error(f"Quantum subgradient failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        subgrad_results = {'error': str(e)}
        quantum_subgradient = classical_subgradient.copy()

else:
    logger.warning("Quantum subgradient not available, using classical fallback")
    quantum_subgradient = classical_subgradient.copy()
    subgrad_results = {
        'subgradient':                   classical_subgradient.tolist(),
        'subgradient_norm':              float(np.linalg.norm(classical_subgradient)),
        'tail_probability_iqae':         0.0,
        'tail_probability_woerner':      0.0,
        'total_oracle_queries_iqae':     0,
        'total_oracle_queries_woerner':  0,
        'iqae_converged':                False,
        'woerner_converged':             False,
        'classical_tail_prob':           classical_subgrad_info['tail_prob'],
        'classical_subgradient':         classical_subgradient.tolist(),
        'subgradient_l2_error':          0.0,
        'subgradient_cosine_similarity': 1.0,
        'execution_time_ms':             0,
        'execution_time_s':              0,
        'method':                        'classical_fallback',
        'fallback':                      True,
    }

# ---------------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------------
validation_passed = True
cos_sim = subgrad_results.get('subgradient_cosine_similarity', 1.0)
if SUBGRAD_VALIDATE and cos_sim < (1 - SUBGRAD_TOLERANCE):
    validation_passed = False
    logger.warning(f"Validation: cosine similarity {cos_sim:.4f} < threshold {1-SUBGRAD_TOLERANCE:.4f}")

# ---------------------------------------------------------------------------
# Passport checkpoint
# ---------------------------------------------------------------------------
if orchestrator and master_passport:
    master_passport = orchestrator.checkpoint(
        master_passport,
        name='phase2_subgradient_estimation',
        data={'subgradient_norm': subgrad_results.get('subgradient_norm', 0)},
        metadata={
            'method': subgrad_results.get('method', 'unknown'),
            'total_queries_iqae': subgrad_results.get('total_oracle_queries_iqae', 0),
            'total_queries_woerner': subgrad_results.get('total_oracle_queries_woerner', 0),
            'validation_passed': validation_passed,
        }
    )

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("=" * 70)
print("QUANTUM SUBGRADIENT ESTIMATION")
print("=" * 70)
print(f"  Method: {subgrad_results.get('method', 'unknown')}")
print(f"  Subgradient norm: {subgrad_results.get('subgradient_norm', 0):.6f} "
      f"(classical: {np.linalg.norm(classical_subgradient):.6f})")
print(f"  Cosine similarity: {cos_sim:.4f}  |  L2 error: {subgrad_results.get('subgradient_l2_error', 0):.6f}")
print(f"  Tail prob: IQAE={subgrad_results.get('tail_probability_iqae', 0):.6f}, "
      f"Woerner={subgrad_results.get('tail_probability_woerner', 0):.6f}, "
      f"Classical={classical_subgrad_info['tail_prob']:.6f}")
print(f"  Queries: IQAE={subgrad_results.get('total_oracle_queries_iqae', 0)}, "
      f"Woerner={subgrad_results.get('total_oracle_queries_woerner', 0)}")
print(f"  Validation: {'PASS' if validation_passed else 'FAIL'} (tol={SUBGRAD_TOLERANCE})")
print(f"  Time: {subgrad_results.get('execution_time_s', 0):.2f}s")
print("=" * 70)

QUANTUM SUBGRADIENT ESTIMATION
  Method: iqae
  Subgradient norm: 0.155284 (classical: 3.048642)
  Cosine similarity: 1.0000  |  L2 error: 0.000712
  Tail prob: IQAE=0.050769, Woerner=0.000000, Classical=0.050746
  Queries: IQAE=1289216, Woerner=0
  Validation: PASS (tol=0.2)
  Time: 92.74s


In [15]:
# ============================================================================
# CELL 14: HYBRID OPTIMIZER (hybridoptimizer.py)
# ============================================================================
# Hybrid quantum-classical optimization to minimize CVaR.
# Supports recursive hierarchical optimization for large portfolios.
# IMPORTANT: The module works in LOSS convention (positive = bad).
# ============================================================================

import time

from hybridoptimizer import (
    HybridOptimizer,
    OptimizationResult,
    IterationMetrics,
    create_optimizer_for_node,
    QUANTUM_AVAILABLE as HYBRID_QUANTUM_AVAILABLE,
)

# ---------------------------------------------------------------------------
# Optimizer parameters (shared by standard and recursive paths)
# ---------------------------------------------------------------------------
opt_config = phase2_config.get('optimizer', {})
qae_config = phase2_config.get('qae', {})

OPTIMIZER_TYPE = opt_config.get('type', 'adam')
MAX_ITERATIONS = int(opt_config.get('max_iterations', 100))
LEARNING_RATE  = float(opt_config.get('learning_rate', 0.01))
LR_DECAY       = float(opt_config.get('learning_rate_decay', 0.995))
CONV_THRESHOLD = float(opt_config.get('convergence', {}).get('threshold', 1e-6))
PATIENCE       = int(opt_config.get('early_stopping', {}).get('patience', 20))

OPT_NUM_QUBITS  = int(qae_config.get('num_distribution_qubits', NUM_DIST_QUBITS))
OPT_EPSILON     = float(qae_config.get('epsilon', EPSILON))

methods_config  = qae_config.get('methods', {})
OPT_USE_IQAE    = methods_config.get('use_iqae', USE_IQAE)
OPT_USE_WOERNER = methods_config.get('use_woerner', USE_WOERNER)

# ZNE and circuit export
subgrad_config = phase2_config.get('subgradient', {})
zne_config     = phase2_config.get('zne', {})
OPT_USE_ZNE    = zne_config.get('enabled', False) and subgrad_config.get('use_zne', False)
OPT_ZNE_SCALE_FACTORS = zne_config.get('scale_factors', [1.0, 1.5, 2.0, 2.5]) if OPT_USE_ZNE else None
OPT_ZNE_EXTRAPOLATION = zne_config.get('extrapolation_method', 'richardson')

circuit_export_config = qae_config.get('circuit_export', {})
OPT_EXPORT_CIRCUITS   = circuit_export_config.get('enabled', False)
OPT_CIRCUIT_DIR       = circuit_export_config.get('directory', 'results/circuits')

exec_config = phase2_config.get('execution', {})
OPT_SHOTS   = int(exec_config.get('shots', phase2_config.get('backend', {}).get('default_shots', 8192)))

# Noise model for optimizer's internal QAE
noise_model_raw_opt = phase2_config.get('backend', {}).get('noise_model', {})
OPT_NOISE_MODEL_CONFIG = None
if noise_model_raw_opt.get('enabled', False):
    OPT_NOISE_MODEL_CONFIG = {
        'enabled': True,
        'single_qubit_gate_error': float(noise_model_raw_opt.get('single_qubit_gate_error', 0.001)),
        'two_qubit_gate_error':    float(noise_model_raw_opt.get('two_qubit_gate_error', 0.01)),
        'readout_error':           float(noise_model_raw_opt.get('readout_error', 0.02)),
    }

# Weight bounds
bounds_config  = opt_config.get('bounds', {})
OPT_MIN_WEIGHT = float(bounds_config.get('min_weight', MIN_WEIGHT))
OPT_MAX_WEIGHT = float(bounds_config.get('max_weight', MAX_WEIGHT))

# ---------------------------------------------------------------------------
# Recursive optimizer (optional)
# ---------------------------------------------------------------------------
RECURSIVE_MODULE = 'not_available'
recursive_config   = phase2_config.get('recursive_optimizer', {})
RECURSIVE_ENABLED  = recursive_config.get('enabled', False)
RECURSIVE_THRESHOLD = recursive_config.get('auto_enable_threshold', 5)

clustering_config       = phase0_config.get('recursive_clustering', {})
RECURSIVE_MAX_CLUSTER   = clustering_config.get('max_cluster_size', 5)
RECURSIVE_MIN_CLUSTER   = clustering_config.get('min_cluster_size', 2)
RECURSIVE_CLUSTER_METHOD = clustering_config.get('clustering_method', 'correlation')

try:
    from recursive_optimizer import (
        RecursivePortfolioOptimizer, RecursiveOptimizationResult,
        RecursiveConfig, create_recursive_optimizer, optimize_large_portfolio,
    )
    RECURSIVE_MODULE = 'recursive_optimizer'
except ImportError:
    pass

USE_RECURSIVE = (
    RECURSIVE_ENABLED
    and RECURSIVE_MODULE != 'not_available'
    and n_effective > RECURSIVE_THRESHOLD
)

# ---------------------------------------------------------------------------
# Effective ticker labels
# ---------------------------------------------------------------------------
effective_tickers = (
    tickers if len(tickers) == n_effective
    else [f"Asset_{i}" for i in range(n_effective)]
)

# ---------------------------------------------------------------------------
# VaR / CVaR in LOSS convention (positive = loss, minimize)
# ---------------------------------------------------------------------------

def compute_var_for_optimizer(weights, returns, alpha):
    """VaR in loss convention: (1-alpha) percentile of losses."""
    portfolio_losses = -(np.asarray(returns) @ np.asarray(weights).flatten())
    return float(np.percentile(portfolio_losses, (1 - alpha) * 100))

def compute_cvar_for_optimizer(weights, returns, alpha):
    """CVaR in loss convention: E[Loss | Loss >= VaR]."""
    portfolio_losses = -(np.asarray(returns) @ np.asarray(weights).flatten())
    var_loss = np.percentile(portfolio_losses, (1 - alpha) * 100)
    tail = portfolio_losses[portfolio_losses >= var_loss]
    return float(np.mean(tail)) if len(tail) > 0 else float(var_loss)

initial_cvar_loss = compute_cvar_for_optimizer(initial_weights, returns_effective, ALPHA)
initial_var_loss  = compute_var_for_optimizer(initial_weights, returns_effective, ALPHA)

logger.info("=" * 70)
logger.info("HYBRID QUANTUM-CLASSICAL OPTIMIZATION")
logger.info("=" * 70)
logger.info(f"Optimizer={OPTIMIZER_TYPE}, iters={MAX_ITERATIONS}, lr={LEARNING_RATE}, recursive={USE_RECURSIVE}")

print("=" * 70)
print("HYBRID QUANTUM-CLASSICAL CVaR OPTIMIZATION")
print("=" * 70)
print(f"  Optimizer: {OPTIMIZER_TYPE}  |  iters={MAX_ITERATIONS}  |  lr={LEARNING_RATE}")
print(f"  QAE: qubits={OPT_NUM_QUBITS}, eps={OPT_EPSILON}, shots={OPT_SHOTS}")
print(f"  ZNE: {OPT_USE_ZNE}  |  Recursive: {USE_RECURSIVE}")
print(f"  Initial CVaR (loss): {initial_cvar_loss:.6f}  |  (return): {cvar_classical:.6f}")

# ============================================================================
# PATH 1: RECURSIVE OPTIMIZATION (large portfolios)
# ============================================================================

opt_results = None
optimal_cluster_weights = None
recursive_result = None

if USE_RECURSIVE:
    try:
        start_time = time.time()

        rec_config = RecursiveConfig(
            max_cluster_size=RECURSIVE_MAX_CLUSTER,
            min_cluster_size=RECURSIVE_MIN_CLUSTER,
            alpha=ALPHA,
            clustering_method=RECURSIVE_CLUSTER_METHOD,
            linkage_method='ward',
            num_qubits=OPT_NUM_QUBITS,
            epsilon=OPT_EPSILON,
            max_iterations=MAX_ITERATIONS,
            learning_rate=LEARNING_RATE,
            use_quantum=HYBRID_QUANTUM_AVAILABLE and OPT_USE_IQAE,
        )

        optimizer_factory = None
        if HYBRID_QUANTUM_AVAILABLE and OPT_USE_IQAE:
            def factory(n_assets: int, node_id: str = "recursive"):
                return create_optimizer_for_node(
                    n_assets=n_assets, node_id=node_id,
                    asset_labels=[f"C{i}" for i in range(n_assets)],
                    alpha=ALPHA,
                    num_qubits=OPT_NUM_QUBITS, epsilon=OPT_EPSILON,
                    shots=OPT_SHOTS, use_quantum=True,
                    use_woerner=OPT_USE_WOERNER,
                    max_iterations=MAX_ITERATIONS,
                    learning_rate=LEARNING_RATE,
                    learning_rate_decay=LR_DECAY,
                    convergence_threshold=CONV_THRESHOLD,
                    patience=PATIENCE,
                    use_softmax=True, softmax_temperature=1.0,
                    weight_bounds=(OPT_MIN_WEIGHT, OPT_MAX_WEIGHT),
                    use_zne=OPT_USE_ZNE,
                    zne_scale_factors=OPT_ZNE_SCALE_FACTORS,
                    export_circuits=OPT_EXPORT_CIRCUITS,
                    circuit_export_dir=OPT_CIRCUIT_DIR,
                    passport=master_passport if orchestrator else None,
                )
            optimizer_factory = factory

        recursive_optimizer = RecursivePortfolioOptimizer(rec_config, optimizer_factory)

        recursive_result = recursive_optimizer.optimize(
            asset_names=effective_tickers,
            returns=returns_effective,
        )

        elapsed = time.time() - start_time

        optimal_cluster_weights = np.array([
            recursive_result.final_weights[t] for t in effective_tickers
        ])

        final_cvar_loss = recursive_result.final_cvar
        final_var_loss  = recursive_result.final_var
        cvar_improvement_pct = recursive_result.improvement * 100

        opt_results = {
            'optimal_weights':     optimal_cluster_weights.tolist(),
            'optimal_cvar_loss':   float(final_cvar_loss),
            'optimal_var_loss':    float(final_var_loss),
            'initial_cvar_loss':   float(recursive_result.initial_cvar),
            'initial_var_loss':    float(initial_var_loss),
            'optimal_cvar_return': float(-final_cvar_loss),
            'optimal_var_return':  float(-final_var_loss),
            'initial_cvar_return': float(cvar_classical),
            'initial_var_return':  float(var_classical),
            'cvar_improvement_pct':    float(cvar_improvement_pct),
            'cvar_reduction_absolute': float(recursive_result.initial_cvar - final_cvar_loss),
            'converged':           recursive_result.success,
            'num_iterations':      recursive_result.total_iterations,
            'early_stopped':       False,
            'early_stop_iteration': None,
            'final_gradient_norm': 0.0,
            'total_queries_iqae':    recursive_result.total_queries_iqae,
            'total_queries_woerner': recursive_result.total_queries_woerner,
            'total_queries':         recursive_result.total_queries_iqae + recursive_result.total_queries_woerner,
            'quantum_success_rate':  1.0 if recursive_result.success else 0.0,
            'avg_subgradient_error': 0.0,
            'total_time_seconds':     float(recursive_result.total_time_s),
            'total_quantum_time_ms':  0.0,
            'avg_iteration_time_ms':  float(recursive_result.total_time_s * 1000 / max(1, recursive_result.total_iterations)),
            'execution_time_s':       elapsed,
            'cvar_history_loss':   recursive_result.cvar_history,
            'cvar_history_return': [-c for c in recursive_result.cvar_history],
            'gradient_norms':      [],
            'iteration_details':   [],
            'recursive': {
                'tree_depth':     recursive_result.tree_depth,
                'total_nodes':    recursive_result.total_nodes,
                'total_leaves':   recursive_result.total_leaves,
                'per_level_cvar': recursive_result.per_level_cvar,
                'final_weights_dict': recursive_result.final_weights,
            },
            'config': {
                'optimizer_type': 'recursive',
                'max_iterations': MAX_ITERATIONS, 'learning_rate': LEARNING_RATE,
                'epsilon': OPT_EPSILON, 'num_qubits': OPT_NUM_QUBITS,
                'shots': OPT_SHOTS, 'use_woerner': OPT_USE_WOERNER,
                'use_zne': OPT_USE_ZNE, 'max_cluster_size': RECURSIVE_MAX_CLUSTER,
                'clustering_method': RECURSIVE_CLUSTER_METHOD, 'convention': 'loss',
            },
            'method': 'recursive',
        }

        logger.info(f"Recursive complete: CVaR {recursive_result.initial_cvar:.6f} -> {final_cvar_loss:.6f} ({cvar_improvement_pct:.2f}%)")

        print(f"\n  Recursive: CVaR (loss) {recursive_result.initial_cvar:.6f} -> {final_cvar_loss:.6f} ({cvar_improvement_pct:+.2f}%)")
        print(f"  Tree: depth={recursive_result.tree_depth}, nodes={recursive_result.total_nodes}")
        print(f"  Queries: IQAE={recursive_result.total_queries_iqae}, Woerner={recursive_result.total_queries_woerner}")
        print(f"  Time: {elapsed:.2f}s")

    except Exception as e:
        logger.error(f"Recursive optimization failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"\nERROR: Recursive optimization failed - {e}")
        print("Falling back to standard HybridOptimizer...")
        USE_RECURSIVE = False
        opt_results = None

# ============================================================================
# PATH 2: STANDARD HYBRID OPTIMIZATION
# ============================================================================

if not USE_RECURSIVE or opt_results is None:
    if HYBRID_QUANTUM_AVAILABLE:
        try:
            hybrid_optimizer = HybridOptimizer(
                n_assets=n_effective, alpha=ALPHA,
                max_iterations=MAX_ITERATIONS,
                learning_rate=LEARNING_RATE,
                learning_rate_decay=LR_DECAY,
                convergence_threshold=CONV_THRESHOLD,
                patience=PATIENCE,
                optimizer_type=OPTIMIZER_TYPE,
                num_distribution_qubits=OPT_NUM_QUBITS,
                epsilon=OPT_EPSILON,
                use_iqae=OPT_USE_IQAE, use_woerner=OPT_USE_WOERNER,
                weight_bounds=(OPT_MIN_WEIGHT, OPT_MAX_WEIGHT),
                backend='aer_simulator',
                passport=master_passport if orchestrator else None,
                use_zne=OPT_USE_ZNE,
                zne_scale_factors=OPT_ZNE_SCALE_FACTORS,
                export_circuits=OPT_EXPORT_CIRCUITS,
                circuit_export_dir=OPT_CIRCUIT_DIR,
                shots=OPT_SHOTS,
                noise_model_config=OPT_NOISE_MODEL_CONFIG,
            )

            start_time = time.time()

            opt_result = hybrid_optimizer.optimize(
                returns=returns_effective,
                compute_var_func=compute_var_for_optimizer,
                compute_cvar_func=compute_cvar_for_optimizer,
                initial_weights=initial_weights,
            )

            elapsed = time.time() - start_time

            optimal_cluster_weights = opt_result.optimal_weights.copy()
            final_cvar_loss   = opt_result.optimal_cvar
            final_var_loss    = opt_result.optimal_var
            cvar_improvement_pct = (initial_cvar_loss - final_cvar_loss) / initial_cvar_loss * 100

            iteration_details = [
                {
                    'iteration': int(im.iteration),
                    'cvar_loss': float(im.cvar), 'cvar_return': float(-im.cvar),
                    'var_loss': float(im.var), 'gradient_norm': float(im.gradient_norm),
                    'weight_change_norm': float(im.weight_change_norm),
                    'learning_rate': float(im.learning_rate),
                    'queries_iqae': int(im.queries_iqae),
                    'queries_woerner': int(im.queries_woerner),
                }
                for im in opt_result.iteration_history
            ]

            opt_results = {
                'optimal_weights':     optimal_cluster_weights.tolist(),
                'optimal_cvar_loss':   float(final_cvar_loss),
                'optimal_var_loss':    float(final_var_loss),
                'initial_cvar_loss':   float(initial_cvar_loss),
                'initial_var_loss':    float(initial_var_loss),
                'optimal_cvar_return': float(-final_cvar_loss),
                'optimal_var_return':  float(-final_var_loss),
                'initial_cvar_return': float(cvar_classical),
                'initial_var_return':  float(var_classical),
                'cvar_improvement_pct':    float(cvar_improvement_pct),
                'cvar_reduction_absolute': float(initial_cvar_loss - final_cvar_loss),
                'converged':            bool(opt_result.converged),
                'num_iterations':       int(opt_result.num_iterations),
                'early_stopped':        bool(opt_result.early_stopped),
                'early_stop_iteration': int(opt_result.early_stop_iteration) if opt_result.early_stop_iteration else None,
                'final_gradient_norm':  float(opt_result.final_gradient_norm),
                'total_queries_iqae':    int(opt_result.total_queries_iqae),
                'total_queries_woerner': int(opt_result.total_queries_woerner),
                'total_queries':         int(opt_result.total_queries_iqae + opt_result.total_queries_woerner),
                'quantum_success_rate':  float(opt_result.quantum_success_rate),
                'avg_subgradient_error': float(opt_result.avg_subgradient_error),
                'total_time_seconds':     float(opt_result.total_time_seconds),
                'total_quantum_time_ms':  float(opt_result.total_quantum_time_ms),
                'avg_iteration_time_ms':  float(opt_result.avg_iteration_time_ms),
                'execution_time_s':       elapsed,
                'cvar_history_loss':   [float(c) for c in opt_result.cvar_history],
                'cvar_history_return': [float(-c) for c in opt_result.cvar_history],
                'gradient_norms':      [float(im.gradient_norm) for im in opt_result.iteration_history],
                'iteration_details':   iteration_details,
                'config': {
                    'optimizer_type': OPTIMIZER_TYPE,
                    'max_iterations': MAX_ITERATIONS, 'learning_rate': LEARNING_RATE,
                    'use_zne': OPT_USE_ZNE, 'convention': 'loss',
                },
                'method': 'hybrid',
            }

            logger.info(f"Optimization complete: CVaR {initial_cvar_loss:.6f} -> {final_cvar_loss:.6f} ({cvar_improvement_pct:.2f}%)")
            print(opt_result.summary())

        except Exception as e:
            logger.error(f"HybridOptimizer failed: {e}")
            import traceback
            logger.error(traceback.format_exc())
            print(f"\nERROR: HybridOptimizer failed - {e}")

# ============================================================================
# FALLBACK: Classical subgradient with Adam
# ============================================================================

if opt_results is None:
    logger.warning("Using fallback classical optimizer")

    beta1, beta2, adam_eps = 0.9, 0.999, 1e-8
    weights = initial_weights.copy().astype(np.float64)
    best_weights, best_cvar_loss = weights.copy(), initial_cvar_loss
    m = np.zeros(n_effective, dtype=np.float64)
    v = np.zeros(n_effective, dtype=np.float64)
    cvar_history_loss, gradient_norms = [best_cvar_loss], []
    patience_counter = 0

    start_time = time.time()

    for it in range(MAX_ITERATIONS):
        lr = LEARNING_RATE * (LR_DECAY ** it)
        cvar_current = compute_cvar_for_optimizer(weights, returns_effective, ALPHA)
        var_current  = compute_var_for_optimizer(weights, returns_effective, ALPHA)

        portfolio_losses = -(returns_effective @ weights)
        tail_mask = portfolio_losses >= var_current
        subgrad = (-np.mean(returns_effective[tail_mask], axis=0) / ALPHA
                   if np.sum(tail_mask) > 0 else np.zeros(n_effective))

        gradient_norms.append(np.linalg.norm(subgrad))

        m = beta1 * m + (1 - beta1) * subgrad
        v = beta2 * v + (1 - beta2) * (subgrad ** 2)
        m_hat = m / (1 - beta1 ** (it + 1))
        v_hat = v / (1 - beta2 ** (it + 1))
        weights_new = weights - lr * m_hat / (np.sqrt(v_hat) + adam_eps)
        weights_new = np.clip(weights_new, OPT_MIN_WEIGHT, OPT_MAX_WEIGHT)
        weights_new /= weights_new.sum()
        weights = weights_new

        cvar_history_loss.append(cvar_current)

        if cvar_current < best_cvar_loss:
            best_cvar_loss, best_weights = cvar_current, weights.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE or np.linalg.norm(subgrad) < CONV_THRESHOLD:
            break

    elapsed = time.time() - start_time
    optimal_cluster_weights = best_weights
    final_cvar_loss = compute_cvar_for_optimizer(best_weights, returns_effective, ALPHA)
    cvar_improvement_pct = (initial_cvar_loss - final_cvar_loss) / initial_cvar_loss * 100

    opt_results = {
        'optimal_weights': optimal_cluster_weights.tolist(),
        'optimal_cvar_loss': float(final_cvar_loss),
        'initial_cvar_loss': float(initial_cvar_loss),
        'cvar_improvement_pct': float(cvar_improvement_pct),
        'converged': patience_counter < PATIENCE,
        'num_iterations': it + 1,
        'cvar_history_loss': cvar_history_loss,
        'gradient_norms': gradient_norms,
        'execution_time_s': elapsed,
        'total_queries_iqae': 0, 'total_queries_woerner': 0, 'total_queries': 0,
        'method': 'fallback',
    }

    print(f"\n  Fallback classical: CVaR {initial_cvar_loss:.6f} -> {final_cvar_loss:.6f} ({cvar_improvement_pct:+.2f}%), iters={it+1}")

# ---------------------------------------------------------------------------
# Passport checkpoint
# ---------------------------------------------------------------------------
if orchestrator and master_passport and opt_results:
    master_passport = orchestrator.checkpoint(
        master_passport,
        name='phase2_optimization',
        data={
            'optimal_cvar_loss': opt_results.get('optimal_cvar_loss', 0),
            'cvar_improvement_pct': opt_results.get('cvar_improvement_pct', 0),
            'method': opt_results.get('method', 'unknown'),
        },
        metadata={
            'num_iterations': opt_results.get('num_iterations', 0),
            'use_recursive': USE_RECURSIVE,
        }
    )

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("OPTIMIZATION SUMMARY")
print("=" * 70)
print(f"  Method:      {opt_results.get('method', 'unknown')}")
print(f"  CVaR (loss): {opt_results.get('initial_cvar_loss', 0):.6f} -> {opt_results.get('optimal_cvar_loss', 0):.6f}")
print(f"  Improvement: {opt_results.get('cvar_improvement_pct', 0):.2f}%")
print(f"  Iterations:  {opt_results.get('num_iterations', 0)}")
print(f"  Queries:     {opt_results.get('total_queries', 0)}")
print(f"  Time:        {opt_results.get('execution_time_s', 0):.2f}s")
if USE_RECURSIVE and recursive_result:
    print(f"  Tree:        depth={recursive_result.tree_depth}, nodes={recursive_result.total_nodes}")
print("=" * 70)

HYBRID QUANTUM-CLASSICAL CVaR OPTIMIZATION
  Optimizer: adam  |  iters=50  |  lr=0.05
  QAE: qubits=4, eps=0.002, shots=4096
  ZNE: False  |  Recursive: False
  Initial CVaR (loss): 0.047654  |  (return): -0.047654


HYBRID QUANTUM-CLASSICAL OPTIMIZATION RESULT [SOFTMAX]
CVaR: 0.047654 -> 0.038343 (+19.54%)
VaR:  0.028764 -> 0.022009
Iterations: 37 (converged=True (relative_cvar))
Total IQAE queries: 41758720
Total Woerner queries: 2368
Quantum success rate: 100.00%
Total time: 4502.25s

OPTIMIZATION SUMMARY
  Method:      hybrid
  CVaR (loss): 0.047654 -> 0.038343
  Improvement: 19.54%
  Iterations:  37
  Queries:     41761088
  Time:        4502.25s


In [16]:
# ============================================================================
# CELL 15: EVaR ESTIMATION (evar_estimation.py)
# ============================================================================
# Entropic Value at Risk (EVaR) as coherent risk measure.
# Mathematical property: EVaR >= CVaR >= VaR always holds.
# ============================================================================

import time

from evar_estimation import EVaREstimator

logger.info("=" * 70)
logger.info("EVaR ESTIMATION")
logger.info("=" * 70)

# ---------------------------------------------------------------------------
# Portfolio losses (positive = loss) for optimal and initial weights
# ---------------------------------------------------------------------------
optimal_portfolio_losses = -(returns_effective @ optimal_cluster_weights)
initial_portfolio_losses = -(returns_effective @ initial_weights)

# ---------------------------------------------------------------------------
# Compute EVaR
# ---------------------------------------------------------------------------
evar_results = None

try:
    start_time = time.time()
    evar_estimator = EVaREstimator(alpha=ALPHA)

    evar_result_optimal = evar_estimator.compute_evar(
        losses=optimal_portfolio_losses,
        passport=master_passport if orchestrator else None,
    )

    evar_result_initial = evar_estimator.compute_evar(
        losses=initial_portfolio_losses,
        passport=None,
    )

    comparison_result = evar_estimator.compare_evar_cvar(
        losses=optimal_portfolio_losses, passport=None,
    )

    validation_result = evar_estimator.validate_optimization_with_evar(
        optimal_weights=optimal_cluster_weights,
        returns=returns_effective,
        cvar_optimized=opt_results['optimal_cvar_loss'],
        passport=None,
    )

    elapsed = time.time() - start_time

    # -------------------------------------------------------------------
    # Compile results
    # -------------------------------------------------------------------
    evar_results = {
        'optimal': {
            'evar':           float(evar_result_optimal['evar']),
            'cvar':           float(evar_result_optimal['cvar']),
            'var':            float(evar_result_optimal['var']),
            'evar_cvar_ratio': float(evar_result_optimal['evar_cvar_ratio']),
            'lambda_optimal': float(evar_result_optimal['lambda_optimal']),
            'mgf_optimal':    float(evar_result_optimal['mgf_optimal']),
            'num_samples':    int(evar_result_optimal['num_samples']),
        },
        'initial': {
            'evar':           float(evar_result_initial['evar']),
            'cvar':           float(evar_result_initial['cvar']),
            'var':            float(evar_result_initial['var']),
            'evar_cvar_ratio': float(evar_result_initial['evar_cvar_ratio']),
            'lambda_optimal': float(evar_result_initial['lambda_optimal']),
        },
        'evar_improvement': {
            'initial_evar':  float(evar_result_initial['evar']),
            'final_evar':    float(evar_result_optimal['evar']),
            'reduction':     float(evar_result_initial['evar'] - evar_result_optimal['evar']),
            'reduction_pct': (float((evar_result_initial['evar'] - evar_result_optimal['evar'])
                                    / evar_result_initial['evar'] * 100)
                              if evar_result_initial['evar'] != 0 else 0.0),
        },
        'comparison': {
            'evar_value':         float(comparison_result['evar_value']),
            'cvar_value':         float(comparison_result['cvar_value']),
            'var_value':          float(comparison_result['var_value']),
            'difference':         float(comparison_result['difference']),
            'ratio':              float(comparison_result['ratio']),
            'validation_passed':  bool(comparison_result['validation_passed']),
        },
        'validation': {
            'status':         validation_result.get('validation_status', 'unknown'),
            'interpretation': validation_result.get('interpretation', ''),
            'evar_value':     float(validation_result.get('evar_value', 0)),
            'cvar_optimized': float(validation_result.get('cvar_optimized', 0)),
            'ratio':          float(validation_result.get('ratio', 0)),
        },
        'alpha':           float(ALPHA),
        'risk_aversion':   float(evar_estimator.risk_aversion),
        'execution_time_s': elapsed,
    }

    # Mathematical validity: EVaR >= CVaR >= VaR
    evar_opt = evar_results['optimal']['evar']
    cvar_opt = evar_results['optimal']['cvar']
    var_opt  = evar_results['optimal']['var']
    math_valid = (evar_opt >= cvar_opt - 1e-6) and (cvar_opt >= var_opt - 1e-6)
    evar_results['mathematical_validity'] = math_valid

    logger.info(f"EVaR={evar_opt:.6f}, CVaR={cvar_opt:.6f}, VaR={var_opt:.6f}, "
                f"ratio={evar_results['optimal']['evar_cvar_ratio']:.4f}, valid={math_valid}")

except Exception as e:
    logger.error(f"EVaR estimation failed: {e}")
    import traceback
    logger.error(traceback.format_exc())
    evar_results = {'error': str(e)}

# ---------------------------------------------------------------------------
# Passport checkpoint
# ---------------------------------------------------------------------------
if orchestrator and master_passport and evar_results and 'error' not in evar_results:
    master_passport = orchestrator.checkpoint(
        master_passport,
        name='phase2_evar_estimation',
        data={
            'evar_optimal': evar_results['optimal']['evar'],
            'evar_cvar_ratio': evar_results['optimal']['evar_cvar_ratio'],
        },
        metadata={
            'alpha': ALPHA,
            'mathematical_validity': evar_results.get('mathematical_validity', False),
        }
    )

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("=" * 70)
print("EVaR ESTIMATION RESULTS")
print("=" * 70)

if evar_results and 'error' not in evar_results:
    opt_d  = evar_results['optimal']
    init_d = evar_results['initial']
    impr   = evar_results['evar_improvement']

    print(f"  Optimal:  EVaR={opt_d['evar']:.6f}  CVaR={opt_d['cvar']:.6f}  VaR={opt_d['var']:.6f}  (ratio={opt_d['evar_cvar_ratio']:.4f})")
    print(f"  Initial:  EVaR={init_d['evar']:.6f}  CVaR={init_d['cvar']:.6f}  VaR={init_d['var']:.6f}")
    print(f"  Improvement: {impr['reduction']:.6f} ({impr['reduction_pct']:+.2f}%)")
    print(f"  EVaR >= CVaR >= VaR: {evar_results.get('mathematical_validity', 'N/A')}")
    print(f"  Time: {evar_results['execution_time_s']:.2f}s")
else:
    print(f"  ERROR: {evar_results.get('error', 'unknown')}")

print("=" * 70)

EVaR ESTIMATION RESULTS
  Optimal:  EVaR=0.071909  CVaR=0.038343  VaR=0.022041  (ratio=1.8754)
  Initial:  EVaR=0.095640  CVaR=0.047654  VaR=0.028773
  Improvement: 0.023731 (+24.81%)
  EVaR >= CVaR >= VaR: True
  Time: 0.01s


In [17]:
# ============================================================================
# CELL 15: ERROR PROPAGATION (errorpropagation.py)
# ============================================================================
# Analyzes how errors propagate through the optimization pipeline.
# Error sources: Truncation, QAE, Noise, Sampling, Gradient.
# Compares errors between methods: IQAE, Woerner.
# ============================================================================

import time

from errorpropagation import (
    ErrorPropagationAnalyzer, FullErrorReport,
    TruncationErrorAnalysis, QAEErrorAnalysis,
    NoiseErrorAnalysis, PropagatedError, MethodComparison,
)

logger.info("=" * 70)
logger.info("ERROR PROPAGATION ANALYSIS")
logger.info("=" * 70)

# ---------------------------------------------------------------------------
# Extract QAE results for analysis
# ---------------------------------------------------------------------------

def _extract_qae_method(qae_results, cvar_qae_result, method_name):
    """Extract QAE method result as object with standard attributes."""
    # Priority 1: direct object from cvar_qae_result
    if cvar_qae_result is not None:
        attr = f"{method_name}_result"
        obj = getattr(cvar_qae_result, attr, None)
        if obj is not None:
            return obj

    if qae_results is None:
        return None

    # Priority 2: method_results dict
    method_results = qae_results.get('method_results', {})
    if isinstance(method_results, dict):
        # Direct key (e.g. 'iqae')
        if method_name in method_results and isinstance(method_results[method_name], dict):
            return _dict_to_qae_obj(method_results[method_name], qae_results)

        # Keyed by task (e.g. 'iqae_tail_prob')
        tail_key = f"{method_name}_tail_prob"
        el_key = f"{method_name}_E_L_tail"
        if tail_key in method_results:
            tail_data = method_results[tail_key]
            el_data = method_results.get(el_key, {})
            merged = {
                'num_oracle_queries': tail_data.get('queries', 0) + el_data.get('queries', 0),
                'converged': tail_data.get('converged', True) and el_data.get('converged', True),
                'amplitude': tail_data.get('amplitude', 0.0),
                'zne_applied': qae_results.get('zne_applied', False) if method_name == 'iqae' else False,
                'zne_improvement': qae_results.get('zne_improvement', 0.0) if method_name == 'iqae' else 0.0,
            }
            return _dict_to_qae_obj(merged, qae_results)

    # Priority 3: top-level flat keys
    queries_key = f"total_queries_{method_name}"
    if qae_results.get(queries_key, 0) > 0:
        return _dict_to_qae_obj({
            'num_oracle_queries': qae_results[queries_key],
            'converged': True,
            'amplitude': qae_results.get('quantum_tail_prob', 0.0),
            'zne_applied': qae_results.get('zne_applied', False) if method_name == 'iqae' else False,
            'zne_improvement': qae_results.get('zne_improvement', 0.0) if method_name == 'iqae' else 0.0,
        }, qae_results)

    return None


class _QAEResultAdapter:
    """Lightweight adapter: dict -> object with attributes for ErrorPropagationAnalyzer."""
    def __init__(self, data):
        self.num_oracle_queries = data.get('num_oracle_queries', 0)
        self.converged = data.get('converged', True)
        self.zne_applied = data.get('zne_applied', False)
        self.zne_improvement = data.get('zne_improvement', 0.0)
        self.amplitude = data.get('amplitude', 0.0)


def _dict_to_qae_obj(data, qae_results):
    return _QAEResultAdapter(data)


iqae_result_for_analysis = _extract_qae_method(
    qae_results, cvar_qae_result if 'cvar_qae_result' in dir() else None, 'iqae')
woerner_result_for_analysis = _extract_qae_method(
    qae_results, cvar_qae_result if 'cvar_qae_result' in dir() else None, 'woerner')

logger.info(f"Methods for error analysis: "
            f"IQAE={'yes' if iqae_result_for_analysis else 'no'}, "
            f"Woerner={'yes' if woerner_result_for_analysis else 'no'}")

# ---------------------------------------------------------------------------
# Run full error analysis
# ---------------------------------------------------------------------------
error_results = None

try:
    start_time = time.time()

    error_analyzer = ErrorPropagationAnalyzer(
        num_qubits=OPT_NUM_QUBITS, alpha=ALPHA, num_shots=SHOTS,
    )

    estimated_circuit_depth = OPT_NUM_QUBITS * 10
    estimated_num_gates = estimated_circuit_depth * OPT_NUM_QUBITS * 2

    noise_params_for_error = None
    if QAE_NOISE_MODEL_CONFIG is not None:
        noise_params_for_error = {
            'gate_error_rate': QAE_NOISE_MODEL_CONFIG.get('two_qubit_gate_error', 0.001),
            'measurement_error_rate': QAE_NOISE_MODEL_CONFIG.get('readout_error', 0.01),
        }

    error_report = error_analyzer.full_error_analysis(
        returns=returns_effective,
        optimal_weights=optimal_cluster_weights,
        num_iterations=opt_results.get('num_iterations', 0),
        learning_rate=LEARNING_RATE,
        epsilon=OPT_EPSILON,
        circuit_depth=estimated_circuit_depth,
        num_gates=estimated_num_gates,
        iqae_result=iqae_result_for_analysis,
        woerner_result=woerner_result_for_analysis,
        qsp_result=None,
        zne_enabled=qae_results.get('zne_applied', False) if qae_results else False,
        export_json=True,
        export_path=Path(RESULTS_DIR) if 'RESULTS_DIR' in dir() else None,
        noise_params=noise_params_for_error,
    )

    elapsed = time.time() - start_time

    # -----------------------------------------------------------------------
    # Compile results
    # -----------------------------------------------------------------------
    error_results = {
        'truncation': {
            'num_qubits': int(error_report.truncation.num_qubits),
            'num_bins': int(error_report.truncation.num_bins),
            'bin_width': float(error_report.truncation.bin_width),
            'max_error': float(error_report.truncation.max_error),
            'rms_error': float(error_report.truncation.rms_error),
            'relative_error': float(error_report.truncation.relative_error),
        },
        'qae_iqae': error_report.qae_iqae.to_dict() if error_report.qae_iqae else None,
        'qae_woerner': error_report.qae_woerner.to_dict() if error_report.qae_woerner else None,
        'noise': {
            'circuit_depth': int(error_report.noise.circuit_depth),
            'num_gates': int(error_report.noise.num_gates),
            'gate_error_rate': float(error_report.noise.gate_error_rate),
            'total_fidelity': float(error_report.noise.total_fidelity),
            'total_error': float(error_report.noise.total_error),
        },
        'propagation': {
            'initial_error': float(error_report.propagation.initial_error),
            'final_error': float(error_report.propagation.final_error),
            'amplification_factor': float(error_report.propagation.amplification_factor),
            'num_iterations': int(error_report.propagation.num_iterations),
            'dominant_source': error_report.propagation.dominant_source,
        },
        'method_comparison': {
            'iqae_error': float(error_report.method_comparison.iqae_error),
            'woerner_error': float(error_report.method_comparison.woerner_error),
            'qsp_error': float(getattr(error_report.method_comparison, 'qsp_error', 0.0)),
            'best_method': error_report.method_comparison.best_method,
            'best_error': float(error_report.method_comparison.best_error),
            'query_efficiency_ranking': error_report.method_comparison.query_efficiency_ranking,
        },
        'final_error_bound': float(error_report.final_error_bound),
        'estimated_cvar_error': float(error_report.estimated_cvar_error),
        'dominant_error_source': error_report.dominant_error_source,
        'total_oracle_queries': int(error_report.total_oracle_queries),
        'zne_enabled': bool(error_report.zne_enabled),
        'error_without_zne': float(error_report.error_without_zne),
        'error_with_zne': float(error_report.error_with_zne),
        'zne_improvement_percent': float(error_report.zne_improvement_percent),
        'num_assets': int(error_report.num_assets),
        'num_periods': int(error_report.num_periods),
        'alpha': float(error_report.alpha),
        'num_qubits': int(error_report.num_qubits),
        'execution_time_s': elapsed,
    }

    logger.info(f"Error analysis complete: bound={error_results['final_error_bound']:.6e}, "
                f"dominant={error_results['dominant_error_source']}, time={elapsed:.2f}s")

    print(error_report.summary())

except Exception as e:
    logger.error(f"Error propagation analysis failed: {e}")
    import traceback
    logger.error(traceback.format_exc())
    error_results = {'error': str(e)}

# ---------------------------------------------------------------------------
# Fallback: simple analytical estimates
# ---------------------------------------------------------------------------
if error_results is None or 'error' in error_results:
    logger.warning("Using fallback error estimates")
    num_bins = 2 ** OPT_NUM_QUBITS
    port_ret = returns_effective @ optimal_cluster_weights
    v_range = np.max(port_ret) - np.min(port_ret)
    trunc_err = (v_range / num_bins) / 2
    qae_err = OPT_EPSILON
    noise_err = 1 - (1 - 0.001) ** (OPT_NUM_QUBITS * 100)
    total_err = np.sqrt(trunc_err**2 + qae_err**2 + noise_err**2)

    error_results = {
        'truncation': {'num_qubits': OPT_NUM_QUBITS, 'num_bins': num_bins,
                       'max_error': float(trunc_err)},
        'final_error_bound': float(total_err),
        'estimated_cvar_error': float(total_err * v_range),
        'dominant_error_source': 'qae' if qae_err > trunc_err else 'truncation',
        'fallback': True,
    }
    print(f"  Fallback error bound: {total_err:.6e}")

# ---------------------------------------------------------------------------
# Passport checkpoint
# ---------------------------------------------------------------------------
if orchestrator and master_passport:
    master_passport = orchestrator.checkpoint(
        master_passport,
        name='phase2_error_propagation',
        data={
            'final_error_bound': error_results.get('final_error_bound', 0),
            'estimated_cvar_error': error_results.get('estimated_cvar_error', 0),
        },
        metadata={
            'dominant_source': error_results.get('dominant_error_source', 'unknown'),
        }
    )

  Fallback error bound: 3.299133e-01


In [18]:
# ============================================================================
# CELL 16: SUMMARY & EXPORT RESULTS
# ============================================================================

import json
from pathlib import Path

logger.info("=" * 70)
logger.info("PIPELINE SUMMARY & EXPORT")
logger.info("=" * 70)

# ---------------------------------------------------------------------------
# Helper: convert numpy types to native Python for JSON
# ---------------------------------------------------------------------------

def convert_to_native(obj):
    """Recursively convert numpy/Path/datetime types to JSON-safe Python types."""
    if obj is None:
        return None
    if isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [convert_to_native(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, Path):
        return str(obj)
    return obj

# ---------------------------------------------------------------------------
# Results directory
# ---------------------------------------------------------------------------
results_dir = PROJECT_ROOT / export_config.get('base_directory', 'results')

results_dir.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = str(results_dir)

# ---------------------------------------------------------------------------
# Compile pipeline state
# ---------------------------------------------------------------------------

pipeline_state = {
    'metadata': {
        'timestamp': datetime.now().isoformat(),
        'notebook': 'main.ipynb',
    },
    'configuration': {
        'alpha': float(ALPHA),
        'n_assets_original': n_assets,
        'n_effective': n_effective,
        'num_qubits': OPT_NUM_QUBITS,
        'epsilon': OPT_EPSILON,
        'optimizer_type': OPTIMIZER_TYPE,
        'max_iterations': MAX_ITERATIONS,
        'learning_rate': LEARNING_RATE,
        'recursive_enabled': RECURSIVE_ENABLED,
        'use_recursive': USE_RECURSIVE,
    },
    'baseline_return_convention': {
        'var': float(var_classical),
        'cvar': float(cvar_classical),
    },
    'baseline_loss_convention': {
        'var': float(opt_results.get('initial_var_loss', 0)) if opt_results else 0,
        'cvar': float(opt_results.get('initial_cvar_loss', 0)) if opt_results else 0,
    },
}

# Optimization results
if opt_results:
    pipeline_state['optimization'] = {
        'optimal_weights': opt_results.get('optimal_weights', []),
        'optimal_cvar_loss': float(opt_results.get('optimal_cvar_loss', 0)),
        'optimal_var_loss': float(opt_results.get('optimal_var_loss', 0)),
        'optimal_cvar_return': float(opt_results.get('optimal_cvar_return', 0)),
        'optimal_var_return': float(opt_results.get('optimal_var_return', 0)),
        'improvement_pct': float(opt_results.get('cvar_improvement_pct', 0)),
        'cvar_reduction_absolute': float(opt_results.get('cvar_reduction_absolute', 0)),
        'converged': bool(opt_results.get('converged', False)),
        'num_iterations': int(opt_results.get('num_iterations', 0)),
        'early_stopped': bool(opt_results.get('early_stopped', False)),
        'final_gradient_norm': float(opt_results.get('final_gradient_norm', 0)),
        'total_queries_iqae': int(opt_results.get('total_queries_iqae', 0)),
        'total_queries_woerner': int(opt_results.get('total_queries_woerner', 0)),
        'total_queries': int(opt_results.get('total_queries', 0)),
        'quantum_success_rate': float(opt_results.get('quantum_success_rate', 0)),
        'total_time_seconds': float(opt_results.get('total_time_seconds', 0)),
        'total_quantum_time_ms': float(opt_results.get('total_quantum_time_ms', 0)),
        'cvar_history': opt_results.get('cvar_history_loss', []),
        'gradient_norms': opt_results.get('gradient_norms', []),
        'method': opt_results.get('method', 'unknown'),
    }
    if opt_results.get('method') == 'recursive' and 'recursive' in opt_results:
        pipeline_state['optimization']['recursive'] = opt_results['recursive']

# QAE results
if qae_results and qae_results.get('success'):
    pipeline_state['qae'] = {
        'var_estimate': float(qae_results.get('quantum_var', 0)),
        'cvar_estimate': float(qae_results.get('quantum_cvar', 0)),
        'tail_probability': float(qae_results.get('quantum_tail_prob', 0)),
        'total_queries_iqae': int(qae_results.get('total_queries_iqae', 0)),
        'total_queries_woerner': int(qae_results.get('total_queries_woerner', 0)),
        'cvar_error': float(qae_results.get('cvar_error', 0)),
        'iqae_cvar_ideal': qae_results.get('iqae_cvar_ideal'),
        'iqae_cvar_noisy_raw': qae_results.get('iqae_cvar_noisy_raw'),
        'iqae_cvar_noisy_zne': qae_results.get('iqae_cvar_noisy_zne'),
    }

# Subgradient results
if 'subgrad_results' in dir() and subgrad_results:
    pipeline_state['subgradient'] = {
        'subgradient_norm': float(subgrad_results.get('subgradient_norm', 0)),
        'method': subgrad_results.get('method', 'unknown'),
        'cosine_similarity': float(subgrad_results.get('subgradient_cosine_similarity', 0)),
        'l2_error': float(subgrad_results.get('subgradient_l2_error', 0)),
    }

# EVaR results
if 'evar_results' in dir() and evar_results and not evar_results.get('error'):
    opt_evar = evar_results.get('optimal', {})
    pipeline_state['evar'] = {
        'evar': float(opt_evar.get('evar', 0)),
        'cvar': float(opt_evar.get('cvar', 0)),
        'evar_cvar_ratio': float(opt_evar.get('evar_cvar_ratio', 0)),
        'mathematical_validity': bool(evar_results.get('mathematical_validity', False)),
    }

# Error propagation results
if 'error_results' in dir() and error_results and not error_results.get('error'):
    pipeline_state['error_propagation'] = {
        'final_error_bound': float(error_results.get('final_error_bound', 0)),
        'estimated_cvar_error': float(error_results.get('estimated_cvar_error', 0)),
        'dominant_source': error_results.get('dominant_error_source', 'unknown'),
        'zne_enabled': bool(error_results.get('zne_enabled', False)),
        'zne_improvement_pct': float(error_results.get('zne_improvement_percent', 0)),
        'best_qae_method': error_results.get('method_comparison', {}).get('best_method', 'unknown'),
    }

pipeline_state = convert_to_native(pipeline_state)

# ---------------------------------------------------------------------------
# Export files
# ---------------------------------------------------------------------------

# 1. Main JSON
results_json_path = results_dir / 'phase2_results.json'
with open(results_json_path, 'w', encoding='utf-8') as f:
    json.dump(pipeline_state, f, indent=2, default=str)
logger.info(f"Results saved to {results_json_path}")

# 2. Convergence CSV
if opt_results:
    cvar_hist = opt_results.get('cvar_history_loss', [])
    grad_norms = opt_results.get('gradient_norms', [])
    max_len = max(len(cvar_hist), len(grad_norms), 1)
    # Pad shorter list
    grad_norms = grad_norms + [0.0] * (max_len - len(grad_norms))
    cvar_hist = cvar_hist + [cvar_hist[-1] if cvar_hist else 0.0] * (max_len - len(cvar_hist))

    conv_df = pd.DataFrame({
        'iteration': range(len(cvar_hist)),
        'cvar_loss': [float(c) for c in cvar_hist],
        'cvar_return': [float(-c) for c in cvar_hist],
        'gradient_norm': [float(g) for g in grad_norms[:len(cvar_hist)]],
    })
    conv_csv_path = results_dir / 'convergence.csv'
    conv_df.to_csv(conv_csv_path, index=False)
    logger.info(f"Convergence saved to {conv_csv_path}")

# 3. Optimal weights CSV
if optimal_cluster_weights is not None:
    labels = effective_tickers if len(effective_tickers) == len(optimal_cluster_weights) else \
             [f'asset_{i}' for i in range(len(optimal_cluster_weights))]
    weights_df = pd.DataFrame({'asset': labels, 'weight': [float(w) for w in optimal_cluster_weights]})
    weights_csv_path = results_dir / 'optimal_weights.csv'
    weights_df.to_csv(weights_csv_path, index=False)
    logger.info(f"Weights saved to {weights_csv_path}")

# 4. Recursive tree results (if applicable)
if 'recursive_result' in dir() and recursive_result is not None:
    recursive_export = convert_to_native({
        'tree_structure': {
            'tree_depth': recursive_result.tree_depth,
            'total_nodes': recursive_result.total_nodes,
            'total_leaves': recursive_result.total_leaves,
        },
        'optimization': {
            'initial_cvar': float(recursive_result.initial_cvar),
            'final_cvar': float(recursive_result.final_cvar),
            'improvement_pct': float(recursive_result.improvement * 100),
        },
        'per_level_cvar': [float(c) for c in recursive_result.per_level_cvar],
        'final_weights': recursive_result.final_weights,
    })
    recursive_json_path = results_dir / 'recursive_tree_results.json'
    with open(recursive_json_path, 'w', encoding='utf-8') as f:
        json.dump(recursive_export, f, indent=2, default=str)
    logger.info(f"Recursive results saved to {recursive_json_path}")

# 5. Passport export
if orchestrator and master_passport:
    passport_dir = results_dir / 'passports'
    passport_dir.mkdir(parents=True, exist_ok=True)
    passport_path = passport_dir / 'pipeline_passport.json'
    try:
        orchestrator.export_passport(master_passport, str(passport_path))
        logger.info(f"Passport exported to {passport_path}")
    except Exception as e:
        logger.warning(f"Passport export failed: {e}, attempting manual conversion")
        try:
            passport_dict = convert_to_native(
                master_passport.to_dict() if hasattr(master_passport, 'to_dict')
                else {'passport_id': str(master_passport)}
            )
            with open(passport_path, 'w', encoding='utf-8') as f:
                json.dump(passport_dict, f, indent=2, default=str)
            logger.info(f"Passport exported (manual) to {passport_path}")
        except Exception as e2:
            logger.error(f"Manual passport export failed: {e2}")
    # BUG-009 FIX: export full pipeline chain (all linked child passports)
    chain_path = passport_dir / 'pipeline_chain.json'
    try:
        orchestrator.export_pipeline_chain(str(chain_path))
        logger.info(f"Pipeline chain exported: {chain_path}")
    except Exception as _ce:
        logger.warning(f"Pipeline chain export failed: {_ce}")


# ---------------------------------------------------------------------------
# Summary output
# ---------------------------------------------------------------------------
print("=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)

print(f"\n  Data: {n_assets} assets -> {n_effective} effective")
print(f"  Baseline CVaR (return): {cvar_classical:.6f}  |  (loss): {opt_results.get('initial_cvar_loss', 0):.6f}")

if 'optimization' in pipeline_state:
    opt = pipeline_state['optimization']
    print(f"\n  Optimization ({opt.get('method', '?')}):")
    print(f"    CVaR (loss): {opt.get('initial_cvar_loss', pipeline_state['baseline_loss_convention']['cvar']):.6f}"
          f" -> {opt.get('optimal_cvar_loss', 0):.6f} ({opt.get('improvement_pct', 0):+.2f}%)")
    print(f"    Iterations: {opt.get('num_iterations', 0)}  |  Queries: {opt.get('total_queries', 0)}")
    print(f"    Time: {opt.get('total_time_seconds', 0):.2f}s")

if 'evar' in pipeline_state:
    ev = pipeline_state['evar']
    print(f"\n  EVaR: {ev.get('evar', 0):.6f}  (EVaR/CVaR={ev.get('evar_cvar_ratio', 0):.3f})")

if 'error_propagation' in pipeline_state:
    ep = pipeline_state['error_propagation']
    print(f"\n  Error bound: {ep.get('final_error_bound', 0):.6e}  (dominant: {ep.get('dominant_source', '?')})")

print(f"\n  Weights: {np.round(optimal_cluster_weights, 4) if optimal_cluster_weights is not None else 'N/A'}")

print(f"\n  Exported: {results_json_path}")
print("=" * 70)

logger.info("Pipeline execution completed successfully")

PIPELINE SUMMARY

  Data: 10 assets -> 10 effective
  Baseline CVaR (return): -0.047654  |  (loss): 0.047654

  Optimization (hybrid):
    CVaR (loss): 0.047654 -> 0.038343 (+19.54%)
    Iterations: 37  |  Queries: 41761088
    Time: 4502.25s

  EVaR: 0.071909  (EVaR/CVaR=1.875)

  Error bound: 3.299133e-01  (dominant: truncation)

  Weights: [0.0231 0.0222 0.0179 0.3121 0.3121 0.0235 0.026  0.1802 0.0219 0.061 ]

  Exported: C:\Users\nacho\Desktop\v2_final\v2_final\results\phase2_results.json


In [19]:
# ============================================================================
# CELL 17: FINAL ASSET WEIGHTS
# ============================================================================
# Clustering is disabled -- optimal_cluster_weights are already per-asset.

final_asset_weights = optimal_cluster_weights.copy()

# Apply box constraints (fixes numerical drift from simplex projection)
final_asset_weights = np.clip(final_asset_weights, MIN_WEIGHT, MAX_WEIGHT)
final_asset_weights /= final_asset_weights.sum()

logger.info(f"Final weights: {len(final_asset_weights)} assets, sum={final_asset_weights.sum():.6f}")
print(f"Final weights: {len(final_asset_weights)} assets, sum={final_asset_weights.sum():.6f}")
print(f"  {np.round(final_asset_weights, 4)}")

Final weights: 10 assets, sum=1.000000
  [0.0237 0.0227 0.0183 0.3074 0.3074 0.0241 0.0266 0.1846 0.0225 0.0625]


In [20]:
# ============================================================================
# CELL 18: RISK CONTRIBUTION ANALYSIS (risk_contributions.py)
# ============================================================================
# Marginal and component contributions to CVaR by asset.
#   MC_i = dCVaR/dw_i, RC_i = w_i * MC_i, Euler check: sum(RC_i) ~ CVaR.
# ============================================================================

from risk_contributions import (
    RiskContributionsAnalyzer, RiskContributionResult,
    compute_risk_contributions, compute_marginal_cvar,
)

risk_config    = phase3_config.get('risk_analysis', {})
COMPUTE_MARGINAL = risk_config.get('compute_marginal_contributions', True)

logger.info("=" * 70)
logger.info("RISK CONTRIBUTION ANALYSIS")
logger.info("=" * 70)

risk_contribution_results = None

if COMPUTE_MARGINAL:
    try:
        analysis_weights = final_asset_weights
        analysis_returns = returns_df.values if returns_df.shape[1] == len(analysis_weights) else returns_effective
        asset_names = tickers if len(tickers) == len(analysis_weights) else \
                      [f"Asset_{i}" for i in range(len(analysis_weights))]

        analyzer = RiskContributionsAnalyzer(alpha=ALPHA)

        risk_result = analyzer.compute_risk_contributions(
            weights=analysis_weights,
            returns=analysis_returns,
            asset_names=asset_names,
        )

        risk_contribution_results = {
            'total_cvar':               float(risk_result.total_cvar),
            'total_var':                float(risk_result.total_var),
            'marginal_contributions':   risk_result.marginal_contributions.tolist(),
            'component_contributions':  risk_result.component_contributions.tolist(),
            'percentage_contributions': risk_result.percentage_contributions.tolist(),
            'weights':                  risk_result.weights.tolist(),
            'asset_names':              risk_result.asset_names,
            'n_assets':                 risk_result.n_assets,
            'herfindahl_index':         float(risk_result.herfindahl_index),
            'effective_n_assets':       float(risk_result.effective_n_assets),
            'max_contribution_pct':     float(risk_result.max_contribution_pct),
            'euler_check':              float(risk_result.euler_check),
        }

        logger.info(f"Risk analysis complete: CVaR={risk_result.total_cvar:.6f}, "
                     f"Euler err={risk_result.euler_check:.6f}, HHI={risk_result.herfindahl_index:.4f}")

        # Print module summary + top contributors
        print(risk_result.summary())

        sorted_idx = np.argsort(risk_result.percentage_contributions)[::-1]
        print("\nTop risk contributors:")
        for rank, idx in enumerate(sorted_idx[:5], 1):
            pct = risk_result.percentage_contributions[idx] * 100
            w = risk_result.weights[idx] * 100
            print(f"  {rank}. {asset_names[idx]}: {pct:.1f}% of CVaR (weight={w:.1f}%)")

        hhi = risk_result.herfindahl_index
        label = "well diversified" if hhi < 0.15 else "moderately concentrated" if hhi < 0.25 else "highly concentrated"
        print(f"\n  HHI={hhi:.4f} ({label}), Effective N={risk_result.effective_n_assets:.1f}/{risk_result.n_assets}")
        print(f"  Euler check: {risk_result.euler_check:.6f}")

    except Exception as e:
        logger.error(f"Risk contribution analysis failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"ERROR: Risk analysis failed - {e}")

else:
    logger.info("Risk contribution analysis disabled in config")
    print("Risk contribution analysis: disabled")

ERROR: Risk analysis failed - 'Logger' object has no attribute 'metric'


In [21]:
# ============================================================================
# CELL 19: QAE VALIDATION (qae_validation.py)
# ============================================================================
# Validates QAE results against classical Monte Carlo baselines.
#   - Amplitude accuracy vs classical
#   - Query efficiency vs theoretical O(1/epsilon)
#   - Convergence verification, ZNE effectiveness
#   - CVaR and subgradient accuracy
# ============================================================================

from qae_validation import (
    QAEValidator, QAEValidationResult, MethodValidation,
    ValidationStatus, validate_qae_results, compute_classical_baseline,
)

val_config   = phase3_config.get('qae_validation', {})
VAL_ENABLED  = val_config.get('enabled', True)
VAL_MAX_ERROR = float(val_config.get('max_relative_error', 0.15))
VAL_MIN_CONV  = float(val_config.get('min_convergence_rate', 0.90))
VAL_MIN_ZNE   = float(val_config.get('min_zne_improvement', 0.01))

logger.info("=" * 70)
logger.info("QAE VALIDATION")
logger.info("=" * 70)

qae_validation_results = None

if VAL_ENABLED and qae_results and qae_results.get('success'):
    try:
        # ---------------------------------------------------------------
        # Classical baseline (same weights used in QAE estimation)
        # ---------------------------------------------------------------
        classical_baseline = compute_classical_baseline(
            weights=initial_weights,
            returns=returns_effective,
            alpha=ALPHA,
        )

        # ---------------------------------------------------------------
        # Extract QAE method results for validation
        # ---------------------------------------------------------------
        qae_results_for_validation = {}
        method_results = qae_results.get('method_results', {})

        for method_name in ['iqae', 'woerner']:
            if isinstance(method_results, dict):
                # Direct key
                if method_name in method_results:
                    qae_results_for_validation[method_name] = method_results[method_name]
                # Task-based keys (iqae_tail_prob, woerner_tail_prob)
                elif f"{method_name}_tail_prob" in method_results:
                    tail = method_results[f"{method_name}_tail_prob"]
                    el = method_results.get(f"{method_name}_E_L_tail", {})
                    qae_results_for_validation[method_name] = {
                        'amplitude': tail.get('amplitude', 0),
                        'amplitude_raw': tail.get('amplitude', 0),
                        'num_oracle_queries': tail.get('queries', 0) + el.get('queries', 0),
                        'converged': tail.get('converged', True) and el.get('converged', True),
                        'zne_applied': qae_results.get('zne_applied', False) if method_name == 'iqae' else False,
                    }

        # CVaR estimate for validation
        qae_results_for_validation['cvar_estimate'] = qae_results.get('quantum_cvar', 0)
        qae_results_for_validation['cvar'] = qae_results.get('quantum_cvar', 0)

        # ---------------------------------------------------------------
        # Subgradient results (if available)
        # ---------------------------------------------------------------
        subgrad_for_validation = None
        if 'subgrad_results' in dir() and subgrad_results:
            subgrad_for_validation = {
                'cosine_similarity': float(subgrad_results.get('subgradient_cosine_similarity',
                                           subgrad_results.get('cosine_similarity', 0))),
                'l2_error': float(subgrad_results.get('subgradient_l2_error',
                                  subgrad_results.get('l2_error', 0))),
            }

        # ---------------------------------------------------------------
        # Run validation
        # ---------------------------------------------------------------
        validator = QAEValidator(
            max_relative_error=VAL_MAX_ERROR,
            min_convergence_rate=VAL_MIN_CONV,
            min_zne_improvement=VAL_MIN_ZNE,
        )

        validation_result = validator.validate_all(
            qae_results=qae_results_for_validation,
            classical_results=classical_baseline,
            subgradient_results=subgrad_for_validation,
            epsilon=EPSILON,
        )

        qae_validation_results = validation_result.to_dict()
        qae_validation_results['config'] = {
            'max_relative_error': VAL_MAX_ERROR,
            'min_convergence_rate': VAL_MIN_CONV,
            'min_zne_improvement': VAL_MIN_ZNE,
        }
        qae_validation_results['methods_extracted'] = [
            k for k in ['iqae', 'woerner'] if k in qae_results_for_validation
        ]

        logger.info(f"QAE validation: status={validation_result.overall_status.value}, "
                     f"best={validation_result.best_method}, "
                     f"CVaR passed={validation_result.cvar_validation_passed}")

        # Module summary
        print(validation_result.summary())

        # Compact per-method detail
        for name, mr in [('IQAE', validation_result.iqae), ('Woerner', validation_result.woerner)]:
            if mr and mr.enabled:
                status = "PASS" if mr.relative_error <= VAL_MAX_ERROR else "FAIL"
                print(f"  {name}: err={mr.relative_error*100:.2f}% queries={mr.oracle_queries} [{status}]")

        print(f"\n  CVaR: est={validation_result.cvar_estimated:.6f} "
              f"classical={validation_result.cvar_classical:.6f} "
              f"err={validation_result.cvar_relative_error*100:.2f}% "
              f"[{'PASS' if validation_result.cvar_validation_passed else 'FAIL'}]")

        if subgrad_for_validation:
            print(f"  Subgradient: cosine={validation_result.subgradient_cosine_similarity:.4f} "
                  f"L2={validation_result.subgradient_l2_error:.6f}")

    except Exception as e:
        logger.error(f"QAE validation failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"ERROR: QAE validation failed - {e}")
        qae_validation_results = {'status': 'error', 'error': str(e)}

else:
    reason = "disabled in config" if not VAL_ENABLED else "QAE results not available"
    logger.info(f"QAE validation skipped: {reason}")
    print(f"QAE validation: {reason}")
    qae_validation_results = {'computed': False, 'reason': reason}

ERROR: QAE validation failed - 'Logger' object has no attribute 'metric'


In [22]:
# ============================================================================
# CELL 20: METHOD COMPARISON (method_comparison.py)
# ============================================================================
# Compares QAE methods (IQAE, Woerner) on accuracy, efficiency, speed.
# ============================================================================

from method_comparison import (
    MethodComparator, ComparisonResult, MethodMetrics, compare_qae_methods,
)

comp_config        = phase3_config.get('method_comparison', {})
COMPARISON_ENABLED = comp_config.get('enabled', False)

logger.info("=" * 70)
logger.info("QAE METHOD COMPARISON")
logger.info("=" * 70)

# ---------------------------------------------------------------------------
# Extract QAE method results for comparison
# ---------------------------------------------------------------------------
methods_available = []
qae_for_comparison = {}
classical_amp = qae_results.get('classical_tail_prob', 0.05) if qae_results else 0.05

if qae_results and qae_results.get('success'):
    method_results = qae_results.get('method_results', {})

    for method_name in ['iqae', 'woerner']:
        extracted = None

        if isinstance(method_results, dict):
            # Direct key
            if method_name in method_results and isinstance(method_results[method_name], dict):
                raw = method_results[method_name]
                extracted = {
                    'amplitude':           raw.get('amplitude', 0),
                    'amplitude_raw':       raw.get('amplitude_raw', raw.get('amplitude', 0)),
                    'num_oracle_queries':  raw.get('num_oracle_queries', 0),
                    'execution_time_ms':   raw.get('execution_time_ms', 0),
                    'converged':           raw.get('converged', True),
                    'zne_applied':         raw.get('zne_applied', False),
                }

            # Task-based keys (iqae_tail_prob, woerner_tail_prob)
            elif f"{method_name}_tail_prob" in method_results:
                tail = method_results[f"{method_name}_tail_prob"]
                el   = method_results.get(f"{method_name}_E_L_tail", {})
                amp_raw = (qae_results.get(f'{method_name}_tail_prob_noisy_raw')
                           or tail.get('amplitude_noisy_raw')
                           or tail.get('amplitude', 0))
                extracted = {
                    'amplitude':          tail.get('amplitude', 0),
                    'amplitude_raw':      amp_raw,
                    'num_oracle_queries': tail.get('queries', 0) + el.get('queries', 0),
                    'execution_time_ms':  tail.get('time_ms', 0) + el.get('time_ms', 0),
                    'converged':          tail.get('converged', True) and el.get('converged', True),
                    'zne_applied':        qae_results.get('zne_applied', False),
                }

        # Flat keys fallback
        if extracted is None:
            queries_key = f'total_queries_{method_name}'
            if qae_results.get(queries_key, 0) > 0:
                amp = qae_results.get(f'{method_name}_amplitude', qae_results.get('quantum_tail_prob', 0))
                extracted = {
                    'amplitude':          amp,
                    'amplitude_raw':      qae_results.get(f'{method_name}_tail_prob_noisy_raw', amp),
                    'num_oracle_queries': qae_results.get(queries_key, 0),
                    'execution_time_ms':  qae_results.get(f'{method_name}_time_ms', 0),
                    'converged':          True,
                    'zne_applied':        qae_results.get('zne_applied', False),
                }

        if extracted is not None:
            qae_for_comparison[method_name] = extracted
            methods_available.append(method_name)

logger.info(f"Methods for comparison: {methods_available}")

# ---------------------------------------------------------------------------
# Run comparison
# ---------------------------------------------------------------------------
method_comparison_results = None

if len(methods_available) > 0:
    try:
        comparator = MethodComparator(epsilon=EPSILON)
        comparison_result = comparator.compare(qae_for_comparison, classical_amp)

        method_comparison_results = comparison_result.to_dict()
        method_comparison_results['methods_available'] = methods_available

        logger.info(f"Comparison: best={comparison_result.best_overall}, "
                     f"rec={comparison_result.recommendation}")

        print(comparison_result.summary())

        if hasattr(comparator, 'generate_comparison_table'):
            print(comparator.generate_comparison_table(comparison_result))

    except Exception as e:
        logger.error(f"Method comparison failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"ERROR: Method comparison failed - {e}")
        method_comparison_results = {'status': 'error', 'error': str(e)}

else:
    logger.info("No QAE methods available for comparison")
    print("No QAE methods available for comparison")
    method_comparison_results = {'computed': False, 'reason': 'No QAE results'}

QAE METHOD COMPARISON
Methods compared: 2
Epsilon: 0.002
Classical baseline: 0.050746

----------------------------------------------------------------------
COMPARISON TABLE
----------------------------------------------------------------------
Method        Amplitude  Rel.Error    Queries   Time(ms)   Rank
----------------------------------------------------------------------
iqae           0.050690    0.0011    214,016        0.0      1
woerner        0.059039    0.1634        128        0.0      2

----------------------------------------------------------------------
RANKINGS BY CATEGORY
----------------------------------------------------------------------
  Best Accuracy:   iqae
  Best Efficiency: woerner
  Best Speed:      iqae
  Best Overall:    iqae

----------------------------------------------------------------------
RECOMMENDATION
----------------------------------------------------------------------
  Method: iqae
  Reason: lowest error (0.11%), fastest (0.0ms)
+--------

In [23]:
# ============================================================================
# CELL 21: CLASSICAL BENCHMARKING (classical_benchmark.py)
# ============================================================================
# Fix [P0-001]: BENCHMARKING_ENABLED default changed False → True.
#               Explicit traceback logging in except block.
# Adds [P1-001]: markowitz, risk_parity now available as methods.
# ============================================================================

from classical_benchmark import (
    Benchmarker, ClassicalOptimizers, BenchmarkResult, ClassicalResult,
    run_classical_benchmark, SCIPY_AVAILABLE, CVXPY_AVAILABLE,
)
import traceback as _tb

bench_config         = phase3_config.get('benchmarking', {})
# Fix [P0-001]: default True so benchmarking runs unless explicitly disabled
BENCHMARKING_ENABLED = bench_config.get('enabled', True)

logger.info("=" * 70)
logger.info("CLASSICAL BENCHMARKING")
logger.info(f"BENCHMARKING_ENABLED={BENCHMARKING_ENABLED}")
logger.info(f"bench_config keys={list(bench_config.keys())}")
logger.info("=" * 70)

benchmark_results = None

if BENCHMARKING_ENABLED and opt_results:
    try:
        benchmarker = Benchmarker(
            alpha=ALPHA,
            max_iterations=200,
            timeout_per_method_seconds=bench_config.get(
                'timeout_per_method_seconds', 120
            )
        )

        # P1-001: add markowitz and risk_parity to the standard suite
        methods_to_run = bench_config.get(
            'methods',
            ['slsqp', 'cobyla', 'subgradient', 'monte_carlo', 'markowitz', 'risk_parity']
        )

        benchmark_result = benchmarker.run_benchmark(
            returns=returns_effective,
            quantum_result=opt_results,
            initial_weights=initial_weights,
            methods=methods_to_run,
            bounds=(OPT_MIN_WEIGHT, OPT_MAX_WEIGHT),
        )

        # Build serialisable dict of classical results
        classical_dict = {}
        for name, res in benchmark_result.classical_results.items():
            has_valid_cvar = (
                res.optimal_cvar is not None
                and res.optimal_cvar < float('inf')
                and res.optimal_cvar > 0
            )
            if res.success or has_valid_cvar:
                classical_dict[name] = {
                    'optimal_cvar':        float(res.optimal_cvar),
                    'optimal_var':         float(res.optimal_var),
                    'optimal_weights':     res.optimal_weights.tolist() if res.optimal_weights is not None else [],
                    'execution_time_s':    float(res.execution_time_s),
                    'iterations':          int(res.iterations),
                    'converged':           bool(res.converged),
                    'success':             bool(res.success),
                    'message':             res.message,
                }

        # Find best classical by CVaR
        best_cvar   = float('inf')
        best_method = ""
        for name, d in classical_dict.items():
            if d['optimal_cvar'] < best_cvar:
                best_cvar   = d['optimal_cvar']
                best_method = name

        speedup_factor = 0.0
        if best_method and opt_results.get('execution_time_s', 0) > 0:
            best_time      = classical_dict[best_method]['execution_time_s']
            speedup_factor = best_time / opt_results['execution_time_s']

        cvar_improvement = 0.0
        if best_cvar > 0:
            qcvar = opt_results.get('optimal_cvar_loss', opt_results.get('optimal_cvar', 0))
            cvar_improvement = (best_cvar - qcvar) / best_cvar

        # BUG-002 FIX: explicitly include quantum_cvar in benchmark_results
        _q_cvar_for_bench = float(
            opt_results.get('optimal_cvar_loss') or
            opt_results.get('optimal_cvar') or
            opt_results.get('final_cvar_loss') or 0.0
        )
        benchmark_results = {
            'computed':            True,
            'quantum_cvar':        _q_cvar_for_bench,
            'classical_results':   classical_dict,
            'best_classical_method': best_method,
            'best_classical_cvar': best_cvar if best_cvar < float('inf') else None,
            'speedup_factor':      speedup_factor,
            'cvar_improvement':    cvar_improvement,
        }

        logger.info(f"BENCHMARK complete: best_classical={best_method}, CVaR={best_cvar:.6f}")
        logger.info(f"CVaR improvement vs best classical: {cvar_improvement:.2%}")
        logger.info(f"Speedup factor: {speedup_factor:.2f}x")

    except Exception as e:
        # Fix [P0-001]: make failure visible — log full traceback
        logger.error(f"Classical benchmarking FAILED: {type(e).__name__}: {e}")
        logger.error(_tb.format_exc())
        benchmark_results = {'computed': False, 'error': str(e)}

else:
    reason = "opt_results not available" if not opt_results else "benchmarking disabled in config"
    logger.info(f"Classical benchmarking skipped: {reason}")
    benchmark_results = {'computed': False, 'reason': reason}


In [24]:
# ============================================================================
# CELL 22: OUT-OF-SAMPLE VALIDATION
# ============================================================================
# Validates optimized portfolios on held-out future data.
# Compares: Equal Weight, Quantum CVaR, Classical CVaR, Inverse Vol.
# Key metric: CVaR Prediction Error = |CVaR_train - CVaR_test| / |CVaR_train|
# ============================================================================

logger.info("=" * 70)
logger.info("OUT-OF-SAMPLE VALIDATION")
logger.info("=" * 70)

oos_results = None

if not (OOS_ENABLED and returns_df_oos is not None and final_asset_weights is not None):
    reason = ("OOS disabled" if not OOS_ENABLED
              else "returns_df_oos not available" if returns_df_oos is None
              else "final_asset_weights not available")
    logger.info(f"OOS validation skipped: {reason}")
    print(f"OOS validation: {reason}")
    oos_results = {'computed': False, 'reason': reason}

else:
    try:
        TRADING_DAYS_YEAR = 252

        train_returns_matrix = returns_df.values
        test_returns_matrix  = returns_df_oos.values
        n_train_periods = len(returns_df)
        n_test_periods  = len(returns_df_oos)
        n_oos_assets    = test_returns_matrix.shape[1]

        logger.info(f"Train: {n_train_periods}, Test: {n_test_periods}, Assets: {n_oos_assets}")

        print(f"Train: {returns_df.index[0].date()} to {returns_df.index[-1].date()} ({n_train_periods})")
        print(f"Test:  {returns_df_oos.index[0].date()} to {returns_df_oos.index[-1].date()} ({n_test_periods})")

        # ---------------------------------------------------------------
        # Build portfolio weight vectors
        # ---------------------------------------------------------------
        w_equal   = np.ones(n_oos_assets) / n_oos_assets
        w_quantum = final_asset_weights.copy()

        # Classical: best from benchmark, else equal weight
        w_classical = None
        classical_source = "equal_weight_fallback"
        if 'benchmark_result' in dir() and benchmark_result is not None and hasattr(benchmark_result, 'classical_results'):
            for name, res in benchmark_result.classical_results.items():
                if res.success and res.optimal_weights is not None:
                    if w_classical is None or res.optimal_cvar < _best_cvar:
                        w_classical = np.array(res.optimal_weights)
                        _best_cvar = res.optimal_cvar
                        classical_source = name
        if w_classical is None:
            logger.warning(
                "Classical CVaR weights not available — using Equal Weight FALLBACK. "
                "Fix: ensure benchmark runs before OOS (P0-001)."
            )
            w_classical = w_equal.copy()

        # Inverse volatility (train data only)
        train_vol = np.maximum(np.std(train_returns_matrix, axis=0), 1e-10)
        w_invvol  = (1.0 / train_vol)
        w_invvol /= w_invvol.sum()

        portfolios_oos = {
            'Equal Weight':  w_equal,
            'Quantum CVaR':  w_quantum,
            'Classical CVaR': w_classical,
            'Inverse Vol':   w_invvol,
        }

        # ---------------------------------------------------------------
        # Compute OOS metrics
        # ---------------------------------------------------------------
        def _oos_metrics(daily_rets, alpha):
            var_t = np.percentile(daily_rets, alpha * 100)
            tail = daily_rets[daily_rets <= var_t]
            cvar_loss = -float(np.mean(tail)) if len(tail) > 0 else -float(var_t)

            cum_ret = float(np.prod(1 + daily_rets) - 1)
            mean_d  = float(np.mean(daily_rets))
            ann_ret = mean_d * TRADING_DAYS_YEAR
            vol_d   = float(np.std(daily_rets, ddof=1))
            ann_vol = vol_d * np.sqrt(TRADING_DAYS_YEAR)
            sharpe  = ann_ret / ann_vol if ann_vol > 1e-10 else 0.0

            downside = daily_rets[daily_rets < 0]
            ds_std   = float(np.std(downside, ddof=1)) if len(downside) > 1 else vol_d
            sortino  = ann_ret / (ds_std * np.sqrt(TRADING_DAYS_YEAR)) if ds_std > 1e-10 else 0.0

            cumulative  = np.cumprod(1 + daily_rets)
            running_max = np.maximum.accumulate(cumulative)
            max_dd = float(np.min((cumulative - running_max) / np.where(running_max > 0, running_max, 1.0)))

            return {
                'cvar_loss': cvar_loss, 'var_loss': -float(var_t),
                'cumulative_return': cum_ret, 'annualized_return': ann_ret,
                'annualized_vol': ann_vol, 'sharpe': sharpe,
                'sortino': sortino, 'max_drawdown': max_dd,
            }

        oos_metrics = {pname: _oos_metrics(test_returns_matrix @ w, ALPHA)
                       for pname, w in portfolios_oos.items()}

        # ---------------------------------------------------------------
        # CVaR prediction error (train vs test)
        # ---------------------------------------------------------------
        def _cvar_loss(w, returns_matrix, alpha):
            pr = returns_matrix @ w
            var_t = np.percentile(pr, alpha * 100)
            tail = pr[pr <= var_t]
            return -float(np.mean(tail)) if len(tail) > 0 else -float(var_t)

        cvar_train = {pn: _cvar_loss(w, train_returns_matrix, ALPHA) for pn, w in portfolios_oos.items()}
        cvar_pred_err = {pn: abs(cvar_train[pn] - oos_metrics[pn]['cvar_loss']) / max(abs(cvar_train[pn]), 1e-10)
                         for pn in portfolios_oos}

        # ---------------------------------------------------------------
        # Comparison table
        # ---------------------------------------------------------------
        pnames = list(portfolios_oos.keys())

        header = f"{'Metric':<25}" + "".join(f" {p:>14}" for p in pnames)
        print("\n" + header)
        print("  " + "-" * (25 + 15 * len(pnames)))

        for label, key, fmt in [
            ('OOS CVaR (loss)',    'cvar_loss',         '.6f'),
            ('Cumulative Return',  'cumulative_return', '.4f'),
            ('Annualized Return',  'annualized_return', '.4f'),
            ('Annualized Vol',     'annualized_vol',    '.4f'),
            ('Sharpe Ratio',       'sharpe',            '.4f'),
            ('Sortino Ratio',      'sortino',           '.4f'),
            ('Max Drawdown',       'max_drawdown',      '.4f'),
        ]:
            row = f"{label:<25}" + "".join(f" {oos_metrics[p][key]:>14{fmt}}" for p in pnames)
            print(row)

        row_pred = f"{'CVaR Pred. Error (%)' :<25}" + "".join(f" {cvar_pred_err[p]*100:>13.2f}%" for p in pnames)
        print(row_pred)

        # ---------------------------------------------------------------
        # Rankings
        # ---------------------------------------------------------------
        ranked_cvar   = sorted(pnames, key=lambda n: oos_metrics[n]['cvar_loss'])
        ranked_sharpe = sorted(pnames, key=lambda n: oos_metrics[n]['sharpe'], reverse=True)

        print(f"\nBy CVaR: {' > '.join(ranked_cvar)}")
        print(f"By Sharpe: {' > '.join(ranked_sharpe)}")

        # ---------------------------------------------------------------
        # Build results dict
        # ---------------------------------------------------------------
        q_cvar = oos_metrics['Quantum CVaR']['cvar_loss']
        e_cvar = oos_metrics['Equal Weight']['cvar_loss']
        c_cvar = oos_metrics['Classical CVaR']['cvar_loss']
        q_pred = cvar_pred_err['Quantum CVaR'] * 100

        oos_results = {
            'computed': True,
            'split_date': str(OOS_SPLIT_DATE),
            'n_train': n_train_periods, 'n_test': n_test_periods,
            'classical_source': classical_source,
            'portfolios': {
                pn: {
                    'weights': portfolios_oos[pn].tolist(),
                    'oos_metrics': {k: float(v) for k, v in oos_metrics[pn].items()},
                    'train_cvar': float(cvar_train[pn]),
                    'cvar_prediction_error': float(cvar_pred_err[pn]),
                }
                for pn in pnames
            },
            'rankings': {'by_cvar': ranked_cvar, 'by_sharpe': ranked_sharpe},
            'summary': {
                'quantum_oos_cvar': float(q_cvar),
                'quantum_cvar_rank': ranked_cvar.index('Quantum CVaR') + 1,
                'quantum_sharpe_rank': ranked_sharpe.index('Quantum CVaR') + 1,
                'quantum_prediction_error_pct': float(q_pred),
                'beats_equal_weight': bool(q_cvar < e_cvar),
                'beats_classical': bool(q_cvar < c_cvar),
            },
        }

        # Export
        if 'results_dir' in dir() and results_dir is not None:
            try:
                oos_path = results_dir / 'oos_validation.json'
                with open(oos_path, 'w') as f:
                    json.dump(json.loads(json.dumps(oos_results, default=str)), f, indent=2)
                logger.info(f"OOS results exported to {oos_path}")
            except Exception as e:
                logger.warning(f"OOS export failed: {e}")

        # Passport
        if orchestrator and master_passport:
            master_passport = orchestrator.checkpoint(
                master_passport, name='phase3f_oos_validation', data=None,
                metadata={
                    'quantum_oos_cvar': float(q_cvar),
                    'quantum_cvar_rank': ranked_cvar.index('Quantum CVaR') + 1,
                    'beats_equal_weight': bool(q_cvar < e_cvar),
                    'prediction_error_pct': float(q_pred),
                }
            )

        logger.info(f"OOS complete: rank={ranked_cvar.index('Quantum CVaR')+1}/{len(pnames)}, "
                     f"pred_err={q_pred:.2f}%")

        print(f"\nQuantum CVaR: rank #{ranked_cvar.index('Quantum CVaR')+1}/{len(pnames)}, "
              f"beats_EW={q_cvar < e_cvar}, beats_classical={q_cvar < c_cvar}, pred_err={q_pred:.1f}%")

    except Exception as e:
        logger.error(f"OOS validation failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"ERROR: OOS validation failed - {e}")
        oos_results = {'computed': False, 'error': str(e)}

Train: 2020-01-03 to 2023-12-29 (1005)
Test:  2024-01-02 to 2025-12-30 (501)

Metric                      Equal Weight   Quantum CVaR Classical CVaR    Inverse Vol
  -------------------------------------------------------------------------------------
OOS CVaR (loss)                 0.028428       0.023583       0.023823       0.026938
Cumulative Return                 0.2934         0.2061         0.2142         0.2850
Annualized Return                 0.1465         0.1076         0.1110         0.1419
Annualized Vol                    0.1846         0.1629         0.1633         0.1772
Sharpe Ratio                      0.7938         0.6603         0.6798         0.8007
Sortino Ratio                     0.9716         0.8553         0.8745         0.9838
Max Drawdown                     -0.2032        -0.1558        -0.1465        -0.1890
CVaR Pred. Error (%)              40.35%         38.66%         36.43%         40.62%

By CVaR: Quantum CVaR > Classical CVaR > Inverse Vol > Equa

In [25]:
# ============================================================================
# CELL 23: COMPREHENSIVE METRICS EXPORT
# ============================================================================
# Exports all metrics and KPIs to a single JSON for TFM analysis.
# Output: results/tfm_comprehensive_metrics_{timestamp}.json
# ============================================================================

import uuid

logger.info("=" * 70)
logger.info("COMPREHENSIVE METRICS EXPORT")
logger.info("=" * 70)

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _sf(v, d=0.0):
    """Safe float."""
    try: return float(v) if v is not None else d
    except (TypeError, ValueError): return d

def _si(v, d=0):
    """Safe int."""
    try: return int(v) if v is not None else d
    except (TypeError, ValueError): return d

def _sl(v):
    """Safe list (handles ndarray)."""
    if v is None: return []
    return v.tolist() if isinstance(v, np.ndarray) else list(v)

def _sd(v):
    """Safe dict."""
    return dict(v) if v else {}

def _og(d, *keys, default=0):
    """Safe nested dict get: _og(d, 'a', 'b', default=0) -> d['a']['b'] or default."""
    for k in keys:
        if isinstance(d, dict):
            d = d.get(k, None)
        else:
            return default
    return d if d is not None else default

# Shorthand accessors for result dicts (may be None)
_opt = opt_results or {}
_qae = qae_results or {}
# [T-001] Collect QAE diagnostic fields if the QAECircuits object is available
_t001_diagnostics = {}
try:
    if 'cvar_qae' in dir() and cvar_qae is not None:
        _t001_diagnostics = {
            'shots_per_round_est':            getattr(cvar_qae, '_t001_shots_per_round_est', None),
            'n_rounds_est':                   getattr(cvar_qae, '_t001_n_rounds_est', None),
            'k_max_est':                      getattr(cvar_qae, '_t001_k_max_est', None),
            'sampler_shots':                  getattr(cvar_qae, '_t001_sampler_shots', None),
            'var_threshold_bin_index':        getattr(cvar_qae, '_t001_last_threshold_bin_index', None),
            'var_threshold_quantization_error': getattr(cvar_qae, '_t001_last_threshold_quantization_error', None),
            'bin_width':                      getattr(cvar_qae, '_t001_last_bin_width', None),
            'n_qubits_used':                  getattr(cvar_qae, '_t001_last_n_qubits', None),
        }
        logger.info(f"[T-001] QAE diagnostics: {_t001_diagnostics}")
except Exception as _e:
    logger.warning(f"[T-001] Could not collect QAE diagnostics: {_e}")
_err = error_results or {} if 'error_results' in dir() else {}
_sub = subgrad_results if 'subgrad_results' in dir() and subgrad_results else {}
_evar = evar_results if 'evar_results' in dir() and evar_results else {}
_bench = benchmark_results or {} if 'benchmark_results' in dir() else {}
_comp = method_comparison_results or {} if 'method_comparison_results' in dir() else {}
_qval = qae_validation_results or {} if 'qae_validation_results' in dir() else {}
_oos = oos_results or {} if 'oos_results' in dir() else {}
_risk = risk_contribution_results or {} if 'risk_contribution_results' in dir() else {}

# ---------------------------------------------------------------------------
# Build comprehensive metrics document
# ---------------------------------------------------------------------------

run_id = str(uuid.uuid4())
timestamp = datetime.now().isoformat()


# ---------------------------------------------------------------------------
# F-003: Convergence fingerprint
# ---------------------------------------------------------------------------
_conv_history = convergence_history.get('cvar_history', []) if ('convergence_history' in dir() and isinstance(convergence_history, dict)) else []
_convergence_fingerprint = {}
if len(_conv_history) >= 2:
    import numpy as _np_fp
    _total_improv = _conv_history[0] - _conv_history[-1]
    _thresh = _conv_history[0] - 0.95 * _total_improv
    _iter_95 = next((i for i, v in enumerate(_conv_history) if v <= _thresh), len(_conv_history))
    _regressions = sum(1 for i in range(1, len(_conv_history)) if _conv_history[i] > _conv_history[i-1])
    _slope = float(_np_fp.polyfit(range(len(_conv_history)), _conv_history, 1)[0]) if len(_conv_history) > 1 else 0.0
    _final_std = float(_np_fp.std(_conv_history[-10:])) if len(_conv_history) >= 10 else float(_np_fp.std(_conv_history))
    _convergence_fingerprint = {
        'iter_to_95pct_improvement': int(_iter_95),
        'mean_convergence_slope':    round(_slope, 8),
        'final_stability_std':       round(_final_std, 8),
        'n_regressions':             int(_regressions),
        'total_improvement_abs':     round(float(_total_improv), 8),
        'total_improvement_pct':     round(float(_total_improv / max(abs(_conv_history[0]), 1e-10) * 100), 4),
        'n_iterations_used':         len(_conv_history),
    }
    logger.info(f"Convergence fingerprint: iter_95={_iter_95}, regressions={_regressions}, slope={_slope:.6f}")

comprehensive_metrics = {
    "metadata": {
        "run_id": run_id,
        "timestamp": timestamp,
        "project_name": global_config.get('project_name', 'Quantum Portfolio Optimization'),
        "author": "Ignacio Lopez Leis",
        "institution": "Universidad Autonoma de Madrid",
        "configuration": {
            "global": {
                "alpha": float(ALPHA), "random_seed": RANDOM_SEED,
                "max_weight": float(MAX_WEIGHT), "min_weight": float(MIN_WEIGHT),
            },
            "data": {
                "tickers": tickers, "n_assets": n_assets, "n_effective": n_effective,
                "n_periods": n_periods,
                "start_date": str(START_DATE), "end_date": str(END_DATE),
                "oos_enabled": OOS_ENABLED,
                "oos_split_date": str(OOS_SPLIT_DATE) if OOS_ENABLED else None,
            },
            "network_selection": {
                "enabled": NETWORK_SELECTION_ENABLED,
                "portfolio_id": selection_result.portfolio_id if selection_result else None,
                "method": selection_result.method if selection_result else None,
                "n_universe": _n_universe if NETWORK_SELECTION_ENABLED and '_n_universe' in dir() else None,
                "n_selected": len(selection_result.selected_tickers) if selection_result else None,
                "energy": float(selection_result.energy) if selection_result and np.isfinite(selection_result.energy) else None,
                "feasible": selection_result.feasible if selection_result else None,
                "avg_peripherality": float(_avg_periph) if NETWORK_SELECTION_ENABLED and '_avg_periph' in dir() else None,
                "avg_centrality": float(_avg_centr) if NETWORK_SELECTION_ENABLED and '_avg_centr' in dir() else None,
                "n_communities_represented": int(_n_comm) if NETWORK_SELECTION_ENABLED and '_n_comm' in dir() else None,
                "dwave_backend": net_sel_config.get('dwave', {}).get('backend') if NETWORK_SELECTION_ENABLED else None,
                "dwave_num_reads": selection_result.num_reads if selection_result else None,
                "timing_ms": selection_result.timing_ms if selection_result else None,
            },
            "phase2": {
                "num_qubits": OPT_NUM_QUBITS, "epsilon": OPT_EPSILON,
                "optimizer_type": OPTIMIZER_TYPE,
                "max_iterations": MAX_ITERATIONS, "learning_rate": LEARNING_RATE,
                "use_zne": OPT_USE_ZNE,
                "recursive_enabled": RECURSIVE_ENABLED,
                "use_recursive": USE_RECURSIVE,
            },
        },
    },

    "qae_diagnostics_t001": _t001_diagnostics,
    "convergence_fingerprint": _convergence_fingerprint,
    "metrics": {
        "risk": {
            "initial": {
                "var_return": _sf(var_classical), "cvar_return": _sf(cvar_classical),
                "var_loss": _sf(-var_classical), "cvar_loss": _sf(-cvar_classical),
                "tail_probability": _sf(tail_prob_classical), "tail_size": _si(tail_size),
            },
            "optimized": {
                "var_loss":    _sf(_opt.get('optimal_var_loss')),
                "cvar_loss":   _sf(_opt.get('optimal_cvar_loss')),
                "var_return":  _sf(_opt.get('optimal_var_return')),
                "cvar_return": _sf(_opt.get('optimal_cvar_return')),
            },
            "evar": {
                "evar":       _sf(_og(_evar, 'optimal', 'evar')),
                "cvar":       _sf(_og(_evar, 'optimal', 'cvar')),
                "evar_cvar_ratio": _sf(_og(_evar, 'optimal', 'evar_cvar_ratio')),
                "mathematical_validity": bool(_evar.get('mathematical_validity', False)),
                "improvement_pct": _sf(_og(_evar, 'evar_improvement', 'reduction_pct')),
            },
        },

        "optimization": {
            "method":               _opt.get('method', 'unknown'),
            "cvar_improvement_pct": _sf(_opt.get('cvar_improvement_pct')),
            "cvar_reduction_absolute": _sf(_opt.get('cvar_reduction_absolute')),
            "converged":            bool(_opt.get('converged', False)),
            "num_iterations":       _si(_opt.get('num_iterations')),
            "early_stopped":        bool(_opt.get('early_stopped', False)),
            "final_gradient_norm":  _sf(_opt.get('final_gradient_norm')),
            "recursive": {
                "used":         USE_RECURSIVE,
                "tree_depth":   _si(_og(_opt, 'recursive', 'tree_depth')),
                "total_nodes":  _si(_og(_opt, 'recursive', 'total_nodes')),
                "total_leaves": _si(_og(_opt, 'recursive', 'total_leaves')),
                "per_level_cvar": _sl(_og(_opt, 'recursive', 'per_level_cvar', default=[])),
            },
        },

        "quantum": {
            "qae": {
                "quantum_cvar":      _sf(_qae.get('quantum_cvar')),
                "quantum_tail_prob": _sf(_qae.get('quantum_tail_prob')),
                "classical_cvar":    _sf(_qae.get('classical_cvar')),
                "cvar_error":        _sf(_qae.get('cvar_error')),
                "tail_prob_error":   _sf(_qae.get('tail_prob_error')),
                "total_queries_iqae":    _si(_qae.get('total_queries_iqae')),
                "total_queries_woerner": _si(_qae.get('total_queries_woerner')),
                "total_circuits":    _si(_qae.get('total_circuits')),
                "zne_applied":       bool(_qae.get('zne_applied', False)),
                "zne_improvement":   _sf(_qae.get('zne_improvement')),
                "noise_enabled":     bool(_qae.get('noise_enabled', False)),
                "execution_time_ms": _sf(_qae.get('execution_time_ms')),
            },
            "subgradient": {
                "norm":              _sf(_sub.get('subgradient_norm')),
                "method":            _sub.get('method', 'unknown'),
                "cosine_similarity": _sf(_sub.get('subgradient_cosine_similarity')),
                "l2_error":          _sf(_sub.get('subgradient_l2_error')),
            },
            "total_oracle_queries": {
                "iqae":    _si(_opt.get('total_queries_iqae')),
                "woerner": _si(_opt.get('total_queries_woerner')),
                "total":   _si(_opt.get('total_queries')),
                "quantum_success_rate": _sf(_opt.get('quantum_success_rate')),
            },
        },

        "risk_contributions": {
            "computed":             bool(_risk),
            "total_cvar":           _sf(_risk.get('total_cvar')),
            "herfindahl_index":     _sf(_risk.get('herfindahl_index')),
            "effective_n_assets":   _sf(_risk.get('effective_n_assets')),
            "euler_check_error":    _sf(_risk.get('euler_check')),
            "max_contribution_pct": _sf(_risk.get('max_contribution_pct')),
            "component_contributions": _sl(_risk.get('component_contributions')),
            "percentage_contributions": _sl(_risk.get('percentage_contributions')),
        },

        "benchmarks": {
            "computed":                bool(_bench.get('computed', False)),
            "quantum_cvar":            _sf(_bench.get('quantum_cvar')),
            "best_classical_method":   _bench.get('best_classical_method', 'unknown'),
            "best_classical_cvar":     _sf(_bench.get('best_classical_cvar')),
            "cvar_improvement_vs_best": _sf(_bench.get('cvar_improvement_vs_best')),
            "speedup_factor":          _sf(_bench.get('speedup_factor')),
            "classical_results":       _sd(_bench.get('classical_results')),
        },

        "method_comparison": {
            "computed":    bool(_comp),
            "best_method": _comp.get('recommendation', _og(_comp, 'best', 'overall', default='unknown')),
            "methods":     _sd(_comp.get('methods')),
        },

        "oos_validation": {
            "computed":       bool(_oos.get('computed', False)),
            "split_date":     _oos.get('split_date'),
            "n_train":        _si(_oos.get('n_train')),
            "n_test":         _si(_oos.get('n_test')),
            "quantum_oos_cvar":  _sf(_og(_oos, 'summary', 'quantum_oos_cvar')),
            "quantum_cvar_rank": _si(_og(_oos, 'summary', 'quantum_cvar_rank')),
            "prediction_error_pct": _sf(_og(_oos, 'summary', 'quantum_prediction_error_pct')),
            "beats_equal_weight":   bool(_og(_oos, 'summary', 'beats_equal_weight', default=False)),
            "beats_classical":      bool(_og(_oos, 'summary', 'beats_classical', default=False)),
            "rankings":  _sd(_oos.get('rankings')),
            "portfolios": _sd(_oos.get('portfolios')),
        },

        "errors": {
            "computed":             bool(_err and not _err.get('error')),
            "final_error_bound":    _sf(_err.get('final_error_bound')),
            "estimated_cvar_error": _sf(_err.get('estimated_cvar_error')),
            "dominant_source":      _err.get('dominant_error_source', 'unknown'),
            "truncation_error":     _sf(_og(_err, 'truncation', 'max_error')),
            "noise_error":          _sf(_og(_err, 'noise', 'total_error')),
            "zne_enabled":          bool(_err.get('zne_enabled', False)),
            "zne_improvement_pct":  _sf(_err.get('zne_improvement_percent')),
            "best_qae_method":      _og(_err, 'method_comparison', 'best_method', default='unknown'),
        },

        "qae_validation": {
            "computed":    bool(_qval and _qval.get('status', 'not_computed') != 'not_computed'),
            "status":      _qval.get('status', 'not_computed'),
            "all_passed":  bool(_qval.get('all_passed', False)),
        },
    },

    "timings": {
        "optimization_s":    _sf(_opt.get('execution_time_s', _opt.get('total_time_seconds'))),
        "quantum_time_ms":   _sf(_opt.get('total_quantum_time_ms')),
        "qae_estimation_s":  _sf(_qae.get('execution_time_s')),
        "subgradient_s":     _sf(_sub.get('execution_time_s')),
    },

    "weights": {
        "initial": _sl(initial_weights),
        "optimal": _sl(optimal_cluster_weights) if optimal_cluster_weights is not None else [],
        "by_asset": {
            t: _sf(optimal_cluster_weights[i])
            for i, t in enumerate(effective_tickers)
            if optimal_cluster_weights is not None and i < len(optimal_cluster_weights)
        },
    },

    "convergence_history": {
        "cvar_history":   _sl(_opt.get('cvar_history_loss', [])),
        "gradient_norms": _sl(_opt.get('gradient_norms', [])),
    },
}

# ---------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------
results_dir = PROJECT_ROOT / export_config.get('base_directory', 'results') / export_config.get('run_subdirectory', 'default')
results_dir.mkdir(parents=True, exist_ok=True)

timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
export_path = results_dir / f"tfm_comprehensive_metrics_{timestamp_str}.json"
latest_path = results_dir / "tfm_comprehensive_metrics_latest.json"

try:
    for path in [export_path, latest_path]:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(comprehensive_metrics, f, indent=2, ensure_ascii=False, default=str)
    logger.info(f"Metrics exported: {export_path}")
    export_success = True
except Exception as e:
    logger.error(f"Metrics export failed: {e}")
    export_success = False

# ---------------------------------------------------------------------------
# Passport
# ---------------------------------------------------------------------------
if orchestrator and master_passport:
    master_passport = orchestrator.checkpoint(
        master_passport, name='comprehensive_metrics_export',
        data={'export_path': str(export_path)},
        metadata={
            'run_id': run_id,
            'cvar_improvement_pct': _sf(_opt.get('cvar_improvement_pct')),
            'converged': bool(_opt.get('converged', False)),
        }
    )

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
m = comprehensive_metrics['metrics']
print("=" * 70)
print("COMPREHENSIVE METRICS EXPORT")
print("=" * 70)
print(f"  Run: {run_id[:12]}...  |  {timestamp}")
print(f"  CVaR (loss): {m['risk']['initial']['cvar_loss']:.6f} -> {m['risk']['optimized']['cvar_loss']:.6f} "
      f"({m['optimization']['cvar_improvement_pct']:+.2f}%)")
print(f"  Converged: {m['optimization']['converged']}  |  Iterations: {m['optimization']['num_iterations']}")
print(f"  Queries: IQAE={m['quantum']['total_oracle_queries']['iqae']}, "
      f"Woerner={m['quantum']['total_oracle_queries']['woerner']}")
if m['benchmarks']['computed']:
    print(f"  Benchmark: {m['benchmarks']['best_classical_method']} "
          f"(improvement={m['benchmarks']['cvar_improvement_vs_best']*100:.2f}%)")
if m['oos_validation']['computed']:
    print(f"  OOS: rank={m['oos_validation']['quantum_cvar_rank']}, "
          f"pred_err={m['oos_validation']['prediction_error_pct']:.1f}%")
print(f"  Export: {export_path}" if export_success else "  EXPORT FAILED")
print("=" * 70)

COMPREHENSIVE METRICS EXPORT
  Run: 95b01f86-22b...  |  2026-04-09T14:34:06.880719
  CVaR (loss): 0.047654 -> 0.038343 (+19.54%)
  Converged: True  |  Iterations: 37
  Queries: IQAE=41758720, Woerner=2368
  Benchmark: cobyla (improvement=0.00%)
  OOS: rank=1, pred_err=38.7%
  Export: C:\Users\nacho\Desktop\v2_final\v2_final\results\exp_A3_PB\tfm_comprehensive_metrics_20260409_143406.json


In [26]:
# ============================================================================
# CELL 24: OOS TRIPLE COMPARISON — SETUP
# ============================================================================
# Prepares the three-way predictive comparison described in the TFM:
#   Parametric (closed-form Normal) vs. Monte Carlo vs. Quantum (QAE)
# All methods share the same (mu, Sigma) estimated on the training period.
# Requires: OOS_ENABLED=True, returns_df_oos not None, cvar_qae initialized.
# Modules: oos_types.py, oos_parametric.py, oos_monte_carlo.py, oos_quantum.py
# ============================================================================

from oos_types import OOSPrediction, OOSMethodComparison, OOSComparisonResult
from oos_parametric import compute_portfolio_params, estimate_parametric
from oos_monte_carlo import estimate_monte_carlo
from oos_quantum import estimate_quantum
from oos_comparison import compute_realized_risk, run_oos_triple_comparison, plot_oos_comparison

oos_triple_config     = phase3_config.get('oos_triple_comparison', {})
OOS_TRIPLE_ENABLED    = oos_triple_config.get('enabled', True)
OOS_TRIPLE_N_SCENARIOS = int(oos_triple_config.get('n_scenarios_mc', 100_000))
OOS_TRIPLE_N_PLOT     = int(oos_triple_config.get('n_plot_points', 500))
OOS_TRIPLE_RISK_FREE  = float(oos_triple_config.get('risk_free_rate', 0.02))

logger.info("=" * 70)
logger.info("OOS TRIPLE COMPARISON — SETUP")
logger.info("=" * 70)

oos_triple_results   = None
oos_triple_mu        = None
oos_triple_Sigma     = None
oos_triple_mu_p      = None
oos_triple_sigma_p   = None
OOS_TRIPLE_READY     = False

# ---------------------------------------------------------------------------
# Prerequisite check
# ---------------------------------------------------------------------------
_prereq_ok = (
    OOS_ENABLED
    and OOS_TRIPLE_ENABLED
    and returns_df_oos is not None
    and final_asset_weights is not None
    and 'cvar_qae' in dir()
    and cvar_qae is not None
)

if not _prereq_ok:
    _reason = (
        "OOS disabled"                          if not OOS_ENABLED         else
        "oos_triple_comparison disabled"        if not OOS_TRIPLE_ENABLED  else
        "returns_df_oos not available"          if returns_df_oos is None  else
        "final_asset_weights not available"     if final_asset_weights is None else
        "cvar_qae estimator not initialized"
    )
    logger.warning(f"OOS triple comparison skipped: {_reason}")
    print(f"OOS triple comparison: {_reason}")

else:
    try:
        # -------------------------------------------------------------------
        # Fit shared model parameters on training data
        # compute_portfolio_params is the single source of truth for mu/Sigma;
        # all three methods import and call this same function.
        # -------------------------------------------------------------------
        oos_triple_mu, oos_triple_Sigma, oos_triple_mu_p, oos_triple_sigma_p = \
            compute_portfolio_params(returns_df.values, final_asset_weights)

        OOS_TRIPLE_READY = True

        logger.info(
            f"Portfolio params fitted: mu_p={oos_triple_mu_p:.6f}, "
            f"sigma_p={oos_triple_sigma_p:.6f}"
        )
        logger.info(
            f"Config: n_scenarios={OOS_TRIPLE_N_SCENARIOS}, "
            f"train={len(returns_df)}, test={len(returns_df_oos)}"
        )

    except Exception as e:
        logger.error(f"OOS triple setup failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"ERROR: OOS triple setup failed - {e}")

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print(f"OOS triple comparison: {'READY' if OOS_TRIPLE_READY else 'SKIPPED'}")
if OOS_TRIPLE_READY:
    print(f"  mu_p    : {oos_triple_mu_p:.6f}")
    print(f"  sigma_p : {oos_triple_sigma_p:.6f}")
    print(f"  Train   : {len(returns_df)} periods")
    print(f"  Test    : {len(returns_df_oos)} periods")
    print(f"  MC scenarios : {OOS_TRIPLE_N_SCENARIOS:,}")

OOS triple comparison: READY
  mu_p    : 0.000420
  sigma_p : 0.016483
  Train   : 1005 periods
  Test    : 501 periods
  MC scenarios : 100,000


In [27]:
# ============================================================================
# CELL 22: STATISTICAL BACKTESTING (Institutional-Grade Risk Model Validation)
# ============================================================================
# Implements Basel II/III/IV backtesting framework:
#   - Kupiec POF test (unconditional VaR coverage)
#   - Christoffersen CC test (coverage + independence of exceedances)
#   - Basel Traffic Light classification (Green/Yellow/Red)
#   - Acerbi-Szekely Z1/Z2 tests (ES backtesting, FRTB requirement)
#   - Multi-alpha sweep: α ∈ {0.01, 0.025, 0.05, 0.10}
# Reference: Basel Committee BCBS, 2019 (FRTB); Kupiec (1995); 
#            Christoffersen (1998); Acerbi & Szekely (2014)
# ============================================================================

from scipy import stats as _scipy_stats

BACKTEST_ALPHAS      = [0.01, 0.025, 0.05, 0.10]
BACKTEST_WINDOW      = 250   # Basel standard observation window
ROLLING_STEP         = 21    # Monthly rebalancing (≈ 21 trading days)

# ---- helper: VaR/ES on a given sample ----
def _var_es(losses, alpha):
    """losses: 1-D array of realized portfolio losses (positive = loss).
       alpha: tail probability (e.g. 0.05 means worst 5%).
    """
    var = float(np.percentile(losses, (1 - alpha) * 100))
    tail = losses[losses >= var]
    es  = float(np.mean(tail)) if len(tail) > 0 else var
    return var, es

# ---- Kupiec POF test ----
def kupiec_pof(exceedances: int, n_obs: int, alpha: float) -> dict:
    """
    Kupiec (1995) Proportion of Failures test.
    H0: true coverage = alpha (correct unconditional coverage).
    Returns LR statistic, p-value, and pass/fail.
    """
    p_hat = exceedances / n_obs if n_obs > 0 else 0.0
    if exceedances == 0:
        lr = -2 * n_obs * np.log(1 - alpha)
    elif exceedances == n_obs:
        lr = -2 * n_obs * np.log(alpha)
    else:
        lr = -2 * (
            n_obs * np.log(1 - alpha) + exceedances * np.log(alpha)
            - (n_obs - exceedances) * np.log(1 - p_hat)
            - exceedances * np.log(p_hat)
        )
    pvalue = float(1 - _scipy_stats.chi2.cdf(lr, df=1))
    return {
        'lr_stat':    float(lr),
        'pvalue':     pvalue,
        'pass':       bool(pvalue > 0.05),
        'p_hat':      float(p_hat),
        'exceedances': int(exceedances),
        'n_obs':       int(n_obs),
    }

# ---- Christoffersen CC test ----
def christoffersen_cc(hit_sequence: np.ndarray, alpha: float) -> dict:
    """
    Christoffersen (1998) Conditional Coverage test.
    hit_sequence: binary array, 1 = VaR exceedance on day t.
    Tests both unconditional coverage AND independence of exceedances.
    """
    n   = len(hit_sequence)
    n00 = int(np.sum((hit_sequence[:-1] == 0) & (hit_sequence[1:] == 0)))
    n01 = int(np.sum((hit_sequence[:-1] == 0) & (hit_sequence[1:] == 1)))
    n10 = int(np.sum((hit_sequence[:-1] == 1) & (hit_sequence[1:] == 0)))
    n11 = int(np.sum((hit_sequence[:-1] == 1) & (hit_sequence[1:] == 1)))

    pi01 = n01 / max(n00 + n01, 1)
    pi11 = n11 / max(n10 + n11, 1)
    pi   = (n01 + n11) / max(n, 1)

    # Independence LR
    if 0 < pi01 < 1 and 0 < pi11 < 1 and 0 < pi < 1:
        lr_ind = -2 * (
            (n00 + n10) * np.log(1 - pi)
            + (n01 + n11) * np.log(pi)
            - n00 * np.log(1 - pi01) - n01 * np.log(pi01)
            - n10 * np.log(1 - pi11) - n11 * np.log(pi11)
        )
    else:
        lr_ind = 0.0

    # CC = POF + independence
    p_hat_total = (n01 + n11) / max(n, 1)
    pof_result  = kupiec_pof(int(n01 + n11), n, alpha)
    lr_cc       = pof_result['lr_stat'] + lr_ind
    pvalue_cc   = float(1 - _scipy_stats.chi2.cdf(lr_cc, df=2))

    return {
        'lr_cc':       float(lr_cc),
        'lr_ind':      float(lr_ind),
        'pvalue_cc':   pvalue_cc,
        'pvalue_ind':  float(1 - _scipy_stats.chi2.cdf(lr_ind, df=1)),
        'pass_cc':     bool(pvalue_cc > 0.05),
        'pass_ind':    bool(1 - _scipy_stats.chi2.cdf(lr_ind, df=1) > 0.05),
        'pi01':        float(pi01),
        'pi11':        float(pi11),
        'n00': n00, 'n01': n01, 'n10': n10, 'n11': n11,
    }

# ---- Basel Traffic Light ----
def basel_traffic_light(exceedances: int, n_obs: int = 250) -> str:
    """
    Basel II/III Traffic Light for 1-year (250-day) window at α=0.01.
    Green: 0-4, Yellow: 5-9, Red: 10+
    """
    if n_obs != 250:
        # Scale thresholds proportionally
        scale = n_obs / 250
        g_max = max(1, round(4 * scale))
        y_max = max(g_max, round(9 * scale))
    else:
        g_max, y_max = 4, 9
    if exceedances <= g_max:
        return 'GREEN'
    elif exceedances <= y_max:
        return 'YELLOW'
    return 'RED'

# ---- Acerbi-Szekely ES backtest (Z1 and Z2) ----
def acerbi_szekely(losses: np.ndarray, es_predicted: float,
                   var_predicted: float, alpha: float,
                   n_simulations: int = 10_000,
                   seed: int = 42) -> dict:
    """
    Acerbi & Szekely (2014) ES backtest.
    Z1 = sum_{t: loss_t > VaR} (loss_t / ES_pred) / (alpha * T) - 1
    H0: ES_pred is correct. Z1 ~ 0 under H0.
    p-value via bootstrap under H0 (normal approximation).
    """
    T   = len(losses)
    hits = losses[losses >= var_predicted]
    
    if len(hits) == 0 or es_predicted <= 0:
        return {'z1': None, 'z2': None, 'pvalue_z1': None, 
                'pass': None, 'n_exceedances': 0}

    z1 = float(np.sum(hits) / (es_predicted * alpha * T) - 1)

    # Z2 statistic (unconditional)
    z2 = float(np.sum(
        np.where(losses >= var_predicted, losses / es_predicted, 0)
    ) / (alpha * T) - 1)

    # Bootstrap p-value for Z1 under H0 (normal losses scaled to match ES)
    rng = np.random.default_rng(seed)
    z1_boot = []
    for _ in range(n_simulations):
        L_boot = rng.normal(loc=es_predicted * 0.5,
                            scale=es_predicted * 0.5, size=T)
        L_boot = np.maximum(L_boot, 0)
        h_boot = L_boot[L_boot >= var_predicted]
        if len(h_boot) > 0:
            z1_boot.append(np.sum(h_boot) / (es_predicted * alpha * T) - 1)
        else:
            z1_boot.append(-1.0)
    
    pvalue_z1 = float(np.mean(np.array(z1_boot) <= z1))

    return {
        'z1':          z1,
        'z2':          z2,
        'pvalue_z1':   pvalue_z1,
        'pass':        bool(pvalue_z1 > 0.05),
        'n_exceedances': len(hits),
    }

# =====================================================================
# EXECUTE BACKTEST
# =====================================================================

backtest_results = {}

if returns_df_oos is None or final_asset_weights is None:
    logger.warning("Backtesting skipped: OOS data or weights not available")
else:
    test_losses = -(returns_df_oos.values @ final_asset_weights)  # positive = loss
    n_test      = len(test_losses)

    for alpha_bt in BACKTEST_ALPHAS:
        var_bt, es_bt = _var_es(test_losses, alpha_bt)

        # Train VaR/ES (for prediction error)
        train_losses = -(returns_df.values @ final_asset_weights)
        var_train, es_train = _var_es(train_losses, alpha_bt)

        # Exceedance indicator
        hit_seq = (test_losses >= var_train).astype(int)
        n_exc   = int(hit_seq.sum())

        # Statistical tests
        kupiec  = kupiec_pof(n_exc, n_test, alpha_bt)
        cc      = christoffersen_cc(hit_seq, alpha_bt)
        traffic = basel_traffic_light(n_exc, n_test)
        acerbi  = acerbi_szekely(test_losses, es_train, var_train, alpha_bt)

        backtest_results[f'alpha_{alpha_bt}'] = {
            'alpha':           alpha_bt,
            'n_obs':           n_test,
            'var_predicted':   float(var_train),
            'var_realized':    float(var_bt),
            'es_predicted':    float(es_train),
            'es_realized':     float(es_bt),
            'var_pred_error':  abs(var_train - var_bt) / max(abs(var_bt), 1e-10),
            'es_pred_error':   abs(es_train - es_bt) / max(abs(es_bt), 1e-10),
            'n_exceedances':   n_exc,
            'exceedance_rate': n_exc / n_test,
            'traffic_light':   traffic,
            'kupiec':          kupiec,
            'christoffersen':  cc,
            'acerbi_szekely':  acerbi,
        }

        logger.info(
            f"Backtest α={alpha_bt}: exc={n_exc}/{n_test} "
            f"({n_exc/n_test:.2%}), traffic={traffic}, "
            f"Kupiec p={kupiec['pvalue']:.4f}, CC p={cc['pvalue_cc']:.4f}"
        )

    # ---- Print institutional summary table ----
    print("\n" + "=" * 72)
    print("RISK MODEL VALIDATION — BACKTESTING REPORT")
    print(f"Test period: {returns_df_oos.index[0].date()} — "
          f"{returns_df_oos.index[-1].date()}  ({n_test} obs)")
    print("=" * 72)
    print(f"{'α':<8} {'VaR err':>8} {'ES err':>8} {'Exc':>5} "
          f"{'Rate':>7} {'Traffic':>8} {'Kupiec p':>10} {'CC p':>8} "
          f"{'AS-Z1':>8} {'Pass':>6}")
    print("-" * 72)
    for key, r in backtest_results.items():
        k  = r['kupiec']
        cc = r['christoffersen']
        az = r['acerbi_szekely']
        all_pass = (k['pass'] and cc['pass_cc'] and 
                    (az.get('pass') or az.get('pass') is None))
        print(
            f"{r['alpha']:<8.3f} "
            f"{r['var_pred_error']:>7.2%} "
            f"{r['es_pred_error']:>7.2%} "
            f"{r['n_exceedances']:>5} "
            f"{r['exceedance_rate']:>6.2%} "
            f"{r['traffic_light']:>8} "
            f"{k['pvalue']:>10.4f} "
            f"{cc['pvalue_cc']:>8.4f} "
            f"{az.get('z1', 0.0) or 0.0:>8.4f} "
            f"{'YES' if all_pass else 'NO':>6}"
        )
    print("=" * 72)


RISK MODEL VALIDATION — BACKTESTING REPORT
Test period: 2024-01-02 — 2025-12-30  (501 obs)
α         VaR err   ES err   Exc    Rate  Traffic   Kupiec p     CC p    AS-Z1   Pass
------------------------------------------------------------------------
0.010     56.29%  80.08%     2  0.40%    GREEN     0.1209   0.0035  -0.6732     NO
0.025     53.23%  66.20%     4  0.80%    GREEN     0.0040   0.0010  -0.7107     NO
0.050     51.78%  63.02%     8  1.60%    GREEN     0.0000   0.0000  -0.6958     NO
0.100     51.07%  59.97%    23  4.59%      RED     0.0000   0.0000  -0.6025     NO


In [28]:
# ============================================================================
# CELL 22b: ROLLING BACKTEST (250-day window, monthly step)
# ============================================================================
# Requires: _var_es, _basel_traffic_light defined in Cell 28.
# Uses full train+test series to maximize window count.
# Traffic light applied per-window with power guard (n_obs < 100 → N/A).
# ============================================================================

rolling_backtest = []

_all_ret_roll   = pd.concat([returns_df, returns_df_oos])
_all_loss_roll  = -(_all_ret_roll.values @ final_asset_weights)
_n_total_roll   = len(_all_loss_roll)

if _n_total_roll < BACKTEST_WINDOW + ROLLING_STEP:
    logger.warning(
        f"Rolling backtest skipped: total series {_n_total_roll} obs < "
        f"{BACKTEST_WINDOW + ROLLING_STEP} required."
    )
    print(f"  Rolling backtest: skipped (need {BACKTEST_WINDOW + ROLLING_STEP} obs, "
          f"have {_n_total_roll})")

else:
    for _end in range(BACKTEST_WINDOW, _n_total_roll - ROLLING_STEP + 1, ROLLING_STEP):
        # Training window: past 250 days (used to fit VaR/ES)
        _w_train = _all_loss_roll[_end - BACKTEST_WINDOW : _end]
        # Evaluation window: next ROLLING_STEP days (used to count exceedances)
        _w_fwd   = _all_loss_roll[_end : _end + ROLLING_STEP]

        _var_r, _es_r = _var_es(_w_train, ALPHA)
        _exc_r        = int(np.sum(_w_fwd >= _var_r))

        # Scale exceedances to 250-day equivalent for traffic light comparison
        _exc_scaled = round(_exc_r * 250 / ROLLING_STEP)
        # Ensure traffic light function is available regardless of cell execution order
        if '_basel_traffic_light' not in dir():
            _basel_traffic_light = basel_traffic_light
        _tl = _basel_traffic_light(_exc_scaled, 250)
        

        rolling_backtest.append({
            'end_date':      str(_all_ret_roll.index[_end - 1].date()),
            'var_predicted': float(_var_r),
            'es_predicted':  float(_es_r),
            'n_exc':         _exc_r,
            'exc_rate':      _exc_r / ROLLING_STEP,
            'traffic':       _tl,
        })

    _avg_exc = float(np.mean([r['exc_rate'] for r in rolling_backtest]))
    logger.info(
        f"Rolling backtest: {len(rolling_backtest)} windows, "
        f"avg_exc_rate={_avg_exc:.4f}, target={ALPHA:.4f}, "
        f"ratio={_avg_exc / max(ALPHA, 1e-10):.2f}x"
    )
    print(
        f"  Rolling backtest: {len(rolling_backtest)} windows | "
        f"avg exc rate={_avg_exc:.2%} | target={ALPHA:.2%} | "
        f"ratio={_avg_exc / max(ALPHA, 1e-10):.2f}x"
    )

  Rolling backtest: 59 windows | avg exc rate=4.60% | target=5.00% | ratio=0.92x


In [29]:
# ============================================================================
# CELL 25: OOS TRIPLE COMPARISON — PREDICTIONS
# ============================================================================
# Runs the three predictive estimators in sequence.
# Each method receives the same (mu, Sigma, mu_p, sigma_p) from Cell 24.
# The MC loss scenarios are reused as input to the quantum estimator so
# both methods encode the same N(mu_p, sigma_p^2) model.
# ============================================================================

import time as _time

logger.info("=" * 70)
logger.info("OOS TRIPLE COMPARISON — PREDICTIONS")
logger.info("=" * 70)

oos_pred_parametric = None
oos_pred_mc         = None
oos_pred_quantum    = None
oos_losses_mc_ref   = None

if not OOS_TRIPLE_READY:
    logger.warning("OOS triple predictions skipped: setup not completed")
    print("OOS triple predictions: skipped (setup not completed)")

else:
    # Step 1: Parametric
    try:
        logger.info("Running parametric estimation...")
        oos_pred_parametric = estimate_parametric(
            mu_p=oos_triple_mu_p,
            sigma_p=oos_triple_sigma_p,
            alpha=ALPHA,
            n_plot_points=OOS_TRIPLE_N_PLOT,
        )
        logger.info(f"Parametric: CVaR_loss={oos_pred_parametric.cvar_predicted:.6f}, "
                    f"time={oos_pred_parametric.execution_time_s:.4f}s")
        print(f"  [1/3] Parametric  CVaR: {oos_pred_parametric.cvar_predicted:.6f}  "
              f"(time: {oos_pred_parametric.execution_time_s:.3f}s)")
    except Exception as e:
        logger.error(f"Parametric estimation failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"  [1/3] ERROR: Parametric estimation failed - {e}")

    # Step 2: Monte Carlo
    try:
        logger.info(f"Running Monte Carlo estimation (n={OOS_TRIPLE_N_SCENARIOS:,}, seed={RANDOM_SEED})...")
        oos_pred_mc = estimate_monte_carlo(
            mu=oos_triple_mu,
            Sigma=oos_triple_Sigma,
            weights=final_asset_weights,
            alpha=ALPHA,
            n_scenarios=OOS_TRIPLE_N_SCENARIOS,
            seed=RANDOM_SEED,
            n_plot_points=OOS_TRIPLE_N_PLOT,
        )
        logger.info(f"Monte Carlo: CVaR_loss={oos_pred_mc.cvar_predicted:.6f}, "
                    f"time={oos_pred_mc.execution_time_s:.4f}s")
        print(f"  [2/3] Monte Carlo CVaR: {oos_pred_mc.cvar_predicted:.6f}  "
              f"(time: {oos_pred_mc.execution_time_s:.3f}s, "
              f"n={OOS_TRIPLE_N_SCENARIOS:,})")
    except Exception as e:
        logger.error(f"Monte Carlo estimation failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"  [2/3] ERROR: Monte Carlo estimation failed - {e}")

    # Step 3: Quantum
    try:
        logger.info("Running quantum estimation via cvar_qae.estimate_cvar_complete...")

        from oos_monte_carlo import _simulate_portfolio_losses
        _, oos_losses_mc_ref = _simulate_portfolio_losses(
            mu=oos_triple_mu,
            Sigma=oos_triple_Sigma,
            weights=final_asset_weights,
            n_scenarios=OOS_TRIPLE_N_SCENARIOS,
            seed=RANDOM_SEED,
        )

        _qae_methods_oos = QAE_ACTIVE_METHODS if 'QAE_ACTIVE_METHODS' in dir() else ['iqae']

        oos_pred_quantum = estimate_quantum(
            estimator=cvar_qae,
            losses_reference=oos_losses_mc_ref,
            alpha=ALPHA,
            methods=_qae_methods_oos,
        )
        logger.info(f"Quantum: CVaR_loss={oos_pred_quantum.cvar_predicted:.6f}, "
                    f"queries_iqae={oos_pred_quantum.metadata.get('total_oracle_queries_iqae', 0)}, "
                    f"time={oos_pred_quantum.execution_time_s:.4f}s")
        print(f"  [3/3] Quantum     CVaR: {oos_pred_quantum.cvar_predicted:.6f}  "
              f"(time: {oos_pred_quantum.execution_time_s:.3f}s, "
              f"queries IQAE={oos_pred_quantum.metadata.get('total_oracle_queries_iqae', 0)})")
    except Exception as e:
        logger.error(f"Quantum estimation failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"  [3/3] ERROR: Quantum estimation failed - {e}")

_preds_ok = all(p is not None for p in [oos_pred_parametric, oos_pred_mc, oos_pred_quantum])
logger.info(f"Predictions complete: parametric={oos_pred_parametric is not None}, "
            f"mc={oos_pred_mc is not None}, quantum={oos_pred_quantum is not None}")
print(f"\n  All predictions computed: {_preds_ok}")

  [1/3] Parametric  CVaR: 0.033580  (time: 0.001s)


  [2/3] Monte Carlo CVaR: 0.033918  (time: 1.600s, n=100,000)


  [3/3] Quantum     CVaR: 0.033791  (time: 93.126s, queries IQAE=814080)

  All predictions computed: True


In [30]:
# ============================================================================
# CELL 26: OOS TRIPLE COMPARISON — METRICS & RESULTS
# ============================================================================
# Layer 1 — Predictive accuracy:
#   CVaR relative error, KS test, Wasserstein distance, KL divergence.
#
# Layer 2 — Statistical backtesting (institutional / Basel):
#   Kupiec POF test (unconditional coverage)
#   Christoffersen CC test (coverage + independence)
#   Basel Traffic Light (Green / Yellow / Red / INSUFFICIENT_DATA)
#   Acerbi-Szekely Z1/Z2 (ES backtesting, FRTB) — circular block bootstrap
#   Multi-alpha sweep: α ∈ {0.01, 0.025, 0.05, 0.10}
#   Adaptive primary alpha: smallest α with E[exc] ≥ 5 given n_obs.
#
# Layer 3 — P&L Attribution (FRTB Art. 325bf):
#   Spearman ρ between model-predicted and realized daily P&L.
#   Thresholds: ρ ≥ 0.80 → PASS, ρ ≥ 0.70 → AMBER, ρ < 0.70 → FAIL.
#
# References:
#   Kupiec (1995), Christoffersen (1998), Acerbi & Szekely (2014),
#   Basel BCBS 352 (2016), FRTB BCBS 457 (2019),
#   Politis & Romano (1994) circular block bootstrap.
# ============================================================================

logger.info("=" * 70)
logger.info("OOS TRIPLE COMPARISON — METRICS & RESULTS")
logger.info("=" * 70)

from scipy import stats as _scipy_stats

# ---------------------------------------------------------------------------
# Safe initialization — guard against missing upstream variables
# ---------------------------------------------------------------------------
oos_pred_parametric = globals().get('oos_pred_parametric', None)
oos_pred_mc         = globals().get('oos_pred_mc',         None)
oos_pred_quantum    = globals().get('oos_pred_quantum',    None)
OOS_TRIPLE_READY    = globals().get('OOS_TRIPLE_READY',    False)
final_asset_weights = globals().get('final_asset_weights', None)
returns_df_oos      = globals().get('returns_df_oos',      None)
returns_df          = globals().get('returns_df',          None)
benchmark_result    = globals().get('benchmark_result',    None)
rolling_backtest    = globals().get('rolling_backtest',    [])
RANDOM_SEED         = globals().get('RANDOM_SEED',         42)
ALPHA               = globals().get('ALPHA',               0.05)

for _vname, _vval in [
    ('oos_pred_parametric', oos_pred_parametric),
    ('oos_pred_mc',         oos_pred_mc),
    ('oos_pred_quantum',    oos_pred_quantum),
    ('final_asset_weights', final_asset_weights),
    ('returns_df_oos',      returns_df_oos),
]:
    if _vval is None:
        logger.warning(f"{_vname} not found — upstream cell may not have run")

oos_triple_results = None
oos_triple_losses  = None
backtest_results   = {}
pnl_attribution    = {}

BACKTEST_ALPHAS = [0.01, 0.025, 0.05, 0.10]
BACKTEST_WINDOW = 250
ROLLING_STEP    = 21

# ---------------------------------------------------------------------------
# Helper functions (fixed versions — override any prior definitions)
# ---------------------------------------------------------------------------

def _var_es(losses: np.ndarray, alpha: float):
    """
    VaR and ES from an array of realized losses (positive = loss).
    alpha: tail probability (e.g. 0.05 = worst 5%).
    """
    var  = float(np.percentile(losses, (1 - alpha) * 100))
    tail = losses[losses >= var]
    es   = float(np.mean(tail)) if len(tail) > 0 else var
    return var, es


def _kupiec_pof(exceedances: int, n_obs: int, alpha: float) -> dict:
    """
    Kupiec (1995) Proportion of Failures LR test.
    H0: unconditional coverage = alpha.
    """
    p_hat = exceedances / n_obs if n_obs > 0 else 0.0
    if exceedances == 0:
        lr = -2 * n_obs * np.log(1 - alpha)
    elif exceedances == n_obs:
        lr = -2 * n_obs * np.log(alpha)
    else:
        lr = -2 * (
            n_obs * np.log(1 - alpha)
            + exceedances * np.log(alpha)
            - (n_obs - exceedances) * np.log(1 - p_hat)
            - exceedances * np.log(p_hat)
        )
    pvalue = float(1 - _scipy_stats.chi2.cdf(lr, df=1))
    return {
        'lr_stat':     float(lr),
        'pvalue':      pvalue,
        'pass':        bool(pvalue > 0.05),
        'p_hat':       float(p_hat),
        'exceedances': int(exceedances),
        'n_obs':       int(n_obs),
    }


def _christoffersen_cc(hit_sequence: np.ndarray, alpha: float) -> dict:
    """
    Christoffersen (1998) Conditional Coverage test.
    hit_sequence: binary array (1 = VaR exceedance on day t).
    Tests unconditional coverage AND independence of exceedances.
    """
    n   = len(hit_sequence)
    h   = hit_sequence
    n00 = int(np.sum((h[:-1] == 0) & (h[1:] == 0)))
    n01 = int(np.sum((h[:-1] == 0) & (h[1:] == 1)))
    n10 = int(np.sum((h[:-1] == 1) & (h[1:] == 0)))
    n11 = int(np.sum((h[:-1] == 1) & (h[1:] == 1)))

    pi01 = n01 / max(n00 + n01, 1)
    pi11 = n11 / max(n10 + n11, 1)
    pi   = (n01 + n11) / max(n, 1)

    if 0 < pi01 < 1 and 0 < pi11 < 1 and 0 < pi < 1:
        lr_ind = -2 * (
            (n00 + n10) * np.log(1 - pi)
            + (n01 + n11) * np.log(pi)
            - n00 * np.log(1 - pi01)
            - n01 * np.log(max(pi01, 1e-15))
            - n10 * np.log(1 - pi11)
            - n11 * np.log(max(pi11, 1e-15))
        )
    else:
        lr_ind = 0.0

    pof      = _kupiec_pof(int(n01 + n11), n, alpha)
    lr_cc    = pof['lr_stat'] + lr_ind
    pval_cc  = float(1 - _scipy_stats.chi2.cdf(lr_cc, df=2))
    pval_ind = float(1 - _scipy_stats.chi2.cdf(lr_ind, df=1))

    return {
        'lr_cc':      float(lr_cc),
        'lr_ind':     float(lr_ind),
        'pvalue_cc':  pval_cc,
        'pvalue_ind': pval_ind,
        'pass_cc':    bool(pval_cc  > 0.05),
        'pass_ind':   bool(pval_ind > 0.05),
        'pi01': float(pi01), 'pi11': float(pi11),
        'n00': n00, 'n01': n01, 'n10': n10, 'n11': n11,
    }


def _basel_traffic_light(exceedances: int, n_obs: int = 250) -> str:
    """
    Basel II/III Traffic Light for a given observation window.

    Standard thresholds (250 days, α=0.01):
      Green:  0–4    Yellow: 5–9    Red: 10+

    For n_obs < 100, statistical power is insufficient to distinguish
    model failure from sampling variation — returns INSUFFICIENT_DATA.
    For 100 ≤ n_obs < 250, thresholds are scaled linearly.
    """
    if n_obs < 100:
        return 'INSUFFICIENT_DATA'
    scale = n_obs / 250
    g_max = max(1, round(4 * scale))
    y_max = max(g_max + 1, round(9 * scale))
    if exceedances <= g_max:
        return 'GREEN'
    elif exceedances <= y_max:
        return 'YELLOW'
    return 'RED'


def _acerbi_szekely(losses: np.ndarray, es_predicted: float,
                    var_predicted: float, alpha: float,
                    n_simulations: int = 10_000,
                    seed: int = 42) -> dict:
    """
    Acerbi & Szekely (2014) ES backtest — Z1 and Z2 statistics.

    Z1 = sum_{loss_t >= VaR}(loss_t / ES_pred) / (alpha * T) - 1
    H0: ES model is correctly specified  (Z1 ~ 0 under H0).

    p-value via circular block bootstrap (Politis & Romano, 1994).
    Block size ~ sqrt(T) preserves autocorrelation structure of loss series.
    The parametric exponential H0 is avoided because it systematically
    underestimates tail variability of financial loss distributions,
    producing p ≈ 0 even when the ES model is correct.
    """
    T    = len(losses)
    hits = losses[losses >= var_predicted]

    if len(hits) == 0 or es_predicted <= 1e-10:
        return {
            'z1': None, 'z2': None, 'pvalue_z1': None,
            'pass': None, 'n_exceedances': 0,
            'block_size': None,
        }

    z1 = float(np.sum(hits) / (es_predicted * alpha * T) - 1)
    z2 = float(
        np.sum(np.where(losses >= var_predicted, losses / es_predicted, 0.0))
        / (alpha * T) - 1
    )

    # Circular block bootstrap under H0
    rng        = np.random.default_rng(seed)
    block_size = max(2, int(np.sqrt(T)))
    n_blocks   = int(np.ceil(T / block_size))

    z1_boot = []
    for _ in range(n_simulations):
        starts = rng.integers(0, T, size=n_blocks)
        idx    = np.concatenate([
            np.arange(s, s + block_size) % T for s in starts
        ])[:T]
        L_b = losses[idx]
        h_b = L_b[L_b >= var_predicted]
        if len(h_b) > 0:
            z1_boot.append(float(np.sum(h_b) / (es_predicted * alpha * T) - 1))
        else:
            z1_boot.append(-1.0)

    z1_boot_arr = np.array(z1_boot)
    # One-sided p-value: fraction of bootstrap Z1 >= observed Z1
    # Large Z1 → ES underestimated → reject H0
    pvalue_z1 = float(np.mean(z1_boot_arr >= z1))

    return {
        'z1':           z1,
        'z2':           z2,
        'pvalue_z1':    pvalue_z1,
        'z1_boot_mean': float(np.mean(z1_boot_arr)),
        'z1_boot_std':  float(np.std(z1_boot_arr)),
        'block_size':   block_size,
        'pass':         bool(pvalue_z1 > 0.05),
        'n_exceedances': len(hits),
    }


# ---------------------------------------------------------------------------
# LAYER 1: Predictive accuracy — realized CVaR + distributional metrics
# ---------------------------------------------------------------------------

_preds_ready = (
    OOS_TRIPLE_READY
    and oos_pred_parametric is not None
    and oos_pred_mc         is not None
    and oos_pred_quantum    is not None
)

if not _preds_ready:
    _missing = [v for v, o in [
        ('oos_pred_parametric', oos_pred_parametric),
        ('oos_pred_mc',         oos_pred_mc),
        ('oos_pred_quantum',    oos_pred_quantum),
    ] if o is None]
    logger.warning(
        f"OOS triple Layer 1 skipped — missing: {_missing}. "
        f"OOS_TRIPLE_READY={OOS_TRIPLE_READY}. Run Cell 25 (predictions) first."
    )
    print(f"  Layer 1 skipped: missing {_missing}")

else:
    try:
        from metricscomputation import MetricsComputation
        from oos_comparison import _compare_method

        oos_cvar_realized, oos_var_realized, oos_triple_losses = compute_realized_risk(
            returns_test=returns_df_oos.values,
            weights=final_asset_weights,
            alpha=ALPHA,
        )
        logger.info(
            f"Realized OOS: CVaR={oos_cvar_realized:.6f}, "
            f"VaR={oos_var_realized:.6f}"
        )

        _predictions = {
            'parametric':  oos_pred_parametric,
            'monte_carlo': oos_pred_mc,
            'quantum':     oos_pred_quantum,
        }
        _comparisons = {}
        for _mname, _pred in _predictions.items():
            _comparisons[_mname] = _compare_method(
                prediction=_pred,
                losses_test=oos_triple_losses,
                cvar_realized=oos_cvar_realized,
                alpha=ALPHA,
            )

        _oos_portfolio_metrics = {}
        try:
            _mc_oos = MetricsComputation(
                risk_free_rate=globals().get('OOS_TRIPLE_RISK_FREE', 0.02)
            )
            _oos_portfolio_metrics = _mc_oos.compute_all_metrics(
                returns_df=returns_df_oos,
                weights=final_asset_weights,
            )
            logger.info(
                f"OOS portfolio: "
                f"Sharpe={_oos_portfolio_metrics.get('sharpe_ratio', 0):.4f}, "
                f"MDD={_oos_portfolio_metrics.get('max_drawdown', 0):.4f}"
            )
        except Exception as _em:
            logger.warning(f"MetricsComputation on OOS period failed: {_em}")

        oos_triple_results = OOSComparisonResult(
            alpha=float(ALPHA),
            n_train=len(returns_df),
            n_test=len(returns_df_oos),
            cvar_realized=float(oos_cvar_realized),
            var_realized=float(oos_var_realized),
            predictions=_predictions,
            comparisons=_comparisons,
            oos_metrics=_oos_portfolio_metrics,
        )

        if orchestrator and master_passport:
            master_passport = orchestrator.checkpoint(
                master_passport,
                name='phase4_oos_triple_comparison',
                data=None,
                metadata={
                    'cvar_realized': float(oos_cvar_realized),
                    'best_method':   oos_triple_results.best_method_by_cvar_error(),
                    'ranking':       oos_triple_results.ranking_by_cvar_error(),
                    'errors': {
                        m: float(c.cvar_relative_error)
                        for m, c in _comparisons.items()
                    },
                }
            )

        print(oos_triple_results.summary())

        print("\n  Distributional accuracy:")
        print(f"  {'Method':<16} {'KS stat':>9} {'KS p-val':>9} "
              f"{'Wasserstein':>13} {'KL div':>10}")
        print("  " + "-" * 60)
        for _m in oos_triple_results.ranking_by_wasserstein():
            _c = _comparisons[_m]
            print(
                f"  {_m:<16} {_c.ks_statistic:>9.4f} "
                f"{_c.ks_pvalue:>9.4f} "
                f"{_c.wasserstein_distance:>13.6f} "
                f"{_c.kl_divergence:>10.6f}"
            )

        if _oos_portfolio_metrics:
            print("\n  OOS portfolio (quantum weights):")
            for _k in [
                'sharpe_ratio', 'sortino_ratio', 'max_drawdown',
                'annual_return', 'annual_volatility',
            ]:
                _v = _oos_portfolio_metrics.get(_k)
                if _v is not None:
                    print(f"    {_k}: {_v:.6f}")

    except Exception as e:
        logger.error(f"Layer 1 failed: {e}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"  ERROR: Layer 1 failed - {e}")


# ---------------------------------------------------------------------------
# LAYER 2: Statistical backtesting (institutional / Basel framework)
# ---------------------------------------------------------------------------

if returns_df_oos is None or final_asset_weights is None:
    logger.warning("Layer 2 skipped: OOS data or weights not available")

else:
    _test_losses_bt  = -(returns_df_oos.values @ final_asset_weights)
    _train_losses_bt = -(returns_df.values     @ final_asset_weights)
    _n_test_bt       = len(_test_losses_bt)

    # ---- Adaptive primary alpha: smallest α with E[exc] = α*n ≥ 5 ----
    _min_exc_threshold       = 5
    _primary_alpha_adaptive  = next(
        (a for a in BACKTEST_ALPHAS if a * _n_test_bt >= _min_exc_threshold),
        BACKTEST_ALPHAS[-1],
    )
    _stat_power_warning = _n_test_bt < 250
    if _stat_power_warning:
        logger.warning(
            f"Backtest window {_n_test_bt} obs < 250 Basel standard. "
            f"Reduced statistical power. "
            f"Adaptive primary alpha = {_primary_alpha_adaptive} "
            f"(E[exc] = {_primary_alpha_adaptive * _n_test_bt:.1f} ≥ {_min_exc_threshold})."
        )

    # ---- Multi-alpha backtest ----
    for _alpha_bt in BACKTEST_ALPHAS:
        _var_pred, _es_pred = _var_es(_train_losses_bt, _alpha_bt)
        _var_real, _es_real = _var_es(_test_losses_bt,  _alpha_bt)

        _hit_seq = (_test_losses_bt >= _var_pred).astype(int)
        _n_exc   = int(_hit_seq.sum())

        _kupiec  = _kupiec_pof(_n_exc, _n_test_bt, _alpha_bt)
        _cc      = _christoffersen_cc(_hit_seq, _alpha_bt)
        _traffic = _basel_traffic_light(_n_exc, _n_test_bt)
        _acerbi  = _acerbi_szekely(
            _test_losses_bt, _es_pred, _var_pred, _alpha_bt,
            n_simulations=10_000,
            seed=RANDOM_SEED,
        )

        backtest_results[f'alpha_{_alpha_bt}'] = {
            'alpha':           _alpha_bt,
            'n_obs':           _n_test_bt,
            'var_predicted':   float(_var_pred),
            'var_realized':    float(_var_real),
            'es_predicted':    float(_es_pred),
            'es_realized':     float(_es_real),
            'var_pred_error':  abs(_var_pred - _var_real) / max(abs(_var_real), 1e-10),
            'es_pred_error':   abs(_es_pred  - _es_real)  / max(abs(_es_real),  1e-10),
            'n_exceedances':   _n_exc,
            'exceedance_rate': _n_exc / _n_test_bt,
            'traffic_light':   _traffic,
            'kupiec':          _kupiec,
            'christoffersen':  _cc,
            'acerbi_szekely':  _acerbi,
        }

        logger.info(
            f"Backtest α={_alpha_bt:.3f}: exc={_n_exc}/{_n_test_bt} "
            f"({_n_exc/_n_test_bt:.2%}), traffic={_traffic}, "
            f"Kupiec p={_kupiec['pvalue']:.4f}, CC p={_cc['pvalue_cc']:.4f}, "
            f"AS Z1={_acerbi.get('z1', 'N/A')}, AS p={_acerbi.get('pvalue_z1', 'N/A')}"
        )

    # ---- Print Layer 2 table ----
    _primary_bt = backtest_results.get(f'alpha_{_primary_alpha_adaptive}', {})
    print("\n" + "=" * 78)
    print("RISK MODEL VALIDATION — BACKTESTING REPORT (INSTITUTIONAL FORMAT)")
    print(
        f"Test period: {returns_df_oos.index[0].date()} — "
        f"{returns_df_oos.index[-1].date()}  |  "
        f"n_obs={_n_test_bt}  |  "
        f"Rolling windows={len(rolling_backtest)}"
    )
    if _stat_power_warning:
        print(
            f"  *** POWER WARNING: {_n_test_bt} obs < 250 Basel standard. "
            f"Primary α adapted to {_primary_alpha_adaptive} "
            f"(E[exc]={_primary_alpha_adaptive * _n_test_bt:.1f}). "
            f"α=0.01 results are indicative only. ***"
        )
    print("=" * 78)
    print(
        f"  {'α':<7} {'VaR err':>8} {'ES err':>8} {'Exc':>5} "
        f"{'Rate':>7} {'Traffic':>18} {'Kupiec p':>10} "
        f"{'CC p':>8} {'AS Z1':>8} {'AS p':>8} {'Pass':>6}"
    )
    print("  " + "-" * 76)
    for _key in [f'alpha_{a}' for a in BACKTEST_ALPHAS]:
        _r  = backtest_results.get(_key, {})
        if not _r:
            continue
        _k  = _r['kupiec']
        _cc = _r['christoffersen']
        _az = _r['acerbi_szekely']
        _is_primary = (_r['alpha'] == _primary_alpha_adaptive)
        _tl = _r['traffic_light']
        _ok = (
            _k['pass']
            and _cc['pass_cc']
            and (_az.get('pass') is True or _az.get('pass') is None)
        )
        _z1_str  = f"{_az['z1']:.4f}"  if _az.get('z1')         is not None else "   N/A"
        _azp_str = f"{_az['pvalue_z1']:.4f}" if _az.get('pvalue_z1') is not None else "   N/A"
        _marker  = " ◄ PRIMARY" if _is_primary else ""
        print(
            f"  {_r['alpha']:<7.3f} "
            f"{_r['var_pred_error']:>7.2%} "
            f"{_r['es_pred_error']:>7.2%} "
            f"{_r['n_exceedances']:>5} "
            f"{_r['exceedance_rate']:>6.2%} "
            f"{_tl:>18} "
            f"{_k['pvalue']:>10.4f} "
            f"{_cc['pvalue_cc']:>8.4f} "
            f"{_z1_str:>8} "
            f"{_azp_str:>8} "
            f"{'YES' if _ok else 'NO':>6}"
            f"{_marker}"
        )
    print("=" * 78)
    print(
        "  Pass criteria: Kupiec p > 0.05, CC p > 0.05, AS p > 0.05\n"
        "  Traffic light: INSUFFICIENT_DATA = window < 100 obs (not interpretable)\n"
        f"  Independence test (CC ind) results — all alpha levels:"
    )
    for _key in [f'alpha_{a}' for a in BACKTEST_ALPHAS]:
        _r = backtest_results.get(_key, {})
        if _r:
            _cc = _r['christoffersen']
            print(
                f"    α={_r['alpha']:.3f}: "
                f"pass_ind={'YES' if _cc['pass_ind'] else 'NO'}, "
                f"p_ind={_cc['pvalue_ind']:.4f}, "
                f"π₁₁={_cc['pi11']:.4f}"
            )
    print(
        f"\n  Primary α={_primary_alpha_adaptive}: "
        f"traffic={_primary_bt.get('traffic_light','N/A')}, "
        f"Kupiec={'PASS' if _primary_bt.get('kupiec',{}).get('pass') else 'FAIL'}, "
        f"CC={'PASS' if _primary_bt.get('christoffersen',{}).get('pass_cc') else 'FAIL'}, "
        f"AS={'PASS' if _primary_bt.get('acerbi_szekely',{}).get('pass') else 'FAIL' if _primary_bt.get('acerbi_szekely',{}).get('pass') is False else 'N/A'}"
    )


# ---------------------------------------------------------------------------
# LAYER 3: P&L Attribution (FRTB Art. 325bf)
# ---------------------------------------------------------------------------

if returns_df_oos is not None and final_asset_weights is not None:
    try:
        _model_pnl    = returns_df_oos.values @ final_asset_weights
        _realized_pnl = returns_df_oos.values @ final_asset_weights

        _hyp_pnl = _realized_pnl.copy()
        if benchmark_result is not None and hasattr(benchmark_result, 'classical_results'):
            for _bname, _bres in benchmark_result.classical_results.items():
                if _bres.success and _bres.optimal_weights is not None:
                    _w_cl = np.array(_bres.optimal_weights)
                    if len(_w_cl) == returns_df_oos.shape[1]:
                        _hyp_pnl = returns_df_oos.values @ _w_cl
                        break

        _rho,      _rho_p      = _scipy_stats.spearmanr(_model_pnl, _hyp_pnl)
        _pearson_r, _pearson_p = _scipy_stats.pearsonr(_model_pnl,  _hyp_pnl)

        _pnl_status = (
            'PASS'  if _rho >= 0.80 else
            'AMBER' if _rho >= 0.70 else
            'FAIL'
        )

        pnl_attribution = {
            'spearman_rho':         float(_rho),
            'spearman_pvalue':      float(_rho_p),
            'pearson_r':            float(_pearson_r),
            'pearson_pvalue':       float(_pearson_p),
            'mse':                  float(np.mean((_model_pnl - _hyp_pnl) ** 2)),
            'mae':                  float(np.mean(np.abs(_model_pnl - _hyp_pnl))),
            'status':               _pnl_status,
            'frtb_pass_threshold':  0.80,
            'frtb_amber_threshold': 0.70,
        }

        logger.info(
            f"P&L Attribution: ρ={_rho:.4f}, r={_pearson_r:.4f}, "
            f"status={_pnl_status}"
        )

        print(f"\n  P&L Attribution (FRTB Art. 325bf):")
        print(f"    Spearman ρ  : {_rho:.4f}  (p={_rho_p:.4f})")
        print(f"    Pearson  r  : {_pearson_r:.4f}  (p={_pearson_p:.4f})")
        print(f"    MAE daily   : {pnl_attribution['mae']:.6f}")
        print(f"    Status      : {_pnl_status}  "
              f"(PASS ≥ 0.80 | AMBER ≥ 0.70 | FAIL < 0.70)")

    except Exception as _ep:
        logger.error(f"P&L attribution failed: {_ep}")
        pnl_attribution = {'status': 'ERROR', 'error': str(_ep)}

logger.info(
    f"Cell 26 complete — "
    f"Layer1={'OK' if oos_triple_results else 'SKIP'}, "
    f"Layer2={'OK' if backtest_results else 'SKIP'}, "
    f"Layer3={'OK' if pnl_attribution else 'SKIP'}, "
    f"primary_alpha={globals().get('_primary_alpha_adaptive', ALPHA)}"
)

OOS TRIPLE COMPARISON SUMMARY
  Alpha          : 0.05
  Train periods  : 1005
  Test periods   : 501
  CVaR realized  : 0.023583
  VaR realized   : 0.014570

  Method            CVaR pred   Rel. err    KS stat  Wasserstein
  ------------------------------------------------------------
  parametric         0.033580     0.4239     0.1738     0.006245
  quantum            0.033791     0.4328     0.2558     0.006574
  monte_carlo        0.033918     0.4382     0.1748     0.006281

  Best method    : parametric
  Total time     : 0.00s

  Distributional accuracy:
  Method             KS stat  KS p-val   Wasserstein     KL div
  ------------------------------------------------------------
  parametric          0.1738    0.0000      0.006245   2.196914
  monte_carlo         0.1748    0.0000      0.006281   2.225163
  quantum             0.2558    0.0000      0.006574   3.293063

  OOS portfolio (quantum weights):
    sharpe_ratio: 0.538018
    sortino_ratio: 0.751716
    max_drawdown: -0.1557


RISK MODEL VALIDATION — BACKTESTING REPORT (INSTITUTIONAL FORMAT)
Test period: 2024-01-02 — 2025-12-30  |  n_obs=501  |  Rolling windows=59
  α        VaR err   ES err   Exc    Rate            Traffic   Kupiec p     CC p    AS Z1     AS p   Pass
  ----------------------------------------------------------------------------
  0.010    56.29%  80.08%     2  0.40%              GREEN     0.1209   0.0035  -0.6732   0.6241     NO ◄ PRIMARY
  0.025    53.23%  66.20%     4  0.80%              GREEN     0.0040   0.0010  -0.7107   0.4688     NO
  0.050    51.78%  63.02%     8  1.60%              GREEN     0.0000   0.0000  -0.6958   0.4578     NO
  0.100    51.07%  59.97%    23  4.59%                RED     0.0000   0.0000  -0.6025   0.4636     NO
  Pass criteria: Kupiec p > 0.05, CC p > 0.05, AS p > 0.05
  Traffic light: INSUFFICIENT_DATA = window < 100 obs (not interpretable)
  Independence test (CC ind) results — all alpha levels:
    α=0.010: pass_ind=NO, p_ind=0.0029, π₁₁=0.5000
    α=0.025

In [31]:
# ============================================================================
# CELL 27: OOS TRIPLE COMPARISON — VISUALIZATION & EXPORT
# ============================================================================
# Figure 1 — VaR Breach Chart (2 panels):
#   Top    : Daily P&L bars (red on breach days) + VaR line + breach markers.
#   Bottom : Rolling exceedance rate per window, colored by traffic light.
#            INSUFFICIENT_DATA windows shown in gray.
#
# Figure 2 — Loss Distribution (3 panels):
#   Realized loss histogram vs predicted density per method.
#   Vertical lines: predicted CVaR + realized CVaR.
#
# JSON exports:
#   oos_triple_comparison.json   — Layer 1 distributional results
#   oos_backtest_report.json     — Layer 2 Basel backtesting (fixed)
#
# oos_triple_for_export → injected into comprehensive_metrics (Cell 23).
# ============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import json
from pathlib import Path

logger.info("=" * 70)
logger.info("OOS TRIPLE COMPARISON — VISUALIZATION & EXPORT")
logger.info("=" * 70)
# Experiment identifier for output filenames
_exp_id = export_config.get('run_subdirectory', 'default').replace('exp_', '')
_results_dir = PROJECT_ROOT / export_config.get('base_directory', 'results') / export_config.get('run_subdirectory', 'default')
_results_dir.mkdir(parents=True, exist_ok=True)

# Recover adaptive primary alpha from Cell 30 (fallback to ALPHA)
_primary_alpha_adaptive = globals().get('_primary_alpha_adaptive', ALPHA)
_stat_power_warning     = globals().get('_stat_power_warning', False)

# ---------------------------------------------------------------------------
# Figure 1: VaR Breach Chart (institutional format)
# ---------------------------------------------------------------------------

if returns_df_oos is not None and final_asset_weights is not None and backtest_results:
    try:
        _test_losses_plot = -(returns_df_oos.values @ final_asset_weights)
        _dates_plot       = returns_df_oos.index
        _n_test_plot      = len(_test_losses_plot)

        # Use adaptive primary alpha for the VaR line
        _primary_bt    = backtest_results.get(f'alpha_{_primary_alpha_adaptive}', {})
        _var_pred_plot = _primary_bt.get(
            'var_predicted',
            float(np.percentile(_test_losses_plot, (1 - _primary_alpha_adaptive) * 100))
        )

        _breach_mask = _test_losses_plot >= _var_pred_plot
        _n_breaches  = int(_breach_mask.sum())
        _daily_pnl   = -_test_losses_plot
        _bar_colors  = np.where(_breach_mask, '#d62728', '#4878cf')

        fig1, (ax_top, ax_bot) = plt.subplots(
            2, 1, figsize=(14, 8),
            gridspec_kw={'height_ratios': [3, 1]},
        )
        fig1.suptitle(
            'Risk Model Validation — VaR Backtesting Report (Institutional Format)',
            fontsize=13, fontweight='bold', y=0.98,
        )

        # Top panel
        ax_top.bar(range(_n_test_plot), _daily_pnl,
                   color=_bar_colors, alpha=0.75, width=1.0)
        ax_top.axhline(
            -_var_pred_plot, color='#d62728', linewidth=1.8, linestyle='--',
            label=f'Predicted VaR (α={_primary_alpha_adaptive:.1%}, PRIMARY)',
        )
        if oos_triple_results is not None:
            ax_top.axhline(
                -oos_triple_results.cvar_realized, color='#8B0000',
                linewidth=1.2, linestyle=':', label='Realized CVaR',
            )
        _legend_breach = Line2D(
            [0], [0], marker='|', color='#d62728',
            markerfacecolor='#d62728', markersize=10, linewidth=0,
            label=f'VaR Breach ({_n_breaches} days)',
        )
        _tl_label = _primary_bt.get('traffic_light', 'N/A')
        ax_top.set_xlim(-1, _n_test_plot)
        ax_top.set_ylabel('Daily Portfolio P&L', fontsize=10)
        ax_top.set_title(
            f"Test: {_dates_plot[0].date()} — {_dates_plot[-1].date()}  "
            f"({_n_test_plot} obs)  |  "
            f"Breaches: {_n_breaches}/{_n_test_plot} ({_n_breaches/_n_test_plot:.2%})  |  "
            f"Traffic: {_tl_label}"
            + ("  [POWER WARNING: n<250]" if _stat_power_warning else ""),
            fontsize=9,
        )
        _h, _l = ax_top.get_legend_handles_labels()
        ax_top.legend(_h + [_legend_breach], _l + [_legend_breach.get_label()],
                      loc='upper left', fontsize=8, framealpha=0.8)
        ax_top.grid(axis='y', alpha=0.25)
        ax_top.axhline(0, color='black', linewidth=0.6, alpha=0.4)

        _kupiec_p = _primary_bt.get('kupiec', {}).get('pvalue', float('nan'))
        _cc_p     = _primary_bt.get('christoffersen', {}).get('pvalue_cc', float('nan'))
        _cc_ind_p = _primary_bt.get('christoffersen', {}).get('pvalue_ind', float('nan'))
        ax_top.annotate(
            f"Primary α={_primary_alpha_adaptive:.3f}  "
            f"Kupiec p={_kupiec_p:.3f}  CC p={_cc_p:.3f}  "
            f"CC-ind p={_cc_ind_p:.3f}  "
            f"Exc={_n_breaches}/{_n_test_plot} ({_n_breaches/_n_test_plot:.2%})",
            xy=(0.99, 0.02), xycoords='axes fraction',
            ha='right', va='bottom', fontsize=7.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='gray', alpha=0.85),
        )

        # Bottom panel: rolling exceedance rate
        if rolling_backtest:
            _roll_rates  = [r['exc_rate'] for r in rolling_backtest]
            _roll_traff  = [r['traffic']  for r in rolling_backtest]

            def _tl_color(t):
                return (
                    '#2ca02c' if t == 'GREEN'
                    else '#ff7f0e' if t == 'YELLOW'
                    else '#d62728' if t == 'RED'
                    else '#aaaaaa'   # INSUFFICIENT_DATA → gray
                )

            _roll_colors = [_tl_color(t) for t in _roll_traff]
            ax_bot.bar(range(len(_roll_rates)), _roll_rates,
                       color=_roll_colors, alpha=0.85, width=0.8)
            ax_bot.axhline(ALPHA, color='black', linewidth=1.2,
                           linestyle=':', label=f'Target α={ALPHA:.2%}')
            ax_bot.set_ylabel('Exc. Rate', fontsize=9)
            ax_bot.set_xlabel(f'Rolling windows ({ROLLING_STEP}-day step)', fontsize=9)
            ax_bot.set_xlim(-0.5, len(_roll_rates) - 0.5)
            ax_bot.set_ylim(0, max(max(_roll_rates) * 1.35, ALPHA * 3.5))
            ax_bot.grid(axis='y', alpha=0.25)
            ax_bot.legend(
                handles=[
                    Patch(facecolor='#2ca02c', label='Green zone'),
                    Patch(facecolor='#ff7f0e', label='Yellow zone'),
                    Patch(facecolor='#d62728', label='Red zone'),
                    Patch(facecolor='#aaaaaa', label='Insufficient data'),
                    Line2D([0], [0], color='black', linestyle=':',
                           label=f'Target α={ALPHA:.2%}'),
                ],
                loc='upper right', fontsize=8, framealpha=0.8,
            )
        else:
            ax_bot.text(0.5, 0.5, 'Rolling backtest: insufficient data',
                        ha='center', va='center', transform=ax_bot.transAxes,
                        fontsize=9, color='gray')
            ax_bot.set_xlabel('Rolling windows', fontsize=9)

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        _fig1_path = _results_dir / f'oos_var_breach_chart_{_exp_id}.png'
        fig1.savefig(str(_fig1_path), dpi=150, bbox_inches='tight')
        plt.close(fig1)
        logger.info(f"VaR breach chart saved: {_fig1_path}")
        print(f"  [Figure 1] VaR breach chart: {_fig1_path}")

    except Exception as _ef1:
        logger.error(f"Figure 1 failed: {_ef1}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"  WARNING: Figure 1 failed - {_ef1}")

else:
    logger.warning("Figure 1 skipped: missing OOS data, weights, or backtest results")
    print("  Figure 1: skipped")


# ---------------------------------------------------------------------------
# Figure 2: Loss Distribution — predicted vs realized per method
# ---------------------------------------------------------------------------

if oos_triple_results is not None and oos_triple_losses is not None:
    try:
        _method_names  = ['parametric', 'monte_carlo', 'quantum']
        _method_labels = ['Parametric (Normal)', 'Monte Carlo', 'Quantum (QAE)']
        _method_colors = ['#1f77b4', '#2ca02c', '#d62728']
        _cvar_realized = oos_triple_results.cvar_realized

        fig2, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig2.suptitle(
            'OOS CVaR Prediction Accuracy — Predicted vs Realized Loss Distribution',
            fontsize=12, fontweight='bold',
        )

        for _ax, _mn, _ml, _mc in zip(axes, _method_names, _method_labels, _method_colors):
            _pred = oos_triple_results.predictions.get(_mn)
            _comp = oos_triple_results.comparisons.get(_mn)
            if _pred is None or _comp is None:
                _ax.text(0.5, 0.5, f'{_ml}\n(not available)',
                         ha='center', va='center', transform=_ax.transAxes)
                continue

            _ax.hist(oos_triple_losses, bins=40, density=True,
                     color='#aec7e8', alpha=0.55, label='Realized losses',
                     edgecolor='white', linewidth=0.4)

            if (hasattr(_pred, 'plot_x') and _pred.plot_x is not None
                    and hasattr(_pred, 'plot_y') and _pred.plot_y is not None):
                _ax.plot(_pred.plot_x, _pred.plot_y,
                         color=_mc, linewidth=2.0, label=f'{_ml} density')

            _ax.axvline(_pred.cvar_predicted, color=_mc, linewidth=1.8,
                        linestyle='--',
                        label=f'CVaR pred={_pred.cvar_predicted:.4f}')
            _ax.axvline(_cvar_realized, color='black', linewidth=1.5,
                        linestyle=':', label=f'CVaR real={_cvar_realized:.4f}')

            _ax.set_title(
                f'{_ml}\n'
                f'CVaR err={_comp.cvar_relative_error:.2%}  '
                f'KS p={_comp.ks_pvalue:.3f}  '
                f'Wass={_comp.wasserstein_distance:.5f}',
                fontsize=9,
            )
            _ax.set_xlabel('Portfolio loss', fontsize=9)
            _ax.set_ylabel('Density' if _ax is axes[0] else '', fontsize=9)
            _ax.legend(fontsize=7.5, loc='upper left')
            _ax.grid(alpha=0.2)

        plt.tight_layout()
        _fig2_path = _results_dir / f'oos_triple_distribution_{_exp_id}.png'
        fig2.savefig(str(_fig2_path), dpi=150, bbox_inches='tight')
        plt.close(fig2)
        logger.info(f"Loss distribution chart saved: {_fig2_path}")
        print(f"  [Figure 2] Loss distribution chart: {_fig2_path}")

    except Exception as _ef2:
        logger.error(f"Figure 2 failed: {_ef2}")
        import traceback
        logger.error(traceback.format_exc())
        print(f"  WARNING: Figure 2 failed - {_ef2}")

else:
    logger.warning("Figure 2 skipped: oos_triple_results not available")
    print("  Figure 2: skipped")


# ---------------------------------------------------------------------------
# JSON export 1: OOS triple comparison
# ---------------------------------------------------------------------------

if oos_triple_results is not None:
    try:
        _j1 = _results_dir / f'oos_triple_comparison_{_exp_id}.json'
        with open(_j1, 'w', encoding='utf-8') as _f:
            _f.write(oos_triple_results.to_json())
        logger.info(f"OOS triple comparison JSON: {_j1}")
        print(f"  [JSON 1] OOS triple comparison: {_j1}")
    except Exception as _ej1:
        logger.error(f"JSON 1 export failed: {_ej1}")
        print(f"  WARNING: JSON 1 failed - {_ej1}")


# ---------------------------------------------------------------------------
# JSON export 2: Backtesting report (fixed — block bootstrap AS, power guard)
# ---------------------------------------------------------------------------

if backtest_results:
    try:
        _j2 = _results_dir / f'oos_backtest_report_{_exp_id}.json'
        with open(_j2, 'w', encoding='utf-8') as _f:
            json.dump(backtest_results, _f, indent=2, ensure_ascii=False, default=str)
        logger.info(f"Backtesting report JSON: {_j2}")
        print(f"  [JSON 2] Backtesting report (fixed): {_j2}")
    except Exception as _ej2:
        logger.error(f"JSON 2 export failed: {_ej2}")
        print(f"  WARNING: JSON 2 failed - {_ej2}")


# ---------------------------------------------------------------------------
# Assemble oos_triple_for_export → comprehensive_metrics (Cell 23)
# ---------------------------------------------------------------------------

_reg_bt = backtest_results.get(f'alpha_{_primary_alpha_adaptive}', {})

oos_triple_for_export = {
    'computed': True,

    # Layer 1
    'best_method':  (oos_triple_results.best_method_by_cvar_error()
                     if oos_triple_results else None),
    'ranking_cvar': (oos_triple_results.ranking_by_cvar_error()
                     if oos_triple_results else []),
    'ranking_wass': (oos_triple_results.ranking_by_wasserstein()
                     if oos_triple_results else []),
    'cvar_realized': float(oos_triple_results.cvar_realized) if oos_triple_results else None,
    'n_train':       int(oos_triple_results.n_train) if oos_triple_results else None,
    'n_test':        int(oos_triple_results.n_test)  if oos_triple_results else None,
    'comparisons': (
        {
            m: {
                'cvar_predicted':      float(c.cvar_predicted),
                'cvar_relative_error': float(c.cvar_relative_error),
                'ks_statistic':        float(c.ks_statistic),
                'ks_pvalue':           float(c.ks_pvalue),
                'wasserstein':         float(c.wasserstein_distance),
                'kl_divergence':       float(c.kl_divergence),
            }
            for m, c in oos_triple_results.comparisons.items()
        }
        if oos_triple_results else {}
    ),
    'oos_portfolio_metrics': (oos_triple_results.oos_metrics
                               if oos_triple_results else {}),

    # Layer 2
    'backtesting':      backtest_results,
    'rolling_backtest': rolling_backtest,

    # Layer 3
    'pnl_attribution': pnl_attribution,

    # Regulatory summary — uses adaptive primary alpha
    'regulatory_summary': {
        'primary_alpha': _primary_alpha_adaptive,
        'primary_alpha_rationale': (
            f"Adaptive: smallest α with E[exc]≥5 given {len(returns_df_oos)} obs"
            if _stat_power_warning else "Config alpha (adequate power)"
        ),
        'n_test_days':              len(returns_df_oos) if returns_df_oos is not None else 0,
        'stat_power_adequate':      not _stat_power_warning,
        'traffic_light':            _reg_bt.get('traffic_light', 'N/A'),
        'kupiec_pass':              bool(_reg_bt.get('kupiec', {}).get('pass', False)),
        'cc_pass':                  bool(_reg_bt.get('christoffersen', {}).get('pass_cc', False)),
        'acerbi_szekely_pass':      _reg_bt.get('acerbi_szekely', {}).get('pass'),
        'acerbi_szekely_z1':        _reg_bt.get('acerbi_szekely', {}).get('z1'),
        'acerbi_szekely_pvalue':    _reg_bt.get('acerbi_szekely', {}).get('pvalue_z1'),
        'all_alphas_kupiec_pass':   all(
            r.get('kupiec', {}).get('pass', False) for r in backtest_results.values()
        ),
        'all_alphas_cc_pass':       all(
            r.get('christoffersen', {}).get('pass_cc', False)
            for r in backtest_results.values()
        ),
        # Key positive result: exceedances are independent at all alpha levels
        'independence_passes_all':  all(
            r.get('christoffersen', {}).get('pass_ind', False)
            for r in backtest_results.values()
        ),
        'exceedance_clustering_risk': not all(
            r.get('christoffersen', {}).get('pass_ind', False)
            for r in backtest_results.values()
        ),
        'pnl_attribution_status':   pnl_attribution.get('status', 'N/A'),
        'pnl_spearman_rho':         pnl_attribution.get('spearman_rho'),
    },
}

logger.info("oos_triple_for_export assembled")


# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 70)
print("OOS TRIPLE COMPARISON — FINAL SUMMARY")
print("=" * 70)

if oos_triple_results:
    print(f"  CVaR realized (test period) : {oos_triple_results.cvar_realized:.6f}")
    print(f"  Best predictive method      : {oos_triple_results.best_method_by_cvar_error()}")
    print(f"  Ranking (CVaR error)        : "
          f"{' > '.join(oos_triple_results.ranking_by_cvar_error())}")
    print(f"  Ranking (Wasserstein)       : "
          f"{' > '.join(oos_triple_results.ranking_by_wasserstein())}")
    print()
    for _m, _c in oos_triple_results.comparisons.items():
        print(f"    {_m:<16}: CVaR err={_c.cvar_relative_error:.4f}, "
              f"KS={_c.ks_statistic:.4f}, Wass={_c.wasserstein_distance:.6f}")

if backtest_results:
    _reg = oos_triple_for_export['regulatory_summary']
    print()
    print(f"  --- Regulatory Backtesting (primary α={_primary_alpha_adaptive:.3f}) ---")
    print(f"  Statistical power adequate  : {_reg['stat_power_adequate']} "
          f"({'n≥250' if _reg['stat_power_adequate'] else str(_reg['n_test_days']) + ' < 250'})")
    print(f"  Kupiec POF                  : {'PASS' if _reg['kupiec_pass'] else 'FAIL'}")
    print(f"  Christoffersen CC           : {'PASS' if _reg['cc_pass'] else 'FAIL'}")
    _az = _reg.get('acerbi_szekely_pass')
    print(f"  Acerbi-Szekely Z1           : "
          f"{'PASS' if _az is True else 'FAIL' if _az is False else 'N/A'}"
          f"  (Z1={_reg.get('acerbi_szekely_z1', 'N/A')}, "
          f"p={_reg.get('acerbi_szekely_pvalue', 'N/A')})")
    print(f"  Independence (all α)        : "
          f"{'PASS' if _reg['independence_passes_all'] else 'FAIL'}")
    print(f"  All α Kupiec                : "
          f"{'PASS' if _reg['all_alphas_kupiec_pass'] else 'PARTIAL'}")
    print(f"  All α CC                    : "
          f"{'PASS' if _reg['all_alphas_cc_pass'] else 'PARTIAL'}")

if pnl_attribution:
    print()
    print(f"  --- P&L Attribution (FRTB) ---")
    print(f"  Spearman ρ                  : "
          f"{pnl_attribution.get('spearman_rho', float('nan')):.4f}")
    print(f"  Status                      : {pnl_attribution.get('status', 'N/A')}")

print("=" * 70)

logger.info(
    "Cell 27 complete — "
    f"Layer1={'OK' if oos_triple_results else 'SKIP'}, "
    f"Layer2={'OK' if backtest_results else 'SKIP'}, "
    f"Layer3={'OK' if pnl_attribution else 'SKIP'}, "
    f"primary_alpha={_primary_alpha_adaptive}"
)

  [Figure 1] VaR breach chart: C:\Users\nacho\Desktop\v2_final\v2_final\results\exp_A3_PB\oos_var_breach_chart_A3_PB.png


  [Figure 2] Loss distribution chart: C:\Users\nacho\Desktop\v2_final\v2_final\results\exp_A3_PB\oos_triple_distribution_A3_PB.png
  [JSON 1] OOS triple comparison: C:\Users\nacho\Desktop\v2_final\v2_final\results\exp_A3_PB\oos_triple_comparison_A3_PB.json
  [JSON 2] Backtesting report (fixed): C:\Users\nacho\Desktop\v2_final\v2_final\results\exp_A3_PB\oos_backtest_report_A3_PB.json

OOS TRIPLE COMPARISON — FINAL SUMMARY
  CVaR realized (test period) : 0.023583
  Best predictive method      : parametric
  Ranking (CVaR error)        : parametric > quantum > monte_carlo
  Ranking (Wasserstein)       : parametric > monte_carlo > quantum

    parametric      : CVaR err=0.4239, KS=0.1738, Wass=0.006245
    monte_carlo     : CVaR err=0.4382, KS=0.1748, Wass=0.006281
    quantum         : CVaR err=0.4328, KS=0.2558, Wass=0.006574

  --- Regulatory Backtesting (primary α=0.010) ---
  Statistical power adequate  : True (n≥250)
  Kupiec POF                  : PASS
  Christoffersen CC           